# HeritageFormer — Heritage Retrieval Engine (clean rebuild v2)

**Goal:** evidence-based restoration reasoning via semantic retrieval — *not* monument classification.

Pipeline: discover datasets -> normalize + clean -> embed with BGE-M3 ->
FAISS search -> knowledge graph -> restoration-reasoning demo.

v2 fixes: strips HTML from descriptions, drops duplicate rows, and lets
restoration search pull evidence from TANGIBLE structures only (so intangible
storytelling rows stop polluting reconstruction evidence).

In [ ]:
# ============================================================
# 0) SETUP
# ============================================================
import os, glob, re, pickle, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__, "| Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

try:
    import faiss
except ImportError:
    os.system("pip -q install faiss-cpu")
    import faiss
print("FAISS ready")

In [ ]:
# ============================================================
# 1) AUTO-DISCOVER every CSV under /kaggle/input
#    Mount paths change between sessions -> never hard-code them.
# ============================================================
ALL_CSVS = glob.glob("/kaggle/input/**/*.csv", recursive=True)
print("CSV files found:", len(ALL_CSVS))

def find_csv(*keywords):
    """First CSV whose path contains ALL keywords (case-insensitive)."""
    for p in ALL_CSVS:
        low = p.lower()
        if all(k.lower() in low for k in keywords):
            return p
    return None

paths = {
    "unesco2021": find_csv("tangible", "2021") or find_csv("2021"),
    "unesco2019": find_csv("whc-sites-2019") or find_csv("2019"),
    "heritage3d": find_csv("cultural_heritage_dataset") or find_csv("heritage", "3d"),
    "intangible": find_csv("intangible"),
}
for k, v in paths.items():
    print(f"{k:12s} -> {v}")

In [ ]:
# ============================================================
# 2) NORMALIZE + CLEAN all datasets into ONE schema
#    name, description, country, region, type, period,
#    material, source, kind, latitude, longitude
#    kind = "tangible" (a real structure) or "intangible"
# ============================================================
TAG_RE = re.compile(r"<[^>]+>")
WS_RE  = re.compile(r"\s+")
def clean(x):
    """Strip HTML tags and collapse whitespace."""
    s = TAG_RE.sub(" ", str(x))
    return WS_RE.sub(" ", s).strip()

def col(df, *cands, default=""):
    for c in cands:
        if c in df.columns:
            return df[c]
    return pd.Series([default] * len(df))

def safe_read(path):
    if not path or not os.path.exists(path):
        return None
    try:
        return pd.read_csv(path)
    except Exception as e:
        print("  ! could not read", path, "|", e)
        return None

frames = []

# --- UNESCO 2021 (tangible sites) ---
df = safe_read(paths["unesco2021"])
if df is not None:
    o = pd.DataFrame()
    o["name"]        = col(df, "Name", "name_en")
    o["description"] = col(df, "short_description", "short_description_en")
    o["country"]     = col(df, "Country name", "states_name_en")
    o["region"]      = col(df, "Region", "region_en")
    o["type"]        = col(df, "category_long", "category")
    o["period"] = ""; o["material"] = ""; o["source"] = "UNESCO2021"; o["kind"] = "tangible"
    o["latitude"]    = col(df, "latitude",  default=np.nan)
    o["longitude"]   = col(df, "longitude", default=np.nan)
    frames.append(o); print("UNESCO2021:", len(o))

# --- UNESCO 2019 (tangible sites) ---
df = safe_read(paths["unesco2019"])
if df is not None:
    o = pd.DataFrame()
    o["name"]        = col(df, "name_en", "Name")
    o["description"] = col(df, "short_description_en", "short_description")
    o["country"]     = col(df, "states_name_en", "Country name")
    o["region"]      = col(df, "region_en", "Region")
    o["type"]        = col(df, "category", "category_long")
    o["period"] = ""; o["material"] = ""; o["source"] = "UNESCO2019"; o["kind"] = "tangible"
    o["latitude"]    = col(df, "latitude",  default=np.nan)
    o["longitude"]   = col(df, "longitude", default=np.nan)
    frames.append(o); print("UNESCO2019:", len(o))

# --- High-Fidelity Cultural Heritage 3D ---
df = safe_read(paths["heritage3d"])
if df is not None:
    o = pd.DataFrame()
    o["name"]        = col(df, "Artifact_Name", "name")
    o["description"] = ("3D scanned heritage artifact. Scan type "
                        + col(df, "Scan_Type").astype(str)
                        + ", resolution " + col(df, "Resolution").astype(str)
                        + ", level of detail " + col(df, "LOD_Level").astype(str) + ".")
    o["country"] = ""; o["region"] = ""
    o["type"]        = "3D Artifact"
    o["period"]      = col(df, "Historical_Period")
    o["material"] = ""; o["source"] = "Heritage3D"; o["kind"] = "tangible"
    o["latitude"] = np.nan; o["longitude"] = np.nan
    frames.append(o); print("Heritage3D:", len(o))

# --- Intangible heritage (visual storytelling) ---
df = safe_read(paths["intangible"])
if df is not None:
    o = pd.DataFrame()
    o["name"]        = col(df, "Heritage_Type", "File_Name", "ID").astype(str)
    o["description"] = col(df, "Description")
    o["country"] = ""
    o["region"]      = col(df, "Region")
    o["type"]        = "Intangible: " + col(df, "Narrative_Label", "Modality").astype(str)
    o["period"] = ""; o["material"] = ""; o["source"] = "Intangible"; o["kind"] = "intangible"
    o["latitude"] = np.nan; o["longitude"] = np.nan
    frames.append(o); print("Intangible:", len(o))

heritage_master = pd.concat(frames, ignore_index=True)

# clean text fields (removes <p> HTML, collapses whitespace)
for c in ["name","description","country","region","type","period","material"]:
    heritage_master[c] = heritage_master[c].fillna("").map(clean)

# drop blanks and exact-duplicate records (kills the "Temple Architecture" spam)
heritage_master = heritage_master[heritage_master["name"] != ""]
heritage_master = heritage_master.drop_duplicates(
    subset=["name","type","region","description"]).reset_index(drop=True)

print("\nHERITAGE MASTER:", heritage_master.shape)
print(heritage_master.groupby(["kind","source"]).size())
heritage_master.head()

In [ ]:
# ============================================================
# 3) Build a rich "Heritage Knowledge Record" per row.
#    Richer text -> stronger BGE-M3 embeddings -> better retrieval.
# ============================================================
def build_record(r):
    parts = [f"Heritage Name: {r['name']}"]
    if r["type"]:        parts.append(f"Heritage Type: {r['type']}")
    if r["period"]:      parts.append(f"Historical Period: {r['period']}")
    if r["country"]:     parts.append(f"Country: {r['country']}")
    if r["region"]:      parts.append(f"Region: {r['region']}")
    if r["material"]:    parts.append(f"Construction Material: {r['material']}")
    if r["description"]: parts.append(f"Description: {r['description']}")
    parts.append("Context: cultural, architectural and archaeological reference "
                 "record used for evidence-based restoration and reconstruction.")
    return "\n".join(parts)

heritage_master["record_text"] = [build_record(r) for _, r in heritage_master.iterrows()]
print(heritage_master["record_text"].iloc[0][:400])

In [ ]:
# ============================================================
# 4) Embed every record with BGE-M3 (1024-dim, multilingual)
# ============================================================
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer("BAAI/bge-m3", device=DEVICE)

records = heritage_master["record_text"].tolist()
embeddings = encoder.encode(
    records,
    batch_size=64,
    normalize_embeddings=True,   # required for cosine / inner-product search
    show_progress_bar=True,
    convert_to_numpy=True,
).astype("float32")

np.save("/kaggle/working/embeddings.npy", embeddings)
print("Embeddings:", embeddings.shape)

In [ ]:
# ============================================================
# 5) FAISS index (inner product == cosine on normalized vectors)
# ============================================================
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
faiss.write_index(index, "/kaggle/working/heritage.index")
print("Indexed vectors:", index.ntotal)

In [ ]:
# ============================================================
# 6) Search engine + restoration-oriented query builder
#    tangible_only=True -> evidence comes from real structures
#    (UNESCO + Heritage3D), excluding intangible storytelling rows.
# ============================================================
def heritage_search(query, top_k=10, source=None, tangible_only=False):
    q = encoder.encode([query], normalize_embeddings=True).astype("float32")
    over = 8 if (source or tangible_only) else 1     # over-fetch then filter
    scores, idx = index.search(q, top_k * over)
    res = heritage_master.iloc[idx[0]].copy()
    res["score"] = scores[0]
    if tangible_only:
        res = res[res["kind"] == "tangible"]
    if source:
        res = res[res["source"] == source]
    cols = ["name","country","region","type","period","source","score"]
    return res.head(top_k)[cols].reset_index(drop=True)

def restoration_query(structure_type, period="", material="", location=""):
    return (f"{period} {structure_type} {material} {location} "
            "historical architecture defensive architecture "
            "heritage conservation restoration reference").strip()

heritage_search(restoration_query("hill fort", "17th century maratha",
                                  "basalt stone", "India"),
                tangible_only=True)

In [ ]:
# ============================================================
# 7) Knowledge graph (NetworkX) — richer node & edge types,
#    plus analogical SIMILAR_TO edges mined from retrieval.
# ============================================================
import networkx as nx
G = nx.MultiDiGraph()

def add_node(n, t):
    n = str(n).strip()
    if n:
        G.add_node(n, node_type=t)

for _, r in heritage_master.iterrows():
    site = str(r["name"]).strip()
    add_node(site, "site")
    for val, typ, rel in [
        (r["country"],  "country",  "LOCATED_IN"),
        (r["region"],   "region",   "IN_REGION"),
        (r["type"],     "type",     "HAS_TYPE"),
        (r["period"],   "period",   "FROM_PERIOD"),
        (r["material"], "material", "USES_MATERIAL"),
        (r["source"],   "source",   "DOCUMENTED_IN"),
    ]:
        if str(val).strip():
            add_node(val, typ)
            G.add_edge(site, str(val).strip(), relation=rel)

# analogical edges: link each site to its 3 nearest neighbours
D, I = index.search(embeddings, 4)          # 4 = self + 3 neighbours
for i in range(len(heritage_master)):
    src = str(heritage_master.iloc[i]["name"]).strip()
    for j, sc in zip(I[i][1:], D[i][1:]):   # skip self (column 0)
        dst = str(heritage_master.iloc[j]["name"]).strip()
        if src != dst:
            G.add_edge(src, dst, relation="SIMILAR_TO", weight=float(sc))

with open("/kaggle/working/heritage_graph.pkl", "wb") as f:
    pickle.dump(G, f)

types = {}
for _, d in G.nodes(data=True):
    t = d.get("node_type", "?"); types[t] = types.get(t, 0) + 1
print("Nodes:", G.number_of_nodes(), "| Edges:", G.number_of_edges())
print("Node types:", types)

In [ ]:
# ============================================================
# 8) RESTORATION REASONING — worked example
#    Konark Sun Temple: its main tower (deul) collapsed long ago.
#    HeritageFormer reasons by analogy to INTACT similar temples.
# ============================================================
def restoration_report(target_hint, structure_type, period, material, location):
    print("=" * 64)
    print("RESTORATION TARGET:", target_hint)
    print("=" * 64)

    # resolve the closest real (tangible) record for the target
    match = heritage_search(target_hint, top_k=1, tangible_only=True)
    target = match.iloc[0]["name"] if len(match) else target_hint
    print("Resolved to record:", target, "\n")

    # evidence pool = real structures only
    refs = heritage_search(restoration_query(structure_type, period, material, location),
                           top_k=8, tangible_only=True)
    print("Top reference structures (tangible evidence pool):\n")
    print(refs.to_string(index=False))

    print("\nClosest analogues (graph SIMILAR_TO edges):")
    if target in G:
        sims = [(v, d["weight"]) for _, v, d in G.out_edges(target, data=True)
                if d.get("relation") == "SIMILAR_TO"]
        for v, w in sorted(sims, key=lambda x: -x[1])[:5]:
            print(f"   - {v}  (similarity {w:.3f})")
    else:
        print("   (target not yet a graph node)")

    print("\nReasoning trace:")
    print(" 1. Retrieve historically/structurally similar real structures (above).")
    print(" 2. Keep only same type / period / material as supporting evidence.")
    print(" 3. Infer missing components from the INTACT analogues.")
    print(" 4. Justify every reconstruction choice with its source record.")

restoration_report(
    target_hint="Konark Sun Temple Odisha",
    structure_type="Kalinga style Hindu temple shikhara tower",
    period="13th century Eastern Ganga dynasty",
    material="khondalite sandstone",
    location="Odisha India",
)

In [ ]:
# ============================================================
# 9) Save artifacts for reuse / next stages
# ============================================================
heritage_master.to_parquet("/kaggle/working/heritage_master.parquet")
print("Saved to /kaggle/working:")
for f in ["heritage_master.parquet", "embeddings.npy",
          "heritage.index", "heritage_graph.pkl"]:
    p = f"/kaggle/working/{f}"
    if os.path.exists(p):
        print(f"  {f:28s} {os.path.getsize(p)/1e6:7.1f} MB")

---
# v3 — Component-Level Restoration Reasoning

The v2 engine retrieves similar *sites* but can't reason about missing *parts*
(Konark's collapsed tower, a fort's lost bastion). v3 adds:

1. A hand-authored **component ontology** (canonical parts per structure archetype) — citeable, no fictional data.
2. **Missing-component inference**: for a target structure, list its canonical
   components, flag documented losses, and retrieve INTACT exemplars of each
   missing component from analogous sites — with an evidence trail + confidence.

No GPU and no training — this runs on the retrieval + graph you already built.

In [ ]:
# ============================================================
# 10) COMPONENT ONTOLOGY  (hand-authored, citeable)
#     Canonical components per structure archetype.
# ============================================================
component_ontology = {
    "kalinga_temple": [
        "deul / rekha shikhara (curvilinear main tower)",
        "jagamohana (assembly hall / mandapa)",
        "nata-mandira (dance hall)",
        "bhoga-mandapa (hall of offerings)",
        "amalaka and kalasha (crowning finial)",
    ],
    "nagara_temple": [
        "shikhara (curvilinear spire)",
        "garbhagriha (sanctum)",
        "mandapa (pillared hall)",
        "antarala (vestibule)",
        "amalaka and kalasha (finial)",
    ],
    "dravidian_temple": [
        "vimana (pyramidal tower over sanctum)",
        "gopuram (gateway tower)",
        "mandapa (pillared hall)",
        "garbhagriha (sanctum)",
        "prakara (enclosure wall)",
    ],
    "mughal_tomb": [
        "double dome",
        "char-bagh (formal garden)",
        "pishtaq / iwan (recessed portal)",
        "minaret",
        "raised plinth",
    ],
    "indo_islamic": [
        "dome",
        "minaret",
        "pointed arch",
        "courtyard (sahn)",
        "iwan (portal)",
    ],
    "hill_fort": [
        "rampart (defensive wall)",
        "bastion (burj)",
        "gateway (darwaza)",
        "watchtower",
        "water cistern / tanka",
        "citadel (inner fort)",
    ],
    "cave_temple": [
        "chaitya hall (prayer hall)",
        "vihara (monastic cells)",
        "rock-cut facade",
        "rock-cut pillars",
        "stupa / shrine",
    ],
    "buddhist_monument": [
        "stupa (hemispherical dome)",
        "harmika and chhatra (crowning umbrella)",
        "toranas (gateways)",
        "circumambulatory path (pradakshina)",
        "vedika (railing)",
    ],
    "generic_monument": [
        "foundation / plinth",
        "load-bearing walls",
        "superstructure / roof",
        "entrance / gateway",
        "ornamentation",
    ],
}

# short retrieval descriptor per archetype (used to find intact exemplars)
ARCHETYPE_DESC = {
    "kalinga_temple":  "Kalinga Odisha Hindu temple",
    "nagara_temple":   "Nagara north Indian Hindu temple",
    "dravidian_temple":"Dravidian south Indian Hindu temple",
    "mughal_tomb":     "Mughal Indo-Islamic tomb mausoleum",
    "indo_islamic":    "Indo-Islamic mosque monument",
    "hill_fort":       "Indian hill fort defensive architecture",
    "cave_temple":     "Indian rock-cut cave temple",
    "buddhist_monument":"Buddhist stupa monument",
    "generic_monument":"historic monument",
}

def infer_archetype(name, typ, desc, region):
    """Map a record to a structure archetype using keyword rules."""
    t = f"{name} {typ} {desc} {region}".lower()
    def has(*ks): return any(k in t for k in ks)
    if has("konâ", "konar", "kalinga") or (has("temple") and "odisha" in t):
        return "kalinga_temple"
    if has("chola", "vimana", "gopuram", "dravid", "mahabalipuram", "mamallapuram", "pallava"):
        return "dravidian_temple"
    if has("khajuraho") or (has("temple") and has("nagara")):
        return "nagara_temple"
    if has("tomb", "mausoleum", "maqbara"):
        return "mughal_tomb"
    if has("stupa") or has("sanchi") or has("buddhist monument"):
        return "buddhist_monument"
    if has("cave", "ajanta", "ellora", "elephanta"):
        return "cave_temple"
    if has("minar", "minaret", "mosque", "masjid", "qutb"):
        return "indo_islamic"
    if has("fort", "fortress", "qila", "durg", "rampart"):
        return "hill_fort"
    if has("temple"):
        return "nagara_temple"
    return "generic_monument"

# documented losses for well-studied sites (curated; cite ASI/UNESCO)
KNOWN_MISSING = {
    "Sun Temple, Konârak": {
        "deul / rekha shikhara (curvilinear main tower)":
            "collapsed by c.1837; once ~70 m and the tallest element, now lost",
    },
    "Group of Monuments at Hampi": {
        "superstructure / roof":
            "extensive ruin after 1565 sack of Vijayanagara; many superstructures lost",
    },
}
print("Ontology archetypes:", list(component_ontology))

In [ ]:
# ============================================================
# 11) MISSING-COMPONENT INFERENCE + worked restoration plan
# ============================================================
def _confidence(score):
    if score >= 0.55: return "HIGH"
    if score >= 0.45: return "MEDIUM"
    return "LOW"

def component_evidence(component, archetype, exclude_name, k=3):
    """Retrieve intact tangible exemplars that best evidence a component."""
    q = f"{ARCHETYPE_DESC.get(archetype,'')} {component} intact complete architecture India"
    r = heritage_search(q, top_k=k + 3, tangible_only=True)
    r = r[r["name"] != exclude_name].head(k)
    return r

def restoration_plan(target_hint, location="India"):
    print("=" * 70)
    print("RESTORATION PLAN:", target_hint)
    print("=" * 70)

    match = heritage_search(f"{target_hint} {location}", top_k=1, tangible_only=True)
    if not len(match):
        print("No tangible record found."); return
    row = match.iloc[0]
    target = row["name"]

    # recover full record (archetype needs the description)
    rec = heritage_master[heritage_master["name"] == target].iloc[0]
    arch = infer_archetype(rec["name"], rec["type"], rec["description"], rec["region"])
    canon = component_ontology[arch]

    # documented losses for this target (substring match, accents safe)
    missing = {}
    for k, v in KNOWN_MISSING.items():
        if k.lower() in target.lower() or target.lower() in k.lower():
            missing = v
    missing_set = set(missing)

    print(f"Resolved record : {target}")
    print(f"Archetype       : {arch}")
    print(f"Canonical parts : {len(canon)}\n")

    print("COMPONENT STATUS")
    print("-" * 70)
    for c in canon:
        tag = "MISSING" if c in missing_set else "present / to verify"
        note = f"  <- {missing[c]}" if c in missing_set else ""
        print(f"  [{tag:18s}] {c}{note}")

    if not missing_set:
        print("\nNo documented losses for this site. Showing reference exemplars "
              "for every canonical component instead.")
    targets = list(missing_set) if missing_set else canon

    print("\nEVIDENCE-BASED RECONSTRUCTION REFERENCES")
    print("-" * 70)
    for c in targets:
        ev = component_evidence(c, arch, target)
        print(f"\n  Component: {c}")
        if not len(ev):
            print("    (no analogous exemplar found)"); continue
        for _, e in ev.iterrows():
            print(f"    - {e['name']}  [{e['source']}]  "
                  f"sim={e['score']:.3f}  confidence={_confidence(e['score'])}")

    print("\nINFERENCE CHAIN")
    print("-" * 70)
    print(" 1. Classify target -> archetype -> canonical component set.")
    print(" 2. Flag components with documented loss (curated from ASI/UNESCO).")
    print(" 3. For each missing part, retrieve INTACT exemplars of the SAME")
    print("    archetype as reconstruction evidence (tangible records only).")
    print(" 4. Confidence = retrieval similarity (style+period+region proximity).")
    print(" 5. Every suggested form is traceable to a cited source record.")
    print("\nNOTE: this proposes evidence-backed references, not generated imagery.")

restoration_plan("Konark Sun Temple Odisha")

---
# Evaluation Harness — measure retrieval quality

Eyeballing results isn't enough for publication or for proving v3 > v2. This
scores the engine on a small labelled set of restoration-style queries using
**Recall@5, Recall@10 and MRR** (mean reciprocal rank). A query "hits" if any
accepted site name appears in the top-k tangible results.

In [ ]:
# ============================================================
# 12) RETRIEVAL EVAL — Recall@k and MRR on labelled queries
# ============================================================
# (query, [accepted name substrings]) — all are real UNESCO India sites
EVAL_SET = [
    ("mughal tomb white marble dome garden",          ["taj mahal"]),
    ("mughal emperor tomb delhi garden",              ["humayun"]),
    ("rajasthan hill fort defensive walls",           ["hill forts of rajasthan"]),
    ("mughal red sandstone fort delhi",               ["red fort"]),
    ("mughal fort agra",                              ["agra fort"]),
    ("buddhist rock cut cave paintings",              ["ajanta"]),
    ("rock cut cave temples kailasa",                 ["ellora"]),
    ("sun temple chariot wheels odisha",              ["sun temple", "konâr", "konar"]),
    ("mountain railway india heritage",               ["mountain railways"]),
    ("vijayanagara ruins hampi temples",              ["hampi"]),
    ("great buddhist stupa sanchi",                   ["sanchi"]),
    ("khajuraho temples sculpture",                   ["khajuraho"]),
    ("great living chola temples south india",        ["chola"]),
    ("qutb minar tower delhi",                        ["qutb"]),
    ("fatehpur sikri mughal city",                    ["fatehpur sikri"]),
    ("churches convents goa portuguese",              ["goa"]),
    ("pallava monuments mahabalipuram shore temple",  ["mahabalipuram", "mamallapuram"]),
    ("elephanta island cave shiva",                   ["elephanta"]),
]

def evaluate(eval_set, k_values=(5, 10)):
    maxk = max(k_values)
    hits = {k: 0 for k in k_values}
    rr_sum = 0.0
    misses = []
    for query, gold in eval_set:
        res = heritage_search(query, top_k=maxk, tangible_only=True)
        names = [n.lower() for n in res["name"].tolist()]
        rank = None
        for i, nm in enumerate(names):
            if any(g in nm for g in gold):
                rank = i + 1; break
        for k in k_values:
            if rank is not None and rank <= k:
                hits[k] += 1
        rr_sum += (1.0 / rank) if rank else 0.0
        if rank is None:
            misses.append((query, gold))
    n = len(eval_set)
    print("RETRIEVAL QUALITY  (tangible_only, n =", n, ")")
    print("-" * 50)
    for k in k_values:
        print(f"  Recall@{k:<2d} : {hits[k]/n:.3f}  ({hits[k]}/{n})")
    print(f"  MRR      : {rr_sum/n:.3f}")
    if misses:
        print("\n  Misses (not found in top", maxk, "):")
        for q, g in misses:
            print(f"    - '{q}'  expected {g}")
    return {"recall": {k: hits[k]/n for k in k_values}, "mrr": rr_sum/n}

eval_metrics = evaluate(EVAL_SET)

---
# Metrics Logging — version history

Every eval run is appended to `metrics_history.json` (+ `.csv`) in
`/kaggle/working`, so Recall@5 / Recall@10 / MRR are tracked **across versions**.

Bump the `version=` label whenever you change something (more records, hybrid
retrieval, reranking…). Re-running the same version label overwrites its row
instead of duplicating. To keep history across Kaggle sessions, download the
JSON (or save it as a dataset) and re-attach it next run — the loader picks it
up automatically from any attached input.

In [ ]:
# ============================================================
# 13) METRICS LOGGING — track Recall/MRR across versions
# ============================================================
import json as _json, datetime
import pandas as pd

METRICS_PATH = "/kaggle/working/metrics_history.json"
METRICS_CSV  = "/kaggle/working/metrics_history.csv"

def load_history():
    """Load existing history from working dir, else from any attached input."""
    candidates = [METRICS_PATH] + glob.glob(
        "/kaggle/input/**/metrics_history.json", recursive=True)
    for p in candidates:
        if os.path.exists(p):
            try:
                return _json.load(open(p))
            except Exception:
                pass
    return []

def log_metrics(version, metrics, n_records, notes="", replace_same_version=True):
    hist = load_history()
    if replace_same_version:
        hist = [r for r in hist if r.get("version") != version]
    row = {
        "version":   version,
        "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
        "n_records": int(n_records),
        "recall@5":  round(metrics["recall"].get(5,  float("nan")), 4),
        "recall@10": round(metrics["recall"].get(10, float("nan")), 4),
        "mrr":       round(metrics["mrr"], 4),
        "encoder":   "BAAI/bge-m3",
        "notes":     notes,
    }
    hist.append(row)
    _json.dump(hist, open(METRICS_PATH, "w"), indent=2)
    df = pd.DataFrame(hist)
    df.to_csv(METRICS_CSV, index=False)
    return df

history_df = log_metrics(
    version="v3-dense-tangible",
    metrics=eval_metrics,
    n_records=len(heritage_master),
    notes="BGE-M3 dense + FAISS IP, tangible-only eval, deduped+cleaned records",
)

print("METRICS HISTORY")
print("=" * 70)
print(history_df[["version","n_records","recall@5","recall@10","mrr","timestamp"]]
      .to_string(index=False))

# improvement vs the previous DIFFERENT version, if any
prev = history_df[history_df["version"] != history_df.iloc[-1]["version"]]
if len(prev):
    p, c = prev.iloc[-1], history_df.iloc[-1]
    print(f"\nvs {p['version']}:  "
          f"Recall@10 {c['recall@10']-p['recall@10']:+.4f}   "
          f"MRR {c['mrr']-p['mrr']:+.4f}")
else:
    print("\n(first logged run — this is your baseline)")

---
# v4 — Cultural-Tradition Layer (cultural-appropriate reconstruction)

The v3 Konark plan offered Sanchi (a Buddhist stupa) and Bagan (Burmese) as
"analogues" for an Odishan temple tower — **culturally wrong**. v4 fixes this:

1. Every record gets a **cultural/architectural tradition** tag.
2. A dedicated **tangible-only FAISS index** (fixes intangible records crowding
   out real structures past the search cutoff — the empty Konark evidence pool).
3. Reconstruction evidence is **constrained to the same tradition**. If none
   exists in the knowledge base, the system **reports the gap honestly** instead
   of substituting a wrong-tradition building. Refusal-by-default is the point.

This is metadata + indexing only — no re-embedding, no training.

In [ ]:
# ============================================================
# 14) CULTURAL / ARCHITECTURAL TRADITION TAGGING
#     World-spanning heuristic v1 (region + keyword rules).
# ============================================================
def infer_tradition(name, typ, desc, country, region):
    t = f"{name} {typ} {desc}".lower()
    c = str(country).lower()
    rg = str(region).lower()
    def has(*ks): return any(k in t for k in ks)

    # --- India: resolve to specific architectural traditions ---
    indian = ("india" in c) or ("india" in rg) or has("indian")
    if indian or has("hindu temple", "mughal", "kalinga", "dravid", "nagara"):
        if has("konâ", "konar", "kalinga") or ("temple" in t and "odisha" in t):
            return "Kalinga (Odisha temple)"
        if has("chola", "dravid", "vimana", "gopuram", "pallava",
               "mahabalipuram", "mamallapuram", "pandya", "tamil"):
            return "Dravidian (South Indian temple)"
        if has("khajuraho", "nagara", "chandela", "orchha"):
            return "Nagara (North Indian temple)"
        if has("tomb", "mausoleum", "mughal", "mosque", "masjid", "minar",
               "qutb", "humayun", "taj mahal", "fatehpur"):
            return "Indo-Islamic / Mughal"
        if has("ajanta", "ellora", "elephanta", "cave", "stupa", "sanchi",
               "buddhist", "vihara", "chaitya"):
            return "Indian Buddhist / rock-cut"
        if has("fort", "fortress", "qila", "durg", "rampart", "citadel"):
            return "Indian fort architecture"
        if has("temple"):
            return "Indian temple (tradition unspecified)"

    # --- World buckets (coarse v1) ---
    if "cambodia" in c or has("angkor", "khmer"):            return "Khmer"
    if "sri lanka" in c or has("sinhal", "anuradhapura"):    return "Sinhalese"
    if "china" in c:                                         return "Chinese imperial"
    if "japan" in c:                                         return "Japanese"
    if "egypt" in c or has("pharaoh", "pyramid of giza"):    return "Ancient Egyptian"
    if has("maya", "aztec", "teotihuacan") or c in ("mexico","guatemala","honduras"):
        return "Mesoamerican"
    if has("inca", "machu picchu") or c in ("peru","bolivia"):
        return "Andean"
    if has("gothic", "cathedral", "romanesque") or ("europe" in rg and has("church","abbey","minster")):
        return "European medieval (Gothic/Romanesque)"
    if has("greek", "roman", "acropolis", "forum", "colosseum", "classical"):
        return "Greco-Roman classical"
    if "arab states" in rg or has("mosque","islamic","caliph","madrasa"):
        return "Islamic (Middle East / North Africa)"
    # fall back to region family
    return f"{region} heritage" if str(region).strip() else "unspecified tradition"

heritage_master["tradition"] = [
    infer_tradition(r["name"], r["type"], r["description"], r["country"], r["region"])
    for _, r in heritage_master.iterrows()
]

print("Tradition distribution (top 15):")
print(heritage_master["tradition"].value_counts().head(15).to_string())

In [ ]:
# ============================================================
# 15) TANGIBLE-ONLY FAISS SUB-INDEX  (fixes empty evidence pool)
# ============================================================
tangible_mask   = (heritage_master["kind"] == "tangible").to_numpy()
tangible_master = heritage_master[tangible_mask].reset_index(drop=True)
tangible_emb    = embeddings[tangible_mask]

tangible_index = faiss.IndexFlatIP(tangible_emb.shape[1])
tangible_index.add(tangible_emb)
print("Tangible records indexed:", tangible_index.ntotal)

def tangible_search(query, top_k=10, tradition=None):
    """Search ONLY real structures. Optionally constrain to one tradition."""
    q = encoder.encode([query], normalize_embeddings=True).astype("float32")
    over = 12 if tradition else 1
    scores, idx = tangible_index.search(q, top_k * over)
    res = tangible_master.iloc[idx[0]].copy()
    res["score"] = scores[0]
    if tradition:
        res = res[res["tradition"] == tradition]
    cols = ["name","country","region","type","tradition","source","score"]
    return res.head(top_k)[cols].reset_index(drop=True)

# sanity: same fort query, now via the clean tangible index
tangible_search(restoration_query("hill fort", "17th century maratha",
                                  "basalt stone", "India"))

In [ ]:
# ============================================================
# 16) CULTURE-CONSTRAINED RESTORATION PLAN (honest refusal)
# ============================================================
# Canonical reference structures NOT yet in the KB — used to report gaps.
KNOWN_REFERENCES = {
    "Kalinga (Odisha temple)": [
        "Lingaraja Temple, Bhubaneswar", "Jagannath Temple, Puri",
        "Mukteshwar Temple", "Rajarani Temple"],
    "Dravidian (South Indian temple)": [
        "Meenakshi Temple, Madurai", "Brihadeeswarar Temple, Thanjavur"],
    "Nagara (North Indian temple)": [
        "Kandariya Mahadeva Temple", "Sun Temple, Modhera"],
}

def restoration_plan_v4(target_hint, location="India"):
    print("=" * 72)
    print("CULTURE-CONSTRAINED RESTORATION PLAN:", target_hint)
    print("=" * 72)

    match = tangible_search(f"{target_hint} {location}", top_k=1)
    if not len(match):
        print("No tangible record found."); return
    target    = match.iloc[0]["name"]
    tradition = match.iloc[0]["tradition"]
    rec  = heritage_master[heritage_master["name"] == target].iloc[0]
    arch = infer_archetype(rec["name"], rec["type"], rec["description"], rec["region"])
    canon = component_ontology[arch]

    missing = {}
    for k, v in KNOWN_MISSING.items():
        if k.lower() in target.lower() or target.lower() in k.lower():
            missing = v
    missing_set = set(missing)

    print(f"Resolved record : {target}")
    print(f"Tradition       : {tradition}")
    print(f"Archetype       : {arch}\n")

    print("COMPONENT STATUS")
    print("-" * 72)
    for c in canon:
        tag = "MISSING" if c in missing_set else "present / to verify"
        note = f"  <- {missing[c]}" if c in missing_set else ""
        print(f"  [{tag:18s}] {c}{note}")

    targets = list(missing_set) if missing_set else canon
    print("\nSAME-TRADITION RECONSTRUCTION EVIDENCE")
    print("-" * 72)
    for c in targets:
        q = f"{tradition} {c} intact complete architecture"
        ev = tangible_search(q, top_k=4, tradition=tradition)
        ev = ev[ev["name"] != target]
        print(f"\n  Component: {c}")
        if len(ev):
            for _, e in ev.iterrows():
                conf = _confidence(e["score"])
                print(f"    - {e['name']}  [{e['source']}]  sim={e['score']:.3f}  {conf}")
        else:
            print(f"    [CULTURAL EVIDENCE GAP] no other '{tradition}' structure in the")
            print(f"     knowledge base — a faithful reconstruction CANNOT be evidenced.")
            refs = KNOWN_REFERENCES.get(tradition)
            if refs:
                print(f"     Acquire these canonical references first: {', '.join(refs)}")

    print("\nWHY THIS IS CULTURALLY APPROPRIATE")
    print("-" * 72)
    print(" - Evidence is restricted to the SAME architectural tradition.")
    print(" - Wrong-tradition 'analogues' (e.g. a Buddhist stupa for a Hindu")
    print("   temple tower) are refused, not substituted.")
    print(" - Missing references are named as a data-acquisition target, so the")
    print("   system fails honestly instead of hallucinating a reconstruction.")

restoration_plan_v4("Konark Sun Temple Odisha")

---
# Harder evaluation — real headroom

The v3 eval hit 1.000 because the queries named the sites. This set **describes**
structures without naming them — forcing genuine semantic retrieval. Expect a
score well below 1.0; that gap is what future improvements must close. Keep this
set fixed from now on so versions stay comparable.

In [ ]:
# ============================================================
# 17) HARDER EVAL (no name leakage) + log a true baseline
# ============================================================
EVAL_HARD = [
    ("thirteenth century stone temple built as a colossal chariot with carved wheels and horses, eastern india", ["sun temple","konâ","konar"]),
    ("white marble domed mausoleum a mughal emperor built for his beloved wife",        ["taj mahal"]),
    ("abandoned red sandstone imperial capital deserted soon after it was built",       ["fatehpur sikri"]),
    ("rock cut cave complex with a giant monolithic temple carved from one rock",       ["ellora"]),
    ("ancient buddhist cave monasteries famous for vivid murals and wall paintings",    ["ajanta"]),
    ("ruined capital of a fallen southern empire with boulder strewn temples and bazaars", ["hampi"]),
    ("hilltop desert fortresses of warrior kings in northwestern india",                ["hill forts of rajasthan"]),
    ("towering brick and sandstone victory minaret beside an early mosque in the capital", ["qutb"]),
    ("medieval temples renowned for intricate sensual stone carvings in central india", ["khajuraho"]),
    ("grand granite temples of a powerful southern dynasty with a towering sanctum tower", ["chola"]),
    ("great hemispherical buddhist relic mound with carved ceremonial gateways",        ["sanchi"]),
    ("garden tomb of an early mughal ruler that inspired later mausoleums",             ["humayun"]),
    ("scenic mountain railway climbing steep hills in the colonial era",               ["mountain railways"]),
    ("baroque churches and convents of a former portuguese colony on the west coast",   ["goa"]),
    ("shore temple and monolithic rock carvings of a southern coastal town",            ["mahabalipuram","mamallapuram"]),
]

def evaluate2(eval_set, search_fn, k_values=(5, 10)):
    maxk = max(k_values); hits = {k: 0 for k in k_values}; rr = 0.0; misses = []
    for query, gold in eval_set:
        names = [n.lower() for n in search_fn(query, top_k=maxk)["name"].tolist()]
        rank = next((i+1 for i, nm in enumerate(names)
                     if any(g in nm for g in gold)), None)
        for k in k_values:
            if rank and rank <= k: hits[k] += 1
        rr += (1.0/rank) if rank else 0.0
        if not rank: misses.append((query, gold))
    n = len(eval_set)
    print("HARDER RETRIEVAL QUALITY (n =", n, ")"); print("-"*50)
    for k in k_values: print(f"  Recall@{k:<2d} : {hits[k]/n:.3f}  ({hits[k]}/{n})")
    print(f"  MRR      : {rr/n:.3f}")
    if misses:
        print("\n  Misses:")
        for q, g in misses: print(f"    - '{q[:60]}...'  expected {g}")
    return {"recall": {k: hits[k]/n for k in k_values}, "mrr": rr/n}

hard_metrics = evaluate2(EVAL_HARD, tangible_search)

history_df = log_metrics(
    version="v4-tradition-hardeval",
    metrics=hard_metrics,
    n_records=len(heritage_master),
    notes="HARDER eval (descriptions, no name leak) + cultural tradition layer + tangible sub-index",
)
print("\nMETRICS HISTORY")
print(history_df[["version","n_records","recall@5","recall@10","mrr"]].to_string(index=False))

---
# v5 — Cultural Grammar Layer ("complete it, but obey the culture")

Design thesis: **completion is the goal, but the method must obey the belief
system and architectural canon of the structure's own culture.**

A tradition's canon (proportional rules, sacred geometry, materials, motifs,
governing treatise) is encoded as a **cultural grammar**. Then `complete_structure`
produces a completion specification where every missing component is resolved by:

1. a same-tradition **exemplar** from the knowledge base (if one exists), AND
2. the tradition's **codified grammar rules** (always).

This means even Konark's collapsed tower — with no exemplar in the KB — can be
completed faithfully **from the Kalinga canon**, not from a wrong-tradition
building and not from invention. Grammar rules below are v1 and need review by a
heritage/Shilpa-shastra specialist before any real-world use.

In [ ]:
# ============================================================
# 18) CULTURAL GRAMMAR  (the "belief system" as code)
#     Hand-authored canon per tradition. Cite the treatise.
#     v1 — REQUIRES expert validation before real restoration use.
# ============================================================
CULTURAL_GRAMMAR = {
    "Kalinga (Odisha temple)": {
        "treatise": "Shilpa Prakasha; Bhubaneswar temple canon",
        "form_language": "rekha deul — a tall curvilinear tower with vertical "
                         "projections (pagas), crowned by amalaka and kalasha, "
                         "fronted by a pyramidal pidha-roofed jagamohana",
        "proportion_rules": [
            "deul (main tower) height roughly 2x the jagamohana height",
            "plan follows pancharatha/saptaratha (5 or 7 vertical offsets)",
            "vertical division into bada (wall), gandi (tower), mastaka (crown)",
        ],
        "materials": ["khondalite", "laterite", "chlorite (for sculpture)"],
        "motifs": ["gavaksha / chaitya-window ornament", "naga pilasters",
                   "presiding deity in the raha niche on each face"],
    },
    "Dravidian (South Indian temple)": {
        "treatise": "Mayamata; Manasara shilpa-shastra",
        "form_language": "vimana — a stepped pyramidal tower of diminishing "
                         "storeys (talas) over the sanctum, capped by a domical "
                         "shikhara; later complexes add taller gopuram gateways",
        "proportion_rules": [
            "square garbhagriha; vimana storeys diminish by a fixed ratio",
            "gopuram (in mature period) exceeds the vimana in height",
            "axial mandapa sequence on an east-west processional axis",
        ],
        "materials": ["granite", "later brick-and-stucco gopurams"],
        "motifs": ["kudu (horseshoe) arches", "yali / rearing-lion pillars",
                   "dvarapala guardians at entrances"],
    },
    "Nagara (North Indian temple)": {
        "treatise": "Samarangana Sutradhara",
        "form_language": "latina/shekhari curvilinear shikhara over the sanctum, "
                         "crowned by amalaka and kalasha, preceded by mandapas",
        "proportion_rules": [
            "shikhara height approx 2x the width of the garbhagriha",
            "sanctum square in plan; vertical bands of gavaksha mesh",
        ],
        "materials": ["sandstone", "marble (regional)"],
        "motifs": ["gavaksha honeycomb", "surasundari figures", "kirtimukha"],
    },
    "Indo-Islamic / Mughal": {
        "treatise": "Timurid-Mughal charbagh and tomb canon",
        "form_language": "bilaterally symmetric mass with a central double dome on "
                         "a drum, recessed pishtaq/iwan portals, corner minarets, "
                         "set within a quadripartite char-bagh garden",
        "proportion_rules": [
            "primary facade roughly square (height approx equal to width)",
            "strict bilateral symmetry about the central axis",
            "double dome: outer profile bulbous, inner shell lower",
        ],
        "materials": ["red sandstone", "white marble with pietra-dura inlay"],
        "motifs": ["cusped/pointed arches", "jali lattice screens",
                   "calligraphic and floral inlay bands"],
    },
}
# stubs to show extensibility (grammar not yet authored)
for trad in ["Khmer", "Greco-Roman classical", "Mesoamerican"]:
    CULTURAL_GRAMMAR.setdefault(trad, {"treatise": "TODO",
        "form_language": "", "proportion_rules": [], "materials": [], "motifs": []})

print("Grammars authored for:")
for k, v in CULTURAL_GRAMMAR.items():
    state = "authored" if v["proportion_rules"] else "stub (needs authoring)"
    print(f"  - {k:34s} [{state}]")

In [ ]:
# ============================================================
# 19) complete_structure() — culturally-governed completion
# ============================================================
def complete_structure(target_hint, location="India"):
    print("=" * 74)
    print("CULTURALLY-GOVERNED COMPLETION:", target_hint)
    print("=" * 74)

    match = tangible_search(f"{target_hint} {location}", top_k=1)
    if not len(match):
        print("No tangible record found."); return
    target    = match.iloc[0]["name"]
    tradition = match.iloc[0]["tradition"]
    rec   = heritage_master[heritage_master["name"] == target].iloc[0]
    arch  = infer_archetype(rec["name"], rec["type"], rec["description"], rec["region"])
    canon = component_ontology[arch]
    gram  = CULTURAL_GRAMMAR.get(tradition)

    missing = {}
    for k, v in KNOWN_MISSING.items():
        if k.lower() in target.lower() or target.lower() in k.lower():
            missing = v
    missing_set = set(missing)

    print(f"Structure : {target}")
    print(f"Tradition : {tradition}")
    print(f"Archetype : {arch}")
    if gram and gram.get("proportion_rules"):
        print(f"Governing canon : {gram['treatise']}")
        print(f"Form language   : {gram['form_language']}")
    else:
        print("Governing canon : NOT YET AUTHORED for this tradition "
              "(completion cannot be culturally grounded)")
        return

    todo = list(missing_set) if missing_set else canon
    print("\nCOMPLETION SPECIFICATION (each part governed by the canon)")
    print("-" * 74)
    for c in todo:
        ev = tangible_search(f"{tradition} {c} intact complete", top_k=3, tradition=tradition)
        ev = ev[ev["name"] != target]
        print(f"\n  COMPONENT: {c}")
        # 1) governing rule from the cultural grammar (always available)
        rule = next((r for r in gram["proportion_rules"]
                     if any(w in c.lower() for w in r.lower().split()[:3])),
                    gram["proportion_rules"][0])
        print(f"    Canon rule : {rule}")
        print(f"    Material   : {', '.join(gram['materials'][:2])}")
        print(f"    Motifs     : {', '.join(gram['motifs'][:2])}")
        # 2) exemplar evidence if the KB has a same-tradition structure
        if len(ev):
            e = ev.iloc[0]
            print(f"    Exemplar   : {e['name']} [{e['source']}] sim={e['score']:.3f}")
            print(f"    Confidence : {_confidence(e['score'])} (canon + same-tradition exemplar)")
        else:
            refs = KNOWN_REFERENCES.get(tradition, [])
            print(f"    Exemplar   : none in KB — completed from CANON ALONE")
            if refs:
                print(f"                 (acquire {refs[0]} et al. to raise confidence)")
            print(f"    Confidence : MEDIUM (canon-grounded, exemplar-unconfirmed)")

    print("\nPRINCIPLE")
    print("-" * 74)
    print(" The structure is completed by any available means, BUT every choice is")
    print(" governed by its own tradition's canon. No wrong-tradition substitution,")
    print(" no free invention. Canon-only completions are flagged for lower confidence")
    print(" and named data-acquisition targets.")

complete_structure("Konark Sun Temple Odisha")

---
# v6 — World Tradition Taxonomy (open-world cultural classification)

To work for **any structure on Earth**, the tradition tag can't be India-only
keyword rules. Here we define a taxonomy of ~45 world architectural traditions
(every inhabited continent) with rich descriptions, embed them with the same
BGE-M3 model, and classify any structure by **nearest tradition in embedding
space**. No keywords, no region hard-coding — it generalises to structures we've
never seen.

This is Phase 1 of going world-scale. Phase 2 (retrieval-augmented cultural
grammar from cited sources) is what lets completion work for traditions we
haven't hand-authored.

In [ ]:
# ============================================================
# 20) WORLD ARCHITECTURAL TRADITION TAXONOMY
# ============================================================
WORLD_TRADITIONS = [
    # --- South Asia ---
    ("Kalinga (Odisha temple)", "Curvilinear rekha-deul Hindu temple towers of Odisha eastern India, Shilpa Prakasha canon, Konark Lingaraja."),
    ("Dravidian (South Indian temple)", "South Indian Hindu temples with pyramidal vimana and tall gopuram gateway towers, granite, Chola Pallava Tamil."),
    ("Nagara (North Indian temple)", "North Indian Hindu temples with curvilinear latina shikhara spire, amalaka finial, Khajuraho sandstone."),
    ("Indo-Islamic / Mughal", "Mughal and sultanate tombs and mosques in India, domes, char-bagh gardens, minarets, red sandstone and marble, Taj Mahal."),
    ("Indian Buddhist / rock-cut", "Indian rock-cut Buddhist and Hindu cave temples, stupas, chaitya halls and viharas, Ajanta Ellora Sanchi."),
    ("Indian fort architecture", "Indian hill and sea forts, ramparts, bastions, gateways, citadels, Rajput Maratha defensive military architecture."),
    ("Sinhalese (Sri Lanka)", "Sri Lankan Buddhist stupas dagobas, monastic complexes and tanks, Anuradhapura Polonnaruwa Sigiriya."),
    # --- Southeast & East Asia ---
    ("Khmer (Angkor)", "Cambodian Khmer temple-mountains with quincunx towers, sandstone galleries and moats, Angkor Wat Bayon."),
    ("Javanese (Indonesia)", "Indonesian Hindu-Buddhist stone temples candi, terraced stupas and reliefs, Borobudur Prambanan Java."),
    ("Burmese (Bagan)", "Burmese Buddhist brick temples and gilded stupas with bell-shaped zedi, Bagan plain Myanmar."),
    ("Thai", "Thai Buddhist temple wat with tiered roofs, prang towers and chedi stupas, gilded ornament Ayutthaya Sukhothai."),
    ("Chinese imperial", "Chinese timber-frame palaces and temples with bracket sets dougong, tiled hip roofs, courtyards, Forbidden City."),
    ("Japanese", "Japanese Buddhist and Shinto timber temples, pagodas and shrines with deep eaves and joinery, Kyoto Nara."),
    ("Korean", "Korean Buddhist temples and palaces, timber-frame with dancheong painted brackets and tiled roofs, Gyeongbokgung."),
    ("Tibetan / Himalayan", "Tibetan Buddhist monasteries and dzong fortresses, battered stone walls, gompa, Himalayan Bhutan Ladakh."),
    # --- Middle East & Central Asia ---
    ("Persian (Safavid / Timurid)", "Persian Islamic mosques and madrasas with iwans, bulbous tiled domes and muqarnas, Isfahan Samarkand."),
    ("Ottoman", "Ottoman Turkish mosques with central domes, semi-domes and pencil minarets, Sinan Istanbul Hagia Sophia heritage."),
    ("Umayyad / Abbasid Islamic", "Early Islamic Arab mosques with hypostyle halls, horseshoe arches and courtyards, Damascus Kairouan."),
    ("Moorish / Andalusian", "Western Islamic palaces and mosques of Iberia and Maghreb, horseshoe arches, stucco and tilework, Alhambra Cordoba."),
    ("Mamluk", "Egyptian and Levantine Mamluk mosques and madrasas with stone-carved domes and striped ablaq masonry, Cairo."),
    ("Achaemenid / Persian ancient", "Ancient Persian ceremonial palaces with columned apadana halls and reliefs, Persepolis Pasargadae."),
    ("Mesopotamian / Assyrian", "Ancient Mesopotamian mud-brick ziggurats, temple platforms and palace reliefs, Babylon Ur Nimrud."),
    # --- Mediterranean & Europe ---
    ("Ancient Egyptian", "Ancient Egyptian stone pyramids, mortuary and cult temples with pylons, hypostyle halls and obelisks, Giza Karnak."),
    ("Greco-Roman classical", "Classical Greek and Roman temples, theatres and forums with columns, pediments, arches and domes, Acropolis Colosseum."),
    ("Byzantine", "Byzantine Orthodox churches with central domes on pendentives, mosaics and Greek-cross plans, Hagia Sophia Ravenna."),
    ("Romanesque", "European Romanesque churches with rounded arches, thick walls, barrel vaults and towers, eleventh twelfth century."),
    ("Gothic", "European Gothic cathedrals with pointed arches, ribbed vaults, flying buttresses and stained glass, Notre-Dame Chartres."),
    ("Renaissance", "Italian Renaissance churches and palazzi with classical orders, symmetry, domes and proportion, Florence Rome."),
    ("Baroque", "European Baroque churches and palaces with dramatic curves, ornament and grand domes, seventeenth century Bernini."),
    ("Russian Orthodox", "Russian Orthodox churches with onion domes, tented roofs and bright tilework, Moscow Kizhi Saint Basil."),
    ("Romano-British / Medieval Castle", "European medieval castles and fortifications with keeps, curtain walls, towers and moats, Norman crusader."),
    # --- Africa ---
    ("Aksumite (Ethiopian)", "Ethiopian carved stone stelae and rock-hewn monolithic churches, Aksum Lalibela highlands."),
    ("Sudano-Sahelian (West African)", "West African mud-brick mosques and palaces with timber toron beams and tapering buttresses, Djenne Timbuktu."),
    ("Swahili coast", "East African Swahili coral-stone towns, mosques and houses with carved doors, Kilwa Lamu Zanzibar."),
    ("Great Zimbabwe", "Southern African dry-stone walled enclosures and conical towers without mortar, Great Zimbabwe."),
    ("Nubian", "Ancient Nubian steep-sided pyramids and temples of Kush along the Nile, Meroe Sudan."),
    # --- Americas ---
    ("Maya", "Mesoamerican Maya stepped limestone pyramids, temples, ball courts and roof combs with glyphs, Tikal Chichen Itza Palenque."),
    ("Aztec / Teotihuacan", "Central Mexican stepped pyramids and ceremonial avenues, twin temples and platforms, Teotihuacan Tenochtitlan."),
    ("Andean (Inca)", "Andean Inca dry-fitted polygonal stone walls, terraces and trapezoidal niches, Machu Picchu Cusco Sacsayhuaman."),
    ("Puebloan (Ancestral)", "Ancestral Puebloan adobe and sandstone cliff dwellings and kivas of the American Southwest, Mesa Verde Chaco."),
    ("Colonial Iberian (Americas)", "Spanish and Portuguese colonial churches and convents in the Americas, baroque facades, Goa Latin America."),
    # --- Modern / cross-cutting ---
    ("Modernist / 20th-century", "Twentieth-century modernist concrete steel and glass buildings, functional form, Le Corbusier Bauhaus heritage."),
    ("Vernacular / earthen", "Local vernacular earthen, timber and thatch dwellings and granaries adapted to climate and materials."),
    ("Industrial heritage", "Industrial-era mines, mills, railways, canals and engineering works of the modern period."),
]

trad_names = [t for t, _ in WORLD_TRADITIONS]
trad_texts = [f"{t}. {d}" for t, d in WORLD_TRADITIONS]
trad_emb   = encoder.encode(trad_texts, normalize_embeddings=True,
                            convert_to_numpy=True).astype("float32")
print("World traditions:", len(trad_names))

In [ ]:
# ============================================================
# 21) EMBEDDING-BASED TRADITION CLASSIFIER (open-world)
# ============================================================
def classify_tradition(text, top_k=3):
    """Nearest world traditions for any free-text structure description."""
    q = encoder.encode([text], normalize_embeddings=True).astype("float32")
    s = (q @ trad_emb.T)[0]
    order = s.argsort()[::-1][:top_k]
    return [(trad_names[i], round(float(s[i]), 3)) for i in order]

# re-tag EVERY record using the global classifier (vectorised, reuses embeddings)
sims = embeddings @ trad_emb.T          # (n_records, n_traditions)
best = sims.argmax(1)
heritage_master["tradition_global"] = [trad_names[i] for i in best]
heritage_master["tradition_conf"]   = sims.max(1).round(3)

print("Top global traditions in the knowledge base:")
print(heritage_master["tradition_global"].value_counts().head(15).to_string())

print("\nSpot-check on diverse world structures (none India-specific rules):")
for probe in [
    "Angkor Wat Cambodia stone temple towers and moat",
    "Notre-Dame de Paris Gothic cathedral pointed arches",
    "Chichen Itza Maya stepped pyramid Mexico",
    "Machu Picchu Inca dry stone walls Peru",
    "Great Mosque of Djenne mud brick Mali",
    "Forbidden City Beijing imperial palace",
    "Sun Temple Konark Odisha India",
]:
    print(f"  {probe[:42]:44s} -> {classify_tradition(probe, top_k=2)}")

---
# v7 — Phase 2: Multi-Source Cultural Grammar Engine

This is the scaler that makes completion work for **any tradition on Earth**, not
just the four we hand-authored. For any tradition it assembles a canon corpus
from every available source, in trust order:

- **Tier 0** hand-authored `CULTURAL_GRAMMAR` (highest trust)
- **Tier 1** curated `.txt` corpus you attach as a Kaggle dataset (auto-detected)
- **Tier 2** Wikipedia architecture article (free, always on, fully cited)
- **Tier 3** optional LLM distillation of the passages (needs `OPENAI_API_KEY`; skipped if absent)

Every canon passage carries its source, citation and trust tier. Machine-sourced
material is flagged — nothing is presented as validated scholarship that isn't.

In [ ]:
# ============================================================
# 22) MULTI-SOURCE CANON ENGINE
# ============================================================
import urllib.parse, urllib.request

# tradition -> Wikipedia article title
TRADITION_WIKI = {
    "Kalinga (Odisha temple)": "Kalinga architecture",
    "Dravidian (South Indian temple)": "Dravidian architecture",
    "Nagara (North Indian temple)": "Nagara architecture",
    "Indo-Islamic / Mughal": "Mughal architecture",
    "Indian Buddhist / rock-cut": "Indian rock-cut architecture",
    "Indian fort architecture": "Fortifications of India",
    "Sinhalese (Sri Lanka)": "Sinhalese architecture",
    "Khmer (Angkor)": "Khmer architecture",
    "Javanese (Indonesia)": "Candi of Indonesia",
    "Burmese (Bagan)": "Burmese architecture",
    "Thai": "Thai temple architecture",
    "Chinese imperial": "Chinese architecture",
    "Japanese": "Japanese architecture",
    "Korean": "Korean architecture",
    "Tibetan / Himalayan": "Tibetan architecture",
    "Persian (Safavid / Timurid)": "Iranian architecture",
    "Ottoman": "Ottoman architecture",
    "Umayyad / Abbasid Islamic": "Islamic architecture",
    "Moorish / Andalusian": "Moorish architecture",
    "Mamluk": "Mamluk architecture",
    "Achaemenid / Persian ancient": "Achaemenid architecture",
    "Mesopotamian / Assyrian": "Architecture of Mesopotamia",
    "Ancient Egyptian": "Ancient Egyptian architecture",
    "Greco-Roman classical": "Ancient Roman architecture",
    "Byzantine": "Byzantine architecture",
    "Romanesque": "Romanesque architecture",
    "Gothic": "Gothic architecture",
    "Renaissance": "Renaissance architecture",
    "Baroque": "Baroque architecture",
    "Russian Orthodox": "Russian Revival architecture",
    "Romano-British / Medieval Castle": "Castle",
    "Aksumite (Ethiopian)": "Aksumite architecture",
    "Sudano-Sahelian (West African)": "Sudano-Sahelian architecture",
    "Swahili coast": "Swahili architecture",
    "Great Zimbabwe": "Great Zimbabwe",
    "Nubian": "Nubian pyramids",
    "Maya": "Maya architecture",
    "Aztec / Teotihuacan": "Mesoamerican architecture",
    "Andean (Inca)": "Inca architecture",
    "Puebloan (Ancestral)": "Ancestral Puebloans",
    "Colonial Iberian (Americas)": "Spanish Colonial architecture",
    "Modernist / 20th-century": "Modern architecture",
    "Vernacular / earthen": "Vernacular architecture",
    "Industrial heritage": "Industrial architecture",
}

def fetch_wikipedia(title, timeout=15):
    """Plain-text extract of a Wikipedia article, or None on any failure."""
    try:
        base = "https://en.wikipedia.org/w/api.php?"
        params = urllib.parse.urlencode({
            "format": "json", "action": "query", "prop": "extracts",
            "explaintext": 1, "redirects": 1, "titles": title})
        req = urllib.request.Request(base + params,
              headers={"User-Agent": "HeritageFormer/1.0 (research)"})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            data = json.loads(r.read().decode())
        pages = data["query"]["pages"]
        page = next(iter(pages.values()))
        return page.get("extract") or None
    except Exception as e:
        print("   (wiki fetch failed for", title, "->", e, ")")
        return None

def _chunks(text, lo=140, hi=600):
    out = []
    for para in (text or "").split("\n"):
        p = para.strip()
        if lo <= len(p) <= hi:
            out.append(p)
    return out[:40]

# optional LLM tier (silently off without a key)
_LLM = None
try:
    import os
    if os.getenv("OPENAI_API_KEY"):
        from openai import OpenAI
        _LLM = OpenAI()
        print("LLM distillation tier: ENABLED")
except Exception:
    _LLM = None
if _LLM is None:
    print("LLM distillation tier: off (no OPENAI_API_KEY)")

# curated corpus auto-detect
CURATED_TXT = glob.glob("/kaggle/input/**/*.txt", recursive=True)

_canon_cache = {}   # tradition -> list[(text, source, url, tier)]
def _build_corpus(tradition):
    if tradition in _canon_cache:
        return _canon_cache[tradition]
    passages = []
    # Tier 0 — hand-authored
    g = CULTURAL_GRAMMAR.get(tradition)
    if g and g.get("proportion_rules"):
        for r in g["proportion_rules"]:
            passages.append((r, f"hand-authored canon ({g['treatise']})", "", "T0"))
    # Tier 1 — curated txt matching the tradition's key words
    key = tradition.split("(")[0].strip().lower().split("/")[0].strip()
    for p in CURATED_TXT:
        if key and key.split()[0] in p.lower():
            try:
                txt = open(p, encoding="utf-8", errors="ignore").read()
                for c in _chunks(txt):
                    passages.append((c, f"curated:{os.path.basename(p)}", p, "T1"))
            except Exception:
                pass
    # Tier 2 — Wikipedia
    title = TRADITION_WIKI.get(tradition)
    if title:
        ext = fetch_wikipedia(title)
        url = "https://en.wikipedia.org/wiki/" + urllib.parse.quote(title.replace(" ", "_"))
        for c in _chunks(ext):
            passages.append((c, f"Wikipedia: {title}", url, "T2"))
    _canon_cache[tradition] = passages
    return passages

def get_canon(tradition, query, k=3):
    """Top-k canon passages for a tradition+component query, with citations."""
    corpus = _build_corpus(tradition)
    if not corpus:
        return []
    texts = [c[0] for c in corpus]
    emb = encoder.encode(texts, normalize_embeddings=True,
                         convert_to_numpy=True).astype("float32")
    q = encoder.encode([query], normalize_embeddings=True).astype("float32")
    s = (q @ emb.T)[0]
    out = []
    for i in s.argsort()[::-1][:k]:
        t, src, url, tier = corpus[i]
        out.append({"text": t, "source": src, "url": url,
                    "tier": tier, "score": round(float(s[i]), 3)})
    return out

print("Canon engine ready. Curated .txt files found:", len(CURATED_TXT))

In [ ]:
# ============================================================
# 23) complete_structure_v3 — world-scale, multi-source canon
# ============================================================
TRUST = {"T0": "hand-authored", "T1": "curated", "T2": "Wikipedia (cited)",
         "T3": "LLM-distilled (flagged)"}

def complete_structure_v3(target_hint, location=""):
    print("=" * 76)
    print("WORLD-SCALE CULTURALLY-GOVERNED COMPLETION:", target_hint)
    print("=" * 76)

    match = tangible_search(f"{target_hint} {location}".strip(), top_k=1)
    if not len(match):
        print("No tangible record found."); return
    target    = match.iloc[0]["name"]
    rec = heritage_master[heritage_master["name"] == target].iloc[0]
    tradition = rec["tradition_global"]                 # open-world classifier
    arch  = infer_archetype(rec["name"], rec["type"], rec["description"], rec["region"])
    canon_parts = component_ontology[arch]

    missing = {}
    for k, v in KNOWN_MISSING.items():
        if k.lower() in target.lower() or target.lower() in k.lower():
            missing = v
    todo = list(missing) if missing else canon_parts

    print(f"Structure : {target}")
    print(f"Tradition : {tradition}  (open-world classifier)")
    print(f"Archetype : {arch}")

    for c in todo:
        print(f"\n  COMPONENT: {c}" + ("   [DOCUMENTED LOSS]" if c in missing else ""))
        passages = get_canon(tradition, f"{c} form proportion construction", k=2)
        if not passages:
            print("    [NO CANON SOURCE] tradition has no authored/curated/wiki canon.")
            continue
        for p in passages:
            print(f"    Canon [{TRUST[p['tier']]}]: {p['text'][:200]}")
            if p["url"]:
                print(f"           source: {p['source']}  <{p['url']}>")
            else:
                print(f"           source: {p['source']}")
        # same-tradition exemplar (optional, raises confidence)
        ev = tangible_search(f"{tradition} {c}", top_k=3, tradition=tradition)
        ev = ev[ev["name"] != target]
        if len(ev):
            e = ev.iloc[0]
            print(f"    Exemplar: {e['name']} [{e['source']}] sim={e['score']:.3f}"
                  f"  -> confidence HIGH")
        else:
            print(f"    Exemplar: none in KB -> confidence MEDIUM (canon-only)")

    print("\n" + "-" * 76)
    print("Every line above is sourced and tier-flagged. No wrong-tradition")
    print("substitution, no unsourced invention. This works for ANY tradition")
    print("with a canon source, not only the hand-authored ones.")

# Indian (hand-authored canon) ...
complete_structure_v3("Konark Sun Temple", "Odisha India")
# ... and a tradition we did NOT hand-author (proves world-scale)
complete_structure_v3("Angkor Wat", "Cambodia")

---
# v8 — Fixes (Wikipedia tier, tradition selection, world components)

This run exposed three bugs. This section fixes all of them without re-embedding:

1. **Wikipedia fetch failed** (`json` not imported) — fixed here; cache cleared so it refetches.
2. **Tradition mis-selection** — prefer the reliable label, only fall back to the
   embedding classifier for traditions it doesn't cover. Adds a confidence floor
   so weak guesses become "unspecified" instead of junk like "Industrial heritage".
3. **India-only components** — add a world component map so non-Indian structures
   (Khmer, Gothic, Maya…) get their OWN canonical parts, not Indian temple parts.

Reminder: enable the **T4 GPU** (Settings -> Accelerator) — CPU embedding took 37 min.

In [ ]:
# ============================================================
# 24) FIX 1 — make the Wikipedia tier work, refetch canon
# ============================================================
import json                      # was only imported as _json -> wiki fetch crashed
_canon_cache.clear()             # drop the empty (failed-fetch) corpora

# confidence floor on the open-world classifier (kills "Industrial heritage" spam)
sims = embeddings @ trad_emb.T
best = sims.argmax(1); conf = sims.max(1)
heritage_master["tradition_global"] = [
    trad_names[b] if conf[i] >= 0.35 else "unspecified tradition"
    for i, b in enumerate(best)]
heritage_master["tradition_conf"] = conf.round(3)
print("Re-tagged with confidence floor. Top traditions now:")
print(heritage_master[heritage_master["tradition_global"] != "unspecified tradition"]
      ["tradition_global"].value_counts().head(12).to_string())

# quick check that Wikipedia now returns text
_test = fetch_wikipedia("Khmer architecture")
print("\nWiki fetch test (Khmer architecture):",
      "OK, %d chars" % len(_test) if _test else "STILL FAILING")

In [ ]:
# ============================================================
# 25) FIX 2 — reliable tradition selection
# ============================================================
WORLD_NAMES = set(trad_names)

def best_tradition(rec):
    """Trust the keyword label when it is a known world tradition;
    otherwise use the embedding classifier."""
    kw = str(rec.get("tradition", ""))
    if kw in WORLD_NAMES:                      # specific & reliable (e.g. Kalinga)
        return kw
    tg = str(rec.get("tradition_global", ""))
    return tg if tg and tg != "unspecified tradition" else (kw or "unspecified tradition")

# ============================================================
# FIX 3 — world component map (non-Indian structures get their OWN parts)
# ============================================================
TRADITION_COMPONENTS = {
    "Khmer (Angkor)": [
        "central sanctuary tower (prasat)", "quincunx of five towers",
        "enclosed galleries with corbelled vaults", "gopura (entrance pavilion)",
        "library annexes", "moat and baray reservoir", "naga causeway balustrade"],
    "Gothic": [
        "pointed-arch nave arcade", "ribbed vault", "flying buttress",
        "rose window and stained glass", "west front with twin towers",
        "spire", "ambulatory with radiating chapels"],
    "Maya": [
        "stepped pyramid platform", "temple shrine with roof comb",
        "corbel-vaulted chambers", "monumental stairway",
        "stelae and carved lintels", "ball court"],
    "Greco-Roman classical": [
        "colonnade of the classical orders", "pediment", "entablature",
        "cella / naos", "podium or stylobate", "dome or barrel vault (Roman)"],
    "Chinese imperial": [
        "raised platform terrace", "timber post-and-beam frame",
        "dougong bracket sets", "tiled hip-and-gable roof",
        "courtyard enclosure", "ceremonial axial gateway"],
    "Ancient Egyptian": [
        "pylon gateway", "hypostyle hall of columns", "obelisk",
        "sanctuary", "enclosure wall", "axial processional way"],
    "Ottoman": [
        "central dome", "semi-domes and exedrae", "pencil minarets",
        "arcaded courtyard (avlu)", "mihrab", "ablution fountain"],
    "Indo-Islamic / Mughal": [
        "double dome on a drum", "pishtaq / iwan portal", "corner minarets",
        "char-bagh garden", "raised plinth", "jali screens"],
}

def components_for(tradition, archetype):
    if tradition in TRADITION_COMPONENTS:
        return TRADITION_COMPONENTS[tradition]
    return component_ontology.get(archetype, component_ontology["generic_monument"])

print("World component maps:", list(TRADITION_COMPONENTS))

In [ ]:
# ============================================================
# 26) complete_structure_v4 — fixed world-scale completion
# ============================================================
def complete_structure_v4(target_hint, location=""):
    print("=" * 76)
    print("WORLD-SCALE CULTURALLY-GOVERNED COMPLETION:", target_hint)
    print("=" * 76)
    match = tangible_search(f"{target_hint} {location}".strip(), top_k=1)
    if not len(match):
        print("No tangible record found."); return
    target = match.iloc[0]["name"]
    rec = heritage_master[heritage_master["name"] == target].iloc[0]
    tradition = best_tradition(rec)
    arch  = infer_archetype(rec["name"], rec["type"], rec["description"], rec["region"])
    parts = components_for(tradition, arch)

    missing = {}
    for k, v in KNOWN_MISSING.items():
        if k.lower() in target.lower() or target.lower() in k.lower():
            missing = v
    todo = list(missing) if missing else parts

    print(f"Structure : {target}")
    print(f"Tradition : {tradition}")
    print(f"Components: {len(parts)} canonical parts for this tradition")

    for c in todo:
        print(f"\n  COMPONENT: {c}" + ("   [DOCUMENTED LOSS]" if c in missing else ""))
        passages = get_canon(tradition, f"{c} form proportion construction", k=2)
        if passages:
            for p in passages:
                print(f"    Canon [{TRUST[p['tier']]}]: {p['text'][:190]}")
                print(f"           source: {p['source']}"
                      + (f"  <{p['url']}>" if p["url"] else ""))
        else:
            print("    [NO CANON SOURCE] no authored/curated/wiki canon for this tradition.")
        ev = tangible_search(f"{tradition} {c}", top_k=3, tradition=tradition)
        ev = ev[ev["name"] != target]
        if len(ev):
            e = ev.iloc[0]
            print(f"    Exemplar: {e['name']} [{e['source']}] sim={e['score']:.3f}  -> HIGH")
        else:
            print(f"    Exemplar: none of this tradition in KB -> MEDIUM (canon-only)")

    print("\n" + "-" * 76)
    print("Same-tradition only. Sourced + tier-flagged. Works for any tradition")
    print("with a canon source — Indian or not.")

complete_structure_v4("Konark Sun Temple", "Odisha India")   # -> Kalinga (fixed)
complete_structure_v4("Angkor Wat", "Cambodia")              # -> Khmer parts + wiki canon
complete_structure_v4("Notre-Dame Paris", "France")          # -> Gothic, never hand-authored

---
# v9 — RECONSTRUCTION MODELLING (procedural geometry generation)

Not retrieval. Not classification. This **generates the 3D geometry of the
missing/defining component** from the cultural canon's proportional rules — the
shape-grammar method real heritage reconstruction uses. Output is an actual mesh
(rendered + exported as .obj). Runs on CPU, no training.

Konark's collapsed deul is a *rekha-deul*: a parametric curvilinear tower. Given
the canon (height = 2x jagamohana, pancharatha plan, rekha curve, amalaka+kalasha)
we reconstruct its geometry. Dome and stepped-pyramid generators are included so
the same engine reconstructs Mughal/Ottoman and Maya/Mesopotamian structures too.

These are parametric models consistent with the canon and surviving proportions —
calibrate parameters to measured evidence before any real-world use.

In [ ]:
# ============================================================
# 27) GEOMETRY ENGINE — mesh accumulator + procedural generators
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

class Mesh:
    def __init__(self): self.V=[]; self.F=[]
    def add_surface(self, X, Y, Z):
        nu, nv = X.shape; base = len(self.V)
        for i in range(nu):
            for j in range(nv):
                self.V.append((float(X[i,j]), float(Y[i,j]), float(Z[i,j])))
        for i in range(nu-1):
            for j in range(nv-1):
                a=base+i*nv+j; b=base+(i+1)*nv+j; c=base+(i+1)*nv+j+1; d=base+i*nv+j+1
                self.F += [(a,b,c), (a,c,d)]
    def arrays(self):
        V=np.array(self.V); F=np.array(self.F)
        return V[:,0], V[:,1], V[:,2], F
    def obj(self):
        L=[f"v {x:.4f} {y:.4f} {z:.4f}" for x,y,z in self.V]
        L+=[f"f {a+1} {b+1} {c+1}" for a,b,c in self.F]
        return "\n".join(L)

def _torus(R, a, z0, nu=20, nv=48):
    u=np.linspace(0,2*np.pi,nu); v=np.linspace(0,2*np.pi,nv)
    U,Vv=np.meshgrid(u,v,indexing="ij")
    X=(R+a*np.cos(U))*np.cos(Vv); Y=(R+a*np.cos(U))*np.sin(Vv); Z=z0+a*np.sin(U)
    return X,Y,Z

def gen_shikhara(H=70.0, base=18.0, n_paga=5, curve=2.2, amp=0.12, nu=64, nv=80):
    """Curvilinear rekha-deul / nagara temple tower."""
    m=Mesh()
    t=np.linspace(0,1,nu); z=H*t
    r=(base/2)*(1-t**curve); r=np.clip(r,0.04*base,None)          # rekha taper
    th=np.linspace(0,2*np.pi,nv)
    star=1+amp*np.cos(n_paga*th)                                  # pancharatha pagas
    X=(r[:,None]*star[None,:])*np.cos(th)[None,:]
    Y=(r[:,None]*star[None,:])*np.sin(th)[None,:]
    Z=z[:,None]*np.ones(nv)[None,:]
    m.add_surface(X,Y,Z)
    m.add_surface(*_torus(0.20*base, 0.07*base, H*0.985))         # amalaka disc
    # kalasha finial (small cone)
    kt=np.linspace(0,1,8); kr=(0.06*base)*(1-kt); kz=H*1.0+ (0.10*base)*kt
    Xk=kr[:,None]*np.cos(th)[None,:]; Yk=kr[:,None]*np.sin(th)[None,:]
    Zk=kz[:,None]*np.ones(nv)[None,:]; m.add_surface(Xk,Yk,Zk)
    spec={"form":"rekha deul (curvilinear tower)","height_m":H,"base_m":base,
          "plan":f"{n_paga}-ratha (pancharatha)","crown":"amalaka + kalasha"}
    return m, spec

def gen_dome(H=35.0, dia=20.0, drum=0.45, bulb=1.15, nu=48, nv=72):
    """Bulbous double dome on a drum (Mughal/Ottoman/Persian)."""
    m=Mesh()
    t=np.linspace(0,1,nu)
    drum_h=H*drum
    # profile radius vs height
    z=np.where(t< drum, (t/drum)*drum_h, drum_h+(t-drum)/(1-drum)*(H-drum_h))
    rr=np.empty_like(t)
    for i,ti in enumerate(t):
        if ti<drum: rr[i]=dia/2
        else:
            u=(ti-drum)/(1-drum)                                   # 0..1 over dome
            rr[i]=(dia/2)*bulb*np.sin(np.pi*(0.5+0.5*u))*(1-u*0.0) # bulbous
            rr[i]=(dia/2)*np.sqrt(max(1-u**2,0))*bulb              # ogee-ish
    th=np.linspace(0,2*np.pi,nv)
    X=rr[:,None]*np.cos(th)[None,:]; Y=rr[:,None]*np.sin(th)[None,:]
    Z=z[:,None]*np.ones(nv)[None,:]
    m.add_surface(X,Y,Z)
    spec={"form":"double dome on drum","height_m":H,"diameter_m":dia,"profile":"bulbous/ogee"}
    return m, spec

def gen_pyramid(base=55.0, H=24.0, steps=9, nv=4):
    """Stepped pyramid (Maya / Mesopotamian ziggurat)."""
    m=Mesh()
    nu=steps*2
    zs=[]; rs=[]
    for s in range(steps):
        z0=H*s/steps; z1=H*(s+1)/steps
        r0=(base/2)*(1-s/steps); r1=(base/2)*(1-(s+0.15)/steps)
        zs+=[z0,z1]; rs+=[r0,r1]
    zs=np.array(zs); rs=np.array(rs)
    th=np.linspace(0,2*np.pi,nv+1)+np.pi/4
    X=rs[:,None]*np.cos(th)[None,:]; Y=rs[:,None]*np.sin(th)[None,:]
    Z=zs[:,None]*np.ones(nv+1)[None,:]
    m.add_surface(X,Y,Z)
    spec={"form":"stepped pyramid","base_m":base,"height_m":H,"terraces":steps}
    return m, spec

print("Generators ready: gen_shikhara, gen_dome, gen_pyramid")

---
# v10 — Geometry + Cultural Accuracy (the synthesis)

Each tradition now generates its OWN form, and every dimension is derived from
that tradition's canon rule (with the treatise cited):

| Tradition | Form generated | NOT |
|---|---|---|
| Kalinga | rekha deul (single curvilinear tower) | — |
| Nagara | shekhari (central spire + clustered urushringa half-spires) | a plain cone |
| Dravidian | vimana (stepped pyramid of diminishing talas + cupola) | curvilinear |
| Mughal/Ottoman | double dome on drum + corner minarets | — |
| Maya/Mesopotamian | stepped platform pyramid | — |

`reconstruct_cultural()` resolves the tradition, derives the geometry parameters
from the canon (calibrating to a surviving measurement when you give one), builds
the correct form, renders + exports `.obj`, and prints a **cultural-accuracy
report** mapping every parameter to its source rule.

In [ ]:
# ============================================================
# 29) CANON-DRIVEN, TRADITION-SPECIFIC GEOMETRY
# ============================================================
def _merge(dst, src, dx=0, dy=0, dz=0, scale=1.0):
    base=len(dst.V)
    for (x,y,z) in src.V: dst.V.append((x*scale+dx, y*scale+dy, z*scale+dz))
    for (a,b,c) in src.F: dst.F.append((a+base, b+base, c+base))

def gen_shekhari(H, base, n_uru=4, curve=2.2):
    """Nagara: central curvilinear spire + clustered urushringa half-spires."""
    m=Mesh(); central,_=gen_shikhara(H, base, n_paga=3, curve=curve); _merge(m, central)
    for k in range(n_uru):
        ang=2*np.pi*k/n_uru
        sub,_=gen_shikhara(H*0.55, base*0.5, n_paga=3, curve=curve)
        _merge(m, sub, 0.42*base*np.cos(ang), 0.42*base*np.sin(ang))
    return m, {"form":"shekhari shikhara","height_m":H,"base_m":base,
               "clustered_half_spires":n_uru}

def gen_vimana(H, base, n_tala=6, diminish=0.86):
    """Dravidian: stepped pyramidal tower of diminishing storeys + cupola."""
    m=Mesh(); nv=4; th=np.linspace(0,2*np.pi,nv+1)+np.pi/4
    cur=base/2; z=0.0; talah=H*0.82/n_tala; zs=[]; rs=[]
    for s in range(n_tala):
        zs+=[z, z+talah*0.8, z+talah*0.8, z+talah]
        rs+=[cur, cur*0.97, cur*1.04, cur*diminish]            # wall, cornice, recess
        z+=talah; cur*=diminish
    zs=np.array(zs); rs=np.array(rs)
    m.add_surface(rs[:,None]*np.cos(th)[None,:], rs[:,None]*np.sin(th)[None,:],
                  zs[:,None]*np.ones(nv+1)[None,:])
    ct=np.linspace(0,np.pi/2,10); cr=cur*1.5*np.cos(ct); cz=z+cur*1.5*np.sin(ct)
    th2=np.linspace(0,2*np.pi,9)
    m.add_surface(cr[:,None]*np.cos(th2)[None,:], cr[:,None]*np.sin(th2)[None,:],
                  cz[:,None]*np.ones(9)[None,:])               # domical shikhara cap
    return m, {"form":"vimana (stepped)","height_m":H,"base_m":base,"storeys_tala":n_tala}

def gen_dome_minarets(H, dia, n_minaret=4):
    m,_=gen_dome(H=H, dia=dia)
    for k in range(n_minaret):
        ang=np.pi/4+2*np.pi*k/n_minaret
        mt=np.linspace(0,1,12); mr=np.full_like(mt, dia*0.05); mz=mt*H*1.1
        th=np.linspace(0,2*np.pi,12); sub=Mesh()
        sub.add_surface(mr[:,None]*np.cos(th)[None,:], mr[:,None]*np.sin(th)[None,:],
                        mz[:,None]*np.ones(12)[None,:])
        _merge(m, sub, 0.75*dia*np.cos(ang), 0.75*dia*np.sin(ang))
    return m, {"form":"double dome + minarets","height_m":H,"diameter_m":dia,"minarets":n_minaret}

# Canon -> geometry parameters, each tied to a cited rule
CANON_GEOMETRY = {
    "Kalinga (Odisha temple)": {"form":"rekha_deul",
        "treatise":"Shilpa Prakasha; Bhubaneswar canon",
        "params":{"height_ratio":2.0,"n_paga":5,"curve":2.4},
        "rules":{"height_ratio":"deul height ~2x jagamohana (porch) height",
                 "n_paga":"pancharatha (5-fold) vertical plan",
                 "curve":"rekha curvilinear gandi profile"}},
    "Nagara (North Indian temple)": {"form":"shekhari",
        "treatise":"Samarangana Sutradhara",
        "params":{"height_ratio":2.5,"n_uru":4,"curve":2.2},
        "rules":{"height_ratio":"shikhara height ~2.5x sanctum width",
                 "n_uru":"clustered urushringa half-spires (shekhari type)",
                 "curve":"latina curvilinear profile"}},
    "Dravidian (South Indian temple)": {"form":"vimana",
        "treatise":"Mayamata; Manasara",
        "params":{"height_ratio":1.6,"n_tala":6,"diminish":0.86},
        "rules":{"height_ratio":"vimana height ~1.6x base width",
                 "n_tala":"stacked diminishing storeys (talas)",
                 "diminish":"each tala ~0.86x the one below"}},
    "Indo-Islamic / Mughal": {"form":"dome",
        "treatise":"Timurid-Mughal tomb canon",
        "params":{"height_ratio":1.75,"n_minaret":4,"dome_ratio":0.55},
        "rules":{"height_ratio":"dome apex ~1.75x drum diameter",
                 "n_minaret":"four corner minarets (char-bagh symmetry)",
                 "dome_ratio":"bulbous double dome"}},
}
print("Canon-driven geometry for:", list(CANON_GEOMETRY))

---
## Results summary (use as notebook abstract / paper intro)

**HeritageFormer** is an evidence-based, culturally-governed reconstruction system
for heritage structures. Given a damaged or lost structure, it retrieves similar
sites, identifies the architectural tradition, pulls the governing canon (cited),
and **reconstructs the missing geometry** as a 3D mesh whose every dimension is
traceable to a documented rule — never free invention, never wrong-tradition
substitution.

**Pipeline & evidence**
- Knowledge base: 3,548 records (UNESCO 2019/2021, Heritage3D, Intangible), BGE-M3 + FAISS.
- Retrieval quality: **Recall@10 = 0.733, MRR = 0.419** on a held-out, no-name-leak benchmark (cell 17). Tracked across versions in `metrics_history.json`.
- Open-world tradition classifier: ~45 traditions, validated (Angkor→Khmer, Notre-Dame→Gothic, Chichén→Maya).
- Cultural canon: hand-authored + live Wikipedia (cited, tier-flagged), works for any tradition.
- Reconstruction: tradition-specific procedural geometry (rekha-deul / shekhari / vimana / dome / pyramid), calibrated to surviving measurements; exported as `.obj`.

**Worked example — Konark Sun Temple:** the deul (main tower) collapsed c.1837. HeritageFormer regenerates it as a 76 m rekha-deul (= 38 m surviving jagamohana × the Shilpa-Prakasha 2.0 ratio), 5-ratha plan, amalaka+kalasha crown — each parameter cited.

**Limitations (honest):** (1) canon proportions need review by a Shilpa-shastra / conservation specialist before real-world use; (2) photo-based parameter calibration failed on in-the-wild tourist images (silhouette = scene, not tower) — it requires orthographic elevation drawings; (3) same-tradition exemplar coverage is thin (e.g. only one Kalinga temple in the KB) — expanding it is the top quality lever.

**Contribution:** a reproducible framework for *culturally-governed* heritage reconstruction that prioritises cited evidence over generative imagery, with a measurable retrieval benchmark and a well-characterised limitation profile.

---
# v13 — MOST ACCURATE: corrected rekha curve + stepped plan + shaded render

Two fixes for a genuinely temple-like result:
1. **Geometry** — the deul is now near-vertical (bada) and curves in only near
   the top (true rekha profile), on a crisp **stepped-square** plan with rathas
   (not a rounded blob), crowned by a **wide ribbed amalaka**.
2. **Rendering** — proper directional lighting on stone via `Poly3DCollection`
   (the old `plot_trisurf` made everything look like a blob).

In [ ]:
# ============================================================
# 34) CORRECTED TEMPLE GEOMETRY (true rekha profile + stepped plan)
# ============================================================
def _plan(theta):
    """Stepped SQUARE plan with projecting rathas (saptaratha)."""
    sq = np.minimum(1.0/np.maximum(np.abs(np.cos(theta)), np.abs(np.sin(theta))), 1.55)
    tab = np.zeros_like(theta)
    for kc in (0, np.pi/2, np.pi, 3*np.pi/2):
        d = np.abs(np.angle(np.exp(1j*(theta-kc))))
        tab += 0.24*(d < 0.15)                       # raha (central projection)
        tab += 0.12*((d >= 0.18) & (d < 0.40))       # anuratha step
    return sq + tab

def gen_rekha_temple(H, base, nu=100, nv=140):
    m = Mesh(); R = base/2
    bada, gtop, shoulder, neck = 0.28, 0.78, 0.83, 0.86
    t = np.linspace(0, neck, nu); r = np.empty_like(t)
    for i, ti in enumerate(t):
        if ti < bada:        r[i] = 1.0                                      # vertical bada
        elif ti < gtop:      u=(ti-bada)/(gtop-bada); r[i]=1-0.55*u**2.6     # rekha gandi
        elif ti < shoulder:  u=(ti-gtop)/(shoulder-gtop); r[i]=0.45-0.15*u   # skandha
        else:                r[i] = 0.30                                     # beki neck
    th = np.linspace(0, 2*np.pi, nv); pf=_plan(th)
    m.add_surface(R*(r[:,None]*pf[None,:])*np.cos(th)[None,:],
                  R*(r[:,None]*pf[None,:])*np.sin(th)[None,:], (H*t)[:,None]*np.ones(nv)[None,:])
    u = np.linspace(0, 2*np.pi, 16); ribs = 1+0.10*np.cos(24*th)            # wide ribbed amalaka
    AR = 0.52*R*(1+0.5*np.cos(u))
    m.add_surface((AR[:,None]*ribs[None,:])*np.cos(th)[None,:],
                  (AR[:,None]*ribs[None,:])*np.sin(th)[None,:],
                  (H*neck+0.10*R+0.10*R*np.sin(u))[:,None]*np.ones(nv)[None,:])
    ku = np.linspace(0, np.pi/2, 10); kr=0.20*R*np.cos(ku); kz=H*(neck+0.03)+0.30*R*np.sin(ku)  # kalasha
    m.add_surface(kr[:,None]*np.cos(th)[None,:], kr[:,None]*np.sin(th)[None,:], kz[:,None]*np.ones(nv)[None,:])
    return m

def gen_pidha(H, base, tiers=8, nv=120):
    m = Mesh(); th = np.linspace(0, 2*np.pi, nv)
    sq = np.minimum(1.0/np.maximum(np.abs(np.cos(th)), np.abs(np.sin(th))), 1.45)
    wallh = H*0.40; th_h=(H-wallh)/tiers; z=wallh
    zs=[0.0, wallh]; rs=[1.0, 1.0]
    for s in range(tiers):
        rb=1-s/(tiers+1.0); rt=1-(s+1)/(tiers+1.0)
        zs+=[z, z+th_h*0.65, z+th_h*0.65, z+th_h]; rs+=[rb, rb*0.93, rb, rt]; z+=th_h
    zs=np.array(zs); rs=np.clip(np.array(rs),0.05,None)*(base/2)
    m.add_surface((rs[:,None]*sq[None,:])*np.cos(th)[None,:],
                  (rs[:,None]*sq[None,:])*np.sin(th)[None,:], zs[:,None]*np.ones(nv)[None,:])
    return m

def build_temple(H, base):
    m=Mesh(); _merge(m, gen_rekha_temple(H, base))
    _merge(m, gen_pidha(H*0.46, base*1.05), dx=base*1.15)
    return m

print("Corrected temple generator ready.")

In [ ]:
# ============================================================
# 35) SHADED RENDER (directional light on stone) + Konark
# ============================================================
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def render_shaded(mesh, title, path, elev=16, azim=-55, color=(0.74,0.55,0.37)):
    V=np.array(mesh.V); F=np.array(mesh.F); tris=V[F]
    n=np.cross(tris[:,1]-tris[:,0], tris[:,2]-tris[:,0])
    ln=np.linalg.norm(n,axis=1,keepdims=True); ln[ln==0]=1; n=n/ln
    light=np.array([0.4,0.5,0.78]); light=light/np.linalg.norm(light)
    inten=0.32+0.68*np.clip(n@light,0,1)
    cols=np.clip(np.array(color)[None,:]*inten[:,None],0,1)
    _c=tris.mean(1); _j=0.06*(np.sin(_c[:,0]*0.7)+np.sin(_c[:,2]*0.9)); cols=np.clip(cols*(1+_j[:,None]),0,1)  # stone-texture weathering
    fig=plt.figure(figsize=(7,9)); ax=fig.add_subplot(111,projection="3d")
    ax.add_collection3d(Poly3DCollection(tris, facecolors=cols, linewidths=0))
    mx=V.max(0); mn=V.min(0); ctr=(mx+mn)/2; rng=(mx-mn).max()/2
    ax.set_xlim(ctr[0]-rng,ctr[0]+rng); ax.set_ylim(ctr[1]-rng,ctr[1]+rng); ax.set_zlim(mn[2],mn[2]+2*rng)
    ax.set_box_aspect((1,1,1.45)); ax.set_axis_off(); ax.view_init(elev=elev, azim=azim)
    ax.set_title(title, fontsize=9)
    plt.tight_layout(); plt.savefig(path, dpi=140, bbox_inches="tight"); plt.show()

H=38*2.0; base=H*0.42                                        # Konark: deul ~2x porch
temple=build_temple(H, base)
open("/kaggle/working/Konark_temple_reconstructed.obj","w").write(temple.obj())
render_shaded(temple,
    "Konark Sun Temple — reconstructed deul + jagamohana (Kalinga canon)\n"
    "bada / gandi (rekha) / bhumi tiers / amalaka / kalasha — Shilpa Prakasha",
    "/kaggle/working/Konark_temple_reconstructed.png")
print(f"Vertices: {len(temple.V)}  Faces: {len(temple.F)}  -> /kaggle/working/Konark_temple_reconstructed.obj")

---
# v14 — Grounded, joined temple (plinth + porch crown)

The deul and porch were floating apart. Now they sit **joined on a shared
stepped plinth (pista)**, axially aligned, and the porch gets its **bell + kalasha
crown** — reading as one complete temple, not two objects.

In [ ]:
# ============================================================
# 36) GROUNDED, JOINED TEMPLE COMPOSITION
# ============================================================
def add_box(m, x0, x1, y0, y1, z0, z1):
    c=[(x0,y0,z0),(x1,y0,z0),(x1,y1,z0),(x0,y1,z0),(x0,y0,z1),(x1,y0,z1),(x1,y1,z1),(x0,y1,z1)]
    b=len(m.V); m.V+=c
    for f in [(0,2,1),(0,3,2),(4,5,6),(4,6,7),(0,1,5),(0,5,4),
              (1,2,6),(1,6,5),(2,3,7),(2,7,6),(3,0,4),(3,4,7)]:
        m.F.append((f[0]+b,f[1]+b,f[2]+b))

def gen_pidha(H, base, tiers=8, nv=120):
    """Jagamohana: cubic hall + pyramidal pidha roof + bell/kalasha crown."""
    m=Mesh(); th=np.linspace(0,2*np.pi,nv)
    sq=np.minimum(1.0/np.maximum(np.abs(np.cos(th)),np.abs(np.sin(th))),1.45)
    wallh=H*0.40; th_h=(H-wallh)/tiers; z=wallh; zs=[0.0,wallh]; rs=[1.0,1.0]
    for s in range(tiers):
        rb=1-s/(tiers+1.0); rt=1-(s+1)/(tiers+1.0)
        zs+=[z, z+th_h*0.6, z+th_h*0.6, z+th_h]; rs+=[rb, rb*0.92, rb, rt]; z+=th_h
    zs=np.array(zs); rs=np.clip(np.array(rs),0.05,None)*(base/2)
    m.add_surface((rs[:,None]*sq[None,:])*np.cos(th)[None,:],
                  (rs[:,None]*sq[None,:])*np.sin(th)[None,:], zs[:,None]*np.ones(nv)[None,:])
    bu=np.linspace(0,np.pi/2,8); br=0.16*base*np.cos(bu); bz=zs.max()+0.18*base*np.sin(bu)
    m.add_surface(br[:,None]*np.cos(th)[None,:], br[:,None]*np.sin(th)[None,:], bz[:,None]*np.ones(nv)[None,:])
    return m

def build_temple(H, base):
    m=Mesh(); R=base/2
    deul_x0=-R*1.55; porch_base=base*1.02
    dx=R*1.34 + porch_base*0.52*1.45 - porch_base*0.06          # porch abuts deul front
    porch_x1=dx + porch_base*0.52*1.45
    ph=base*0.16                                                # stepped plinth (pista)
    add_box(m, deul_x0-base*0.10, porch_x1+base*0.10, -R*1.7, R*1.7, 0, ph*0.55)
    add_box(m, deul_x0-base*0.02, porch_x1+base*0.02, -R*1.6, R*1.6, ph*0.55, ph)
    _merge(m, gen_rekha_temple(H, base), dz=ph)
    _merge(m, gen_pidha(H*0.46, porch_base), dx=dx, dz=ph)
    return m

H=38*2.0; base=H*0.42
temple=build_temple(H, base)
open("/kaggle/working/Konark_temple_reconstructed.obj","w").write(temple.obj())
render_shaded(temple,
    "Konark Sun Temple — reconstructed deul + jagamohana on shared plinth\n"
    "Kalinga canon (Shilpa Prakasha): pista / bada / gandi / amalaka / kalasha",
    "/kaggle/working/Konark_temple_reconstructed.png", elev=14, azim=-58)
print(f"Vertices: {len(temple.V)}  Faces: {len(temple.F)}")
print("Exported Konark_temple_reconstructed.obj / .png to /kaggle/working")

---
# v15 — THE WHOLE THING: the complete Konark chariot

Konark is conceived as the Sun God's colossal **chariot**. The full reconstruction
assembles every element on one platform:

- the **deul** (reconstructed 76 m tower) + **jagamohana** porch
- the great stepped **platform** (chariot bed)
- the **24 spoked stone wheels** (12 per side) — Konark's signature
- the **7 horses** drawing the chariot at the front

One procedural model, every part placed by the canon layout.

In [ ]:
# ============================================================
# 37) CHARIOT PARTS — wheels, horses, placement helpers
# ============================================================
def _place(dst, src, rot=None, t=(0,0,0), s=1.0):
    base=len(dst.V); V=np.array(src.V, dtype=float)*s
    if rot is not None: V=V@np.array(rot).T
    V=V+np.array(t, dtype=float)
    for v in V: dst.V.append((float(v[0]),float(v[1]),float(v[2])))
    for (a,b,c) in src.F: dst.F.append((a+base,b+base,c+base))

def _cyl(p0, p1, r, n=10):
    p0=np.array(p0,float); p1=np.array(p1,float); d=p1-p0; L=np.linalg.norm(d); d=d/L
    a=np.array([1,0,0]) if abs(d[0])<0.9 else np.array([0,1,0])
    u=np.cross(d,a); u/=np.linalg.norm(u); v=np.cross(d,u); th=np.linspace(0,2*np.pi,n); m=Mesh()
    X=np.zeros((2,n)); Y=np.zeros((2,n)); Z=np.zeros((2,n))
    for i,si in enumerate([0.0,1.0]):
        c=p0+si*d*L
        for j,tj in enumerate(th): X[i,j],Y[i,j],Z[i,j]=c+r*(np.cos(tj)*u+np.sin(tj)*v)
    m.add_surface(X,Y,Z); return m

def gen_wheel(R=3.0, tube=0.45, spokes=8, hubr=0.6):
    """Spoked stone chariot wheel (disc in X-Z plane, axle along Y)."""
    m=Mesh(); nphi=28; npsi=10
    phi=np.linspace(0,2*np.pi,nphi); psi=np.linspace(0,2*np.pi,npsi); PH,PS=np.meshgrid(phi,psi,indexing="ij")
    m.add_surface((R+tube*np.cos(PS))*np.cos(PH), tube*np.sin(PS), (R+tube*np.cos(PS))*np.sin(PH))
    _place(m, _cyl((0,-tube,0),(0,tube,0),hubr,12))
    for k in range(spokes):
        a=2*np.pi*k/spokes
        _place(m, _cyl((hubr*np.cos(a),0,hubr*np.sin(a)),((R-tube)*np.cos(a),0,(R-tube)*np.sin(a)),tube*0.30,6))
    return m

def gen_horse(s=1.0):
    """Stylised horse facing +X."""
    m=Mesh(); add_box(m,-1.3*s,1.3*s,-0.45*s,0.45*s,1.4*s,2.5*s)
    for (x,y) in [(1.0,0.32),(1.0,-0.32),(-1.0,0.32),(-1.0,-0.32)]:
        _place(m,_cyl((x*s,y*s,0),(x*s,y*s,1.5*s),0.16*s,8))
    _place(m,_cyl((1.2*s,0,2.3*s),(2.1*s,0,3.4*s),0.30*s,10))
    add_box(m,2.0*s,2.9*s,-0.28*s,0.28*s,3.2*s,3.9*s)
    _place(m,_cyl((-1.3*s,0,2.3*s),(-2.0*s,0,1.4*s),0.10*s,6))
    return m

In [ ]:
# ============================================================
# 38) BUILD THE WHOLE KONARK CHARIOT + render
# ============================================================
def build_konark(H, base):
    m=Mesh(); R=base/2
    deul_x0=-R*1.55; porch_base=base*1.02
    dx=R*1.34+porch_base*0.52*1.45-porch_base*0.06; porch_x1=dx+porch_base*0.52*1.45
    plat_x0=deul_x0-base*0.25; plat_x1=porch_x1+base*0.25; plat_W=R*1.9; ph=base*0.30
    add_box(m, plat_x0, plat_x1, -plat_W, plat_W, 0, ph*0.5)                       # platform tier 1
    add_box(m, plat_x0+base*0.06, plat_x1-base*0.06, -(plat_W-base*0.06), plat_W-base*0.06, ph*0.5, ph)
    Rw=base*0.085; nW=12                                                            # 12 wheels per side = 24
    xs_w=np.linspace(plat_x0+(plat_x1-plat_x0)*0.06, plat_x1-(plat_x1-plat_x0)*0.06, nW)
    for xw in xs_w:
        for ysign in (-1,1):
            _place(m, gen_wheel(R=Rw, tube=Rw*0.16, spokes=8, hubr=Rw*0.2), t=(xw, ysign*plat_W, Rw+ph*0.05))
    hs=base*0.05                                                                    # 7 horses at the front
    for yy in np.linspace(-plat_W*0.7, plat_W*0.7, 7):
        _place(m, gen_horse(s=hs), t=(plat_x1+base*0.28, yy, 0))
    _merge(m, gen_rekha_temple(H, base), dz=ph)                                     # deul + porch
    _merge(m, gen_pidha(H*0.46, porch_base), dx=dx, dz=ph)
    return m

H=38*2.0; base=H*0.42
konark=build_konark(H, base)
open("/kaggle/working/Konark_full_chariot.obj","w").write(konark.obj())
render_shaded(konark,
    "Konark Sun Temple — complete chariot reconstruction\n"
    "deul + jagamohana + platform + 24 wheels + 7 horses (Kalinga canon)",
    "/kaggle/working/Konark_full_chariot.png", elev=12, azim=-62)
print(f"Vertices: {len(konark.V)}  Faces: {len(konark.F)}")
print("24 wheels, 7 horses, deul + porch on platform")
print("Exported Konark_full_chariot.obj / .png to /kaggle/working")

## v16 — THE WORLD ASSEMBLER: build *any* structure, culturally appropriate

The Konark chariot proves the pipeline end-to-end for one structure. This generalises
it: **one procedural model that assembles a full, culturally-correct building for any
tradition the classifier resolves** — not a single tower, the whole monument.

Each tradition gets its **own** assembler, built from shared primitives but obeying its
own canon:

| Tradition | Assembled form |
|---|---|
| Kalinga | deul + jagamohana on a plinth (Konark → full chariot) |
| Nagara | shekhari: central spire + clustered urushringa half-spires |
| Dravidian | vimana + gopuram gateway on a jagati |
| Mughal / Ottoman | double dome on a cubic body + 4 corner minarets + char-bagh plinth |
| Khmer | quincunx of five prasats on stepped terraces |
| Gothic | cruciform nave + transept + twin west towers + spires |
| Greco-Roman | stylobate + peripteral colonnade + pediment |
| Maya | stepped pyramid + summit shrine + roof comb + stairway |
| Chinese | raised terrace + body + bracketed tiered hip roofs |
| Buddhist | stupa: hemispherical anda + harmika + chhatra |
| Indian fort | curtain ramparts + corner bastions + gateway tower |

`build_structure(name, location)` retrieves the structure → classifies its tradition →
derives scale from the canon (calibrated to a surviving measurement when given) →
assembles the correct form → renders + exports `.obj`, with every dimension tied to a
cited rule. Same engine, any culture on Earth.


In [ ]:
# ============================================================
# 39) WORLD ASSEMBLER (DEEPENED) — one model, any structure, every part
# Self-contained _w primitives + detail helpers (mouldings, niches, jali,
# doorways, angasikhara, fluted columns, gadrooned domes...). Reuses notebook
# Mesh/_merge/add_box/_place/_cyl/build_konark/render_shaded/tangible_search/
# classify_tradition/best_tradition/heritage_master.
# ============================================================

# --- detail helpers (additive shape-grammar relief, no CSG) ---
def _cornice(m, x0,x1,y0,y1,z, th, ov): add_box(m, x0-ov, x1+ov, y0-ov, y1+ov, z, z+th)
def _steps(m, x_front, x_back, y0, y1, z0, ztop, n=12):
    for i in range(n):
        zb=z0+(ztop-z0)*i/n; zt=z0+(ztop-z0)*(i+1)/n; xa=x_front+(x_back-x_front)*i/n
        add_box(m, xa, x_front, y0, y1, zb, zt)
def _crenellate(m, x0, x1, y0, y1, z, mw, gap, h):
    x=x0
    while x < x1-1e-6: add_box(m, x, min(x+mw,x1), y0, y1, z, z+h); x+=mw+gap
def _moldings(m, x0,x1,y0,y1, z0, courses=4, ch=None, step=None):
    w=min(x1-x0,y1-y0); ch=ch or w*0.04; step=step or w*0.04; z=z0
    for i in range(courses):
        e=step*(courses-i-1); add_box(m, x0-e, x1+e, y0-e, y1+e, z, z+ch); z+=ch
    return z
def _niches4(m, R, w, h, z0, proj, deity=True):
    for nx,ny in [(1,0),(-1,0),(0,1),(0,-1)]:
        if nx:
            x0,x1=(R,R+proj) if nx>0 else (-R-proj,-R); cx=(x0+x1)/2
            add_box(m,x0,x1,-w/2,w/2,z0,z0+h); add_box(m,x0,x1,-w*0.62,w*0.62,z0+h,z0+h+proj*0.6)
            if deity: add_box(m,cx-w*0.2,cx+w*0.2,-w*0.18,w*0.18,z0+h*0.12,z0+h*0.78)
        else:
            y0,y1=(R,R+proj) if ny>0 else (-R-proj,-R); cy=(y0+y1)/2
            add_box(m,-w/2,w/2,y0,y1,z0,z0+h); add_box(m,-w*0.62,w*0.62,y0,y1,z0+h,z0+h+proj*0.6)
            if deity: add_box(m,-w*0.18,w*0.18,cy-w*0.2,cy+w*0.2,z0+h*0.12,z0+h*0.78)
def _angasikhara(m, H, base, dz):
    for sx in (-1,1):
        for sy in (-1,1):
            _merge(m, gen_rekha_w(H*0.42, base*0.34, 30, 40), sx*base*0.42, sy*base*0.42, dz)
def _jali(m, face_R, half, z0, z1, n=6, axis="x"):
    bar=(z1-z0)*0.03; proj=bar*1.2; ys=np.linspace(-half,half,n); zs=np.linspace(z0,z1,n)
    if axis=="x":
        for y in ys: add_box(m,face_R,face_R+proj,y-bar,y+bar,z0,z1)
        for z in zs: add_box(m,face_R,face_R+proj,-half,half,z-bar,z+bar)
    else:
        for x in ys: add_box(m,x-bar,x+bar,face_R,face_R+proj,z0,z1)
        for z in zs: add_box(m,-half,half,face_R,face_R+proj,z-bar,z+bar)
def _doorway(m, R, w, h, z0, proj, axis="x", sign=1):
    j=w*0.16
    if axis=="x":
        x0=(R if sign>0 else -R-proj); x1=x0+proj
        add_box(m,x0,x1,-w/2,-w/2+j,z0,z0+h); add_box(m,x0,x1,w/2-j,w/2,z0,z0+h); add_box(m,x0,x1,-w/2,w/2,z0+h,z0+h+j)
    else:
        y0=(R if sign>0 else -R-proj); y1=y0+proj
        add_box(m,-w/2,-w/2+j,y0,y1,z0,z0+h); add_box(m,w/2-j,w/2,y0,y1,z0,z0+h); add_box(m,-w/2,w/2,y0,y1,z0+h,z0+h+j)
def _rose_window(m, faceR, z, rad, n=12):
    th=np.linspace(0,2*np.pi,n,endpoint=False)
    for a in th:
        y=rad*np.cos(a); zz=z+rad*np.sin(a); add_box(m,faceR-rad*0.14,faceR,y-rad*0.09,y+rad*0.09,zz-rad*0.09,zz+rad*0.09)
    add_box(m,faceR-rad*0.12,faceR,-rad*0.14,rad*0.14,z-rad*0.14,z+rad*0.14)
def _lancets(m, faceR, ys, z0, h, w, sign=-1):
    x0=(faceR-abs(w)*0.5) if sign<0 else faceR; x1=x0+abs(w)*0.5
    for y in ys:
        add_box(m,x0,x1,y-w/2,y+w/2,z0,z0+h); add_box(m,x0,x1,y-w*0.3,y+w*0.3,z0+h,z0+h+w*0.5)
def _triglyphs(m, x0,x1, yface, z0,z1, n):
    for x in np.linspace(x0,x1,n): add_box(m,x-(x1-x0)*0.012,x+(x1-x0)*0.012,yface,yface+(z1-z0)*0.3,z0,z1)

# --- self-contained mesh primitives (mesh-only, _w suffix) ---
def _w_plan(theta):
    sq=np.minimum(1.0/np.maximum(np.abs(np.cos(theta)),np.abs(np.sin(theta))),1.55); tab=np.zeros_like(theta)
    for kc in (0,np.pi/2,np.pi,3*np.pi/2):
        d=np.abs(np.angle(np.exp(1j*(theta-kc)))); tab+=0.24*(d<0.15)+0.12*((d>=0.18)&(d<0.40))
    return sq+tab
def gen_rekha_w(H, base, nu=64, nv=72):
    m=Mesh(); R=base/2; bada,gtop,shoulder,neck=0.28,0.78,0.83,0.86; t=np.linspace(0,neck,nu); r=np.empty_like(t)
    for i,ti in enumerate(t):
        if ti<bada: r[i]=1.0
        elif ti<gtop: u=(ti-bada)/(gtop-bada); r[i]=1-0.55*u**2.6
        elif ti<shoulder: u=(ti-gtop)/(shoulder-gtop); r[i]=0.45-0.15*u
        else: r[i]=0.30
    th=np.linspace(0,2*np.pi,nv); pf=_w_plan(th)
    m.add_surface(R*(r[:,None]*pf[None,:])*np.cos(th)[None,:], R*(r[:,None]*pf[None,:])*np.sin(th)[None,:], (H*t)[:,None]*np.ones(nv)[None,:])
    u=np.linspace(0,2*np.pi,16); ribs=1+0.10*np.cos(24*th); AR=0.52*R*(1+0.5*np.cos(u))
    m.add_surface((AR[:,None]*ribs[None,:])*np.cos(th)[None,:], (AR[:,None]*ribs[None,:])*np.sin(th)[None,:], (H*neck+0.10*R+0.10*R*np.sin(u))[:,None]*np.ones(nv)[None,:])
    ku=np.linspace(0,np.pi/2,10); kr=0.20*R*np.cos(ku); kz=H*(neck+0.03)+0.30*R*np.sin(ku)
    m.add_surface(kr[:,None]*np.cos(th)[None,:], kr[:,None]*np.sin(th)[None,:], kz[:,None]*np.ones(nv)[None,:])
    _cornice(m,-R*0.95,R*0.95,-R*0.95,R*0.95,H*bada,H*0.02,R*0.12); return m
def gen_pidha_w(H, base, tiers=8, nv=72):
    m=Mesh(); th=np.linspace(0,2*np.pi,nv); sq=np.minimum(1.0/np.maximum(np.abs(np.cos(th)),np.abs(np.sin(th))),1.45)
    wallh=H*0.40; th_h=(H-wallh)/tiers; z=wallh; zs=[0.0,wallh]; rs=[1.0,1.0]
    for s in range(tiers):
        rb=1-s/(tiers+1.0); rt=1-(s+1)/(tiers+1.0); zs+=[z,z+th_h*0.65,z+th_h*0.65,z+th_h]; rs+=[rb,rb*0.93,rb,rt]; z+=th_h
    zs=np.array(zs); rs=np.clip(np.array(rs),0.05,None)*(base/2)
    m.add_surface((rs[:,None]*sq[None,:])*np.cos(th)[None,:], (rs[:,None]*sq[None,:])*np.sin(th)[None,:], zs[:,None]*np.ones(nv)[None,:]); return m
def gen_vimana_w(H, base, n_tala=6, diminish=0.86):
    m=Mesh(); nv=4; th=np.linspace(0,2*np.pi,nv+1)+np.pi/4; cur=base/2; z=0.0; talah=H*0.82/n_tala; zs=[]; rs=[]
    for s in range(n_tala):
        zs+=[z,z+talah*0.8,z+talah*0.8,z+talah]; rs+=[cur,cur*0.97,cur*1.04,cur*diminish]
        add_box(m,-cur*1.1,cur*1.1,-cur*1.1,cur*1.1, z+talah*0.8, z+talah*0.92); z+=talah; cur*=diminish
    zs=np.array(zs); rs=np.array(rs)
    m.add_surface(rs[:,None]*np.cos(th)[None,:], rs[:,None]*np.sin(th)[None,:], zs[:,None]*np.ones(nv+1)[None,:])
    ct=np.linspace(0,np.pi/2,10); cr=cur*1.5*np.cos(ct); cz=z+cur*1.5*np.sin(ct); th2=np.linspace(0,2*np.pi,9)
    m.add_surface(cr[:,None]*np.cos(th2)[None,:], cr[:,None]*np.sin(th2)[None,:], cz[:,None]*np.ones(9)[None,:]); return m

# --- detailed component primitives ---
def gen_column(h, r, nv=12):
    m=Mesh(); th=np.linspace(0,2*np.pi,nv); add_box(m,-r*1.35,r*1.35,-r*1.35,r*1.35,0,0.08*h)
    t=np.linspace(0,1,14); shaft=r*(1-0.05*np.sin(np.pi*t)); flute=1-0.05*np.abs(np.cos(10*th)); Rg=shaft[:,None]*flute[None,:]; z=0.08*h+t*0.82*h
    m.add_surface(Rg*np.cos(th)[None,:], Rg*np.sin(th)[None,:], z[:,None]*np.ones(nv)[None,:])
    ce=np.linspace(0,1,5); cr=r*(1+0.55*ce); cz=0.90*h+0.05*h*ce
    m.add_surface(cr[:,None]*np.cos(th)[None,:], cr[:,None]*np.sin(th)[None,:], cz[:,None]*np.ones(nv)[None,:]); add_box(m,-r*1.7,r*1.7,-r*1.7,r*1.7,0.95*h,h); return m
def gen_chhatri(h, r, nv=16):
    m=Mesh(); th=np.linspace(0,2*np.pi,nv)
    for sx in (-1,1):
        for sy in (-1,1): _place(m,_cyl((sx*r,sy*r,0),(sx*r,sy*r,0.6*h),r*0.16,8))
    add_box(m,-r*1.3,r*1.3,-r*1.3,r*1.3,0.6*h,0.68*h)
    du=np.linspace(0,np.pi/2,8); dr=r*1.1*np.cos(du); dz=0.68*h+r*1.3*np.sin(du)
    m.add_surface(dr[:,None]*np.cos(th)[None,:], dr[:,None]*np.sin(th)[None,:], dz[:,None]*np.ones(nv)[None,:]); return m
def gen_pinnacle(h, r, nv=4):
    m=Mesh(); th=np.linspace(0,2*np.pi,nv+1)+np.pi/4; add_box(m,-r,r,-r,r,0,0.5*h); t=np.linspace(0,1,6); rr=r*1.2*(1-t); z=0.5*h+0.5*h*t
    m.add_surface(rr[:,None]*np.cos(th)[None,:], rr[:,None]*np.sin(th)[None,:], z[:,None]*np.ones(nv+1)[None,:]); return m
def gen_dome_d(H, dia, ribs=20, nv=52):
    m=Mesh(); nu=30; t=np.linspace(0,1,nu); drum=0.30; dh=H*drum; z=np.where(t<drum,(t/drum)*dh, dh+(t-drum)/(1-drum)*(H-dh))
    th=np.linspace(0,2*np.pi,nv); gad=1+0.045*np.cos(ribs*th); u=np.clip((t-drum)/(1-drum),0,1); prof=np.sqrt(np.clip(1-u**2,0,1)); bulb=1+0.16*np.sin(np.pi*u)
    rr=(dia/2)*np.where(t<drum,1.0,prof*bulb)
    m.add_surface((rr[:,None]*gad[None,:])*np.cos(th)[None,:], (rr[:,None]*gad[None,:])*np.sin(th)[None,:], z[:,None]*np.ones(nv)[None,:])
    add_box(m,-dia*0.62,dia*0.62,-dia*0.62,dia*0.62,dh*0.5,dh*0.62); fz=np.linspace(0,1,8); fr=(dia*0.06)*(1-0.7*fz); fzz=H+(dia*0.22)*fz
    m.add_surface(fr[:,None]*np.cos(th)[None,:], fr[:,None]*np.sin(th)[None,:], fzz[:,None]*np.ones(nv)[None,:]); return m
def gen_minaret_d(H, r, nv=14):
    m=Mesh(); t=np.linspace(0,1,18); rr=r*(1-0.35*t); z=t*H; th=np.linspace(0,2*np.pi,nv); flute=1-0.05*np.abs(np.cos(8*th))
    m.add_surface((rr[:,None]*flute[None,:])*np.cos(th)[None,:], (rr[:,None]*flute[None,:])*np.sin(th)[None,:], z[:,None]*np.ones(nv)[None,:])
    bz=0.68*H; add_box(m,-r*1.5,r*1.5,-r*1.5,r*1.5,bz,bz+r*0.6); cu=np.linspace(0,np.pi/2,8); cr=r*0.85*np.cos(cu); cz=H+r*1.3*np.sin(cu)
    m.add_surface(cr[:,None]*np.cos(th)[None,:], cr[:,None]*np.sin(th)[None,:], cz[:,None]*np.ones(nv)[None,:]); return m
def gen_sqtower_d(H, base, tiers=7, taper=0.84, nv=4):
    m=Mesh(); th=np.linspace(0,2*np.pi,nv+1)+np.pi/4; cur=base/2; z=0.0; th_h=H/tiers; zs=[]; rs=[]
    for s in range(tiers):
        zs+=[z, z+th_h*0.80, z+th_h*0.80, z+th_h]; rs+=[cur, cur*0.99, cur*1.06, cur*taper]
        add_box(m,-cur*1.12,cur*1.12,-cur*1.12,cur*1.12, z+th_h*0.80, z+th_h*0.93)
        for sx in (-1,1): add_box(m, sx*cur*0.95, sx*cur*1.05, -cur*0.30, cur*0.30, z+th_h*0.12, z+th_h*0.66)
        z+=th_h; cur*=taper
    zs=np.array(zs); rs=np.array(rs)
    m.add_surface(rs[:,None]*np.cos(th)[None,:], rs[:,None]*np.sin(th)[None,:], zs[:,None]*np.ones(nv+1)[None,:])
    cu=np.linspace(0,1,6); cr=cur*1.35*(1-cu); cz=z+cur*1.7*cu
    m.add_surface(cr[:,None]*np.cos(th)[None,:], cr[:,None]*np.sin(th)[None,:], cz[:,None]*np.ones(nv+1)[None,:]); return m
def gen_spire_d(H, base, nv=4):
    m=Mesh(); th=np.linspace(0,2*np.pi,nv+1)+np.pi/4; t=np.linspace(0,1,12); rr=(base/2)*(1-t)**1.1; z=t*H
    m.add_surface(rr[:,None]*np.cos(th)[None,:], rr[:,None]*np.sin(th)[None,:], z[:,None]*np.ones(nv+1)[None,:])
    for sx in (-1,1):
        for sy in (-1,1): _place(m, gen_pinnacle(H*0.34, base*0.10), t=(sx*base*0.34, sy*base*0.34, 0))
    return m

# --- DEEPENED assemblers (every part articulated) ---
def asm_kalinga(H, base, name=""):
    if "konar" in name.lower() or "konâr" in name.lower(): return build_konark(H, base)
    m=Mesh(); R=base/2; porch_base=base*1.02
    dx=R*1.34+porch_base*0.52*1.45-porch_base*0.06; porch_x1=dx+porch_base*0.52*1.45; ph=base*0.18
    z=_moldings(m,-R*1.55-base*0.10,porch_x1+base*0.10,-R*1.7,R*1.7,0,courses=4)
    _merge(m, gen_rekha_w(H, base), dz=z)
    _moldings(m,-R*0.92,R*0.92,-R*0.92,R*0.92, z, courses=3, ch=base*0.05, step=base*0.04)
    _niches4(m, R*0.95, base*0.34, H*0.16, z+H*0.10, base*0.10)
    _doorway(m, R*0.95, base*0.42, H*0.16, z+H*0.02, base*0.12, axis="x", sign=1)
    _angasikhara(m, H, base, z); _merge(m, gen_pidha_w(H*0.46, porch_base), dx=dx, dz=z); return m
def asm_nagara(H, base, name=""):
    m=Mesh(); ph=base*0.16; add_box(m,-base*0.98,base*0.98,-base*0.98,base*0.98,0,ph*0.6)
    _cornice(m,-base*0.95,base*0.95,-base*0.95,base*0.95,ph*0.6,base*0.05,base*0.10); add_box(m,-base*0.95,base*0.95,-base*0.95,base*0.95,ph*0.6,ph)
    _merge(m, gen_rekha_w(H, base), dz=ph)
    for k in range(4):
        ang=2*np.pi*k/4; _merge(m, gen_rekha_w(H*0.5, base*0.42, 36, 44), 0.40*base*np.cos(ang), 0.40*base*np.sin(ang), ph)
    _niches4(m, base*0.95, base*0.30, H*0.14, ph+H*0.06, base*0.08)
    _merge(m, gen_pidha_w(H*0.42, base*0.95), dx=base*1.75, dz=ph); return m
def asm_dravidian(H, base, name=""):
    m=Mesh(); ph=base*0.14
    z=_moldings(m,-base*1.05,base*1.05,-base*1.05,base*1.05,0,courses=3,ch=ph*0.25,step=base*0.05)
    _merge(m, gen_vimana_w(H, base*0.9, 6, 0.86), dz=z); _niches4(m, base*0.46, base*0.26, H*0.12, z+H*0.06, base*0.08)
    g=gen_sqtower_d(H*0.72, base*0.72, 5, 0.80); add_box(g,-base*0.40,base*0.40,-base*0.10,base*0.10, H*0.72, H*0.80)
    _merge(m, g, dx=base*2.0, dz=z); _doorway(m, base*0.7, base*0.5, H*0.30, z, base*0.14, axis="x", sign=1)
    add_box(m,base*0.95,base*2.05,-base*0.85,base*0.85,z,z+base*0.12); return m
def asm_mughal(H, base, name=""):
    m=Mesh(); ph=base*0.12; body=base*1.0; bz=ph
    _moldings(m,-body*1.95,body*1.95,-body*1.95,body*1.95,0,courses=3,ch=ph*0.2,step=body*0.06)
    add_box(m,-body,body,-body,body,bz,bz+body*1.2); _cornice(m,-body,body,-body,body,bz+body*1.2,body*0.05,body*0.10)
    for fx,fy in [(1,0),(-1,0),(0,1),(0,-1)]:
        add_box(m, fx*body if fx else -body*0.42, fx*body*1.12 if fx else body*0.42, fy*body if fy else -body*0.42, fy*body*1.12 if fy else body*0.42, bz, bz+body*0.95)
        _doorway(m, body*1.12 if fx>0 else body, body*0.55, body*0.7, bz, body*0.12, axis="x" if fx else "y", sign=1 if (fx>0 or fy>0) else -1)
    _jali(m, body, body*0.5, bz+body*0.15, bz+body*0.9, n=6, axis="x")
    _merge(m, gen_dome_d(H*0.6, base*1.05), dz=bz+body*1.25)
    for sx in (-1,1):
        for sy in (-1,1): _place(m, gen_chhatri(body*0.5, body*0.18), t=(sx*body*0.78, sy*body*0.78, bz+body*1.2))
    for sx in (-1,1):
        for sy in (-1,1): _place(m, gen_minaret_d(H*1.0, base*0.11), t=(sx*body*1.62, sy*body*1.62, ph))
    return m
def asm_khmer(H, base, name=""):
    m=Mesh(); add_box(m,-base*2.2,base*2.2,-base*2.2,base*2.2,0,base*0.16); _cornice(m,-base*1.9,base*1.9,-base*1.9,base*1.9,base*0.16,base*0.05,base*0.12)
    add_box(m,-base*1.7,base*1.7,-base*1.7,base*1.7,base*0.21,base*0.40); dz=base*0.40
    _merge(m, gen_sqtower_d(H, base*0.95, 6, 0.83), dz=dz)
    for sx in (-1,1):
        for sy in (-1,1): _place(m, gen_sqtower_d(H*0.62, base*0.52, 6, 0.83), t=(sx*base*1.35, sy*base*1.35, dz))
    for s in (-1,1): add_box(m, base*2.2, base*3.2, s*base*0.5-base*0.08, s*base*0.5+base*0.08, base*0.16, base*0.34)
    return m
def asm_gothic(H, base, name=""):
    m=Mesh(); naveL=base*3.0; naveW=base*1.0; naveH=H*0.42; tw=base*0.58
    add_box(m,-naveL,naveL,-naveW,naveW,0,naveH); _cornice(m,-naveL,naveL,-naveW,naveW,naveH,base*0.04,base*0.08)
    add_box(m,-base*0.7,base*0.7,-naveW*1.95,naveW*1.95,0,naveH*0.95)
    for bx in np.linspace(-naveL*0.7, naveL*0.7, 7):
        _lancets(m, naveW, [bx], naveH*0.45, naveH*0.4, base*0.16, sign=+1); _lancets(m, -naveW, [bx], naveH*0.45, naveH*0.4, base*0.16, sign=-1)
    for sy in (-1,1):
        for bx in np.linspace(-naveL*0.85, naveL*0.85, 6): add_box(m, bx-base*0.05, bx+base*0.05, sy*naveW, sy*naveW+base*0.32, 0, naveH*0.85)
    for sy in (-1,1):
        add_box(m,-naveL,-naveL+tw*1.7,sy*naveW-tw,sy*naveW+tw,0,H*0.80)
        _cornice(m,-naveL,-naveL+tw*1.7,sy*naveW-tw,sy*naveW+tw,H*0.80,base*0.04,base*0.06)
        _lancets(m, -naveL, [sy*naveW], H*0.30, H*0.30, base*0.22, sign=-1)
        _place(m, gen_spire_d(H*0.55, tw*1.8), t=(-naveL+tw*0.85, sy*naveW, H*0.80))
    _rose_window(m, -naveL, naveH*0.72, base*0.55)
    _place(m, gen_spire_d(H*0.78, base*1.0), t=(0,0,naveH)); return m
def asm_maya(H, base, name=""):
    m=Mesh(); top=H*0.85; nv=4; th=np.linspace(0,2*np.pi,nv+1)+np.pi/4; steps=8; zs=[]; rs=[]
    for s in range(steps):
        r=(base*1.2)*(1-s/steps); rt=(base*1.2)*(1-(s+0.85)/steps); z0=top*s/steps; z1=top*(s+1)/steps; zs+=[z0,z1,z1]; rs+=[r,r,rt]
    zs=np.array(zs); rs=np.clip(np.array(rs),0.04*base,None)
    m.add_surface(rs[:,None]*np.cos(th)[None,:], rs[:,None]*np.sin(th)[None,:], zs[:,None]*np.ones(nv+1)[None,:])
    sw=base*0.42; add_box(m,-sw,sw,-sw,sw,top,top+H*0.16); add_box(m,-sw*0.55,sw*0.55,-sw*0.16,sw*0.16,top+H*0.16,top+H*0.34)
    add_box(m,-sw*0.6,sw*0.25,-sw*0.55,sw*0.55,top,top+H*0.10); _steps(m, base*1.25, base*0.18, -base*0.30, base*0.30, 0, top, n=14)
    for sx in (-1,1): add_box(m, sx*base*0.30, sx*base*0.45, -base*0.34, base*0.34, 0, top*1.02)
    return m
def asm_classical(H, base, name=""):
    m=Mesh(); L=base*2.2; W=base*1.2; colh=H*0.62
    for i,sf in enumerate([1.0,0.94,0.88]): add_box(m,-L*sf,L*sf,-W*sf,W*sf, i*base*0.07,(i+1)*base*0.07)
    sb=base*0.21; add_box(m,-L*0.6,L*0.6,-W*0.52,W*0.52,sb,sb+colh*0.96); pts=set()
    for x in np.linspace(-L*0.9,L*0.9,8):
        for y in (-W*0.82,W*0.82): pts.add((round(x,3),round(y,3)))
    for y in np.linspace(-W*0.82,W*0.82,3): pts.add((round(-L*0.9,3),round(y,3))); pts.add((round(L*0.9,3),round(y,3)))
    for (x,y) in pts: _place(m, gen_column(colh, base*0.10), t=(x,y,sb))
    add_box(m,-L,L,-W,W,sb+colh,sb+colh+base*0.16)
    _triglyphs(m, -L*0.95, L*0.95,  W, sb+colh+base*0.02, sb+colh+base*0.14, 9); _triglyphs(m, -L*0.95, L*0.95, -W, sb+colh+base*0.02, sb+colh+base*0.14, 9)
    _cornice(m,-L,L,-W,W,sb+colh+base*0.16,base*0.04,base*0.12)
    rl=L; ph=base*0.5; b=len(m.V); m.V+=[(-rl,-W,0),(rl,-W,0),(rl,W,0),(-rl,W,0),(-rl,0,ph),(rl,0,ph)]
    for f in [(0,1,5),(0,5,4),(3,4,5),(3,5,2),(0,4,3),(1,2,5)]: m.F.append((f[0]+b,f[1]+b,f[2]+b))
    zoff=sb+colh+base*0.16+base*0.04
    for i in range(b,len(m.V)): x,y,z=m.V[i]; m.V[i]=(x,y,z+zoff)
    return m
def asm_chinese(H, base, name=""):
    m=Mesh(); terr=base*2.0; dz=base*0.26; add_box(m,-terr,terr,-terr*0.72,terr*0.72,0,dz); _cornice(m,-terr*0.96,terr*0.96,-terr*0.68,terr*0.68,dz,base*0.04,base*0.08)
    for x in np.linspace(-base*1.35,base*1.35,6):
        for y in (-base*0.85,base*0.85): _place(m,_cyl((x,y,dz),(x,y,dz+H*0.5),base*0.07,8))
    add_box(m,-base*1.5,base*1.5,-base*0.95,base*0.95,dz,dz+H*0.5)
    def roof(L,W,ph,lift,z0):
        b=len(m.V); m.V+=[(-L,-W,z0+ph*lift),(L,-W,z0+ph*lift),(L,W,z0+ph*lift),(-L,W,z0+ph*lift),(-L*0.28,0,z0+ph),(L*0.28,0,z0+ph)]
        for f in [(0,1,5),(0,5,4),(3,4,5),(3,5,2),(0,4,3),(1,2,5)]: m.F.append((f[0]+b,f[1]+b,f[2]+b))
        for sx in (-1,1):
            for sy in (-1,1): add_box(m,sx*L-base*0.12,sx*L+base*0.12,sy*W-base*0.12,sy*W+base*0.12,z0+ph*lift,z0+ph*lift+base*0.18)
    roof(base*2.4, base*1.5, H*0.24, 0.35, dz+H*0.5); roof(base*1.7, base*1.05, H*0.20, 0.33, dz+H*0.5+H*0.22); return m
def asm_stupa(H, base, name=""):
    m=Mesh(); R=base*1.3; add_box(m,-R*1.25,R*1.25,-R*1.25,R*1.25,0,base*0.22); dz=base*0.22
    phi=np.linspace(0,np.pi/2,20); th=np.linspace(0,2*np.pi,32)
    m.add_surface(R*np.cos(phi)[:,None]*np.cos(th)[None,:], R*np.cos(phi)[:,None]*np.sin(th)[None,:], dz+R*np.sin(phi)[:,None]*np.ones(48)[None,:])
    top=dz+R; add_box(m,-base*0.24,base*0.24,-base*0.24,base*0.24,top,top+base*0.26)
    for i in range(3):
        zc=top+base*0.26+i*base*0.22; rd=base*(0.40-0.10*i); cu=np.linspace(0,2*np.pi,24)
        m.add_surface((rd*np.cos(cu))[None,:]*np.ones(2)[:,None],(rd*np.sin(cu))[None,:]*np.ones(2)[:,None], np.array([zc,zc+base*0.04])[:,None]*np.ones(24)[None,:])
    _place(m,_cyl((0,0,top+base*0.26),(0,0,top+base*0.9),base*0.05,8))
    for ang in (0,np.pi/2,np.pi,3*np.pi/2):
        gx,gy=R*1.35*np.cos(ang),R*1.35*np.sin(ang)
        _place(m,_cyl((gx-base*0.25,gy,0),(gx-base*0.25,gy,base*0.8),base*0.06,8)); _place(m,_cyl((gx+base*0.25,gy,0),(gx+base*0.25,gy,base*0.8),base*0.06,8))
        add_box(m,gx-base*0.32,gx+base*0.32,gy-base*0.06,gy+base*0.06,base*0.8,base*0.92)
    return m
def asm_fort(H, base, name=""):
    m=Mesh(); L=base*2.5; wt=base*0.22; wh=H*0.5
    for (x0,x1,y0,y1) in [(-L,L,-L,-L+wt),(-L,L,L-wt,L),(-L,-L+wt,-L,L),(L-wt,L,-L,L)]:
        add_box(m,x0,x1,y0,y1,0,wh); _crenellate(m,x0,x1,y0,y1,wh,base*0.22,base*0.16,base*0.22)
    for sx in (-1,1):
        for sy in (-1,1):
            _place(m,_cyl((sx*L,sy*L,0),(sx*L,sy*L,wh*1.2),base*0.38,18))
            for k in range(10):
                a=2*np.pi*k/10; bx=sx*L+base*0.38*np.cos(a); by=sy*L+base*0.38*np.sin(a)
                add_box(m, bx-base*0.06, bx+base*0.06, by-base*0.06, by+base*0.06, wh*1.2, wh*1.2+base*0.18)
    add_box(m,L-wt,L+base*0.35,-base*0.55,base*0.55,0,wh*1.35); return m
def asm_javanese(H, base, name=""):
    m=Mesh(); sq=base*2.4; th=np.linspace(0,2*np.pi,24)
    for i in range(5):
        s=sq*(1-i*0.13); h=base*0.16; add_box(m,-s,s,-s,s, i*h, (i+1)*h); _cornice(m,-s,s,-s,s,(i+1)*h,base*0.03,base*0.05)
        _niches4(m, s, base*0.18, h*0.7, i*h+h*0.15, base*0.05, deity=False)
    z=5*base*0.16
    for i in range(3):
        rr=sq*0.62*(1-i*0.20); add_box(m,-rr,rr,-rr,rr,z,z+base*0.04); ns=16-i*4
        for k in range(ns):
            a=2*np.pi*k/ns; cx,cy=rr*0.86*np.cos(a),rr*0.86*np.sin(a)
            du=np.linspace(0,np.pi/2,6); dr=base*0.12*np.cos(du); dz=z+base*0.04+base*0.22*np.sin(du)
            sub=Mesh(); sub.add_surface(dr[:,None]*np.cos(th)[None,:], dr[:,None]*np.sin(th)[None,:], dz[:,None]*np.ones(40)[None,:]); _place(m, sub, t=(cx,cy,0))
        z+=base*0.18
    du=np.linspace(0,np.pi/2,10); dr=base*0.55*np.cos(du); dz=z+base*0.9*np.sin(du)
    m.add_surface(dr[:,None]*np.cos(th)[None,:], dr[:,None]*np.sin(th)[None,:], dz[:,None]*np.ones(40)[None,:]); return m

def gen_frustum(wb, db, wt, dt, H, z0=0.0):
    m=Mesh(); b=len(m.V)
    m.V+=[(-wb/2,-db/2,z0),(wb/2,-db/2,z0),(wb/2,db/2,z0),(-wb/2,db/2,z0),(-wt/2,-dt/2,z0+H),(wt/2,-dt/2,z0+H),(wt/2,dt/2,z0+H),(-wt/2,dt/2,z0+H)]
    for f in [(0,2,1),(0,3,2),(4,5,6),(4,6,7),(0,1,5),(0,5,4),(1,2,6),(1,6,5),(2,3,7),(2,7,6),(3,0,4),(3,4,7)]: m.F.append((f[0]+b,f[1]+b,f[2]+b))
    return m
def gen_obelisk(H, w):
    m=Mesh(); _merge(m, gen_frustum(w,w,w*0.6,w*0.6,H*0.9)); b=len(m.V); wt=w*0.6
    m.V+=[(-wt/2,-wt/2,H*0.9),(wt/2,-wt/2,H*0.9),(wt/2,wt/2,H*0.9),(-wt/2,wt/2,H*0.9),(0,0,H)]
    for f in [(0,1,4),(1,2,4),(2,3,4),(3,0,4)]: m.F.append((f[0]+b,f[1]+b,f[2]+b))
    return m
def gen_onion_dome(H, dia, nv=32):
    m=Mesh(); t=np.linspace(0,1,24); th=np.linspace(0,2*np.pi,nv); drum=0.45; dh=H*drum
    z=np.where(t<drum,(t/drum)*dh, dh+(t-drum)/(1-drum)*(H-dh)); rr=np.empty_like(t)
    for i,ti in enumerate(t):
        if ti<drum: rr[i]=dia/2
        else: u=(ti-drum)/(1-drum); rr[i]=(dia/2)*(1+0.6*np.sin(np.pi*u))*(1-u)**0.6
    m.add_surface(rr[:,None]*np.cos(th)[None,:], rr[:,None]*np.sin(th)[None,:], z[:,None]*np.ones(nv)[None,:])
    add_box(m,-dia*0.03,dia*0.03,-dia*0.10,dia*0.10,H,H+dia*0.22); add_box(m,-dia*0.10,dia*0.10,-dia*0.03,dia*0.03,H+dia*0.07,H+dia*0.13)
    return m
def gen_pagoda(H, base, tiers=5):
    m=Mesh(); z=0.0; w=base*1.0
    for s in range(tiers):
        th_h=H/tiers; bodyh=th_h*0.55; add_box(m,-w/2,w/2,-w/2,w/2,z,z+bodyh); L=w*0.92; ph=th_h*0.5; b=len(m.V)
        m.V+=[(-L,-L,z+bodyh),(L,-L,z+bodyh),(L,L,z+bodyh),(-L,L,z+bodyh),(0,0,z+bodyh+ph)]
        for f in [(0,1,4),(1,2,4),(2,3,4),(3,0,4)]: m.F.append((f[0]+b,f[1]+b,f[2]+b))
        for sx in (-1,1):
            for sy in (-1,1): add_box(m,sx*L-base*0.06,sx*L+base*0.02,sy*L-base*0.06,sy*L+base*0.02,z+bodyh+ph*0.1,z+bodyh+ph*0.1+base*0.10)
        z+=th_h; w*=0.84
    _place(m,_cyl((0,0,z),(0,0,z+H*0.14),base*0.035,8)); return m
def _toron(m, faceR, half, zs, proj, axis="x"):
    for z in zs:
        for y in np.linspace(-half,half,6):
            if axis=="x": add_box(m,faceR,faceR+proj,y-proj*0.15,y+proj*0.15,z-proj*0.15,z+proj*0.15)
            else: add_box(m,y-proj*0.15,y+proj*0.15,faceR,faceR+proj,z-proj*0.15,z+proj*0.15)
def _semidome(m, cx, cy, dia, z0, nv=24):
    th=np.linspace(0,2*np.pi,nv); phi=np.linspace(0,np.pi/2,12)
    m.add_surface((dia/2)*np.cos(phi)[:,None]*np.cos(th)[None,:]+cx, (dia/2)*np.cos(phi)[:,None]*np.sin(th)[None,:]+cy, z0+(dia/2)*np.sin(phi)[:,None]*np.ones(nv)[None,:])
def asm_egyptian(H, base, name=""):
    m=Mesh()
    for sy in (-1,1): _merge(m, gen_frustum(base*0.7,base*1.4,base*0.55,base*1.2,H*0.7), 0, sy*base*0.9)
    add_box(m,-base*0.18,base*0.18,-base*0.9,base*0.9,0,H*0.55)
    for sy in (-1,1): _place(m, gen_obelisk(H*0.85, base*0.16), t=(base*1.0, sy*base*0.55, 0))
    depth=base*2.4; add_box(m,-base*1.0-depth,-base*0.7,-base*1.5,base*1.5,0,base*0.12)
    for x in np.linspace(-base*1.0-depth+base*0.3,-base*1.0,4):
        for y in np.linspace(-base*1.1,base*1.1,4): _place(m,_cyl((x,y,base*0.12),(x,y,base*0.12+H*0.5),base*0.13,10))
    add_box(m,-base*1.0-depth,-base*1.0+base*0.05,-base*1.5,base*1.5,0,H*0.55); return m
def asm_ziggurat(H, base, name=""):
    m=Mesh(); z=0
    for s in range(4):
        f=1-s*0.22; _merge(m, gen_frustum(base*2.2*f,base*1.7*f,base*2.0*f,base*1.55*f,H*0.22), 0,0,z); z+=H*0.22
    add_box(m,-base*0.5,base*0.5,-base*0.4,base*0.4,z,z+H*0.14); _steps(m, base*1.5, base*0.0, -base*0.22, base*0.22, 0, z, n=18); return m
def asm_persian(H, base, name=""):
    m=Mesh(); add_box(m,-base*1.6,base*1.6,-base*0.9,base*0.9,0,base*0.12)
    add_box(m,-base*1.1,base*1.1,-base*0.7,base*0.7,base*0.12,base*0.12+H*0.55)
    add_box(m,-base*0.45,base*0.45,base*0.7,base*0.9,base*0.12,base*0.12+H*0.62); add_box(m,-base*0.32,base*0.32,base*0.86,base*0.95,base*0.12,base*0.12+H*0.5)
    _merge(m, gen_dome_d(H*0.5, base*1.0), dz=base*0.12+H*0.55)
    for sx in (-1,1): _place(m, gen_minaret_d(H*0.95, base*0.09), t=(sx*base*0.55, base*0.95, base*0.12))
    return m
def asm_russian(H, base, name=""):
    m=Mesh(); body=base*1.0
    _moldings(m,-body*1.3,body*1.3,-body*1.3,body*1.3,0,courses=2,ch=base*0.12,step=base*0.08)
    add_box(m,-body,body,-body,body,base*0.24,base*0.24+H*0.45); _cornice(m,-body,body,-body,body,base*0.24+H*0.45,base*0.05,base*0.08)
    dz=base*0.24+H*0.45; _place(m,_cyl((0,0,dz),(0,0,dz+H*0.18),base*0.34,24)); _merge(m, gen_onion_dome(H*0.42, base*0.78), dz=dz+H*0.18)
    for sx in (-1,1):
        for sy in (-1,1):
            cx,cy=sx*body*0.62,sy*body*0.62; _place(m,_cyl((cx,cy,dz),(cx,cy,dz+H*0.12),base*0.16,16)); _merge(m, gen_onion_dome(H*0.26, base*0.34), cx, cy, dz+H*0.12)
    return m
def asm_japanese(H, base, name=""):
    m=Mesh(); add_box(m,-base*1.4,base*1.4,-base*1.4,base*1.4,0,base*0.16); _merge(m, gen_pagoda(H, base*0.9), dz=base*0.16); return m
def asm_inca(H, base, name=""):
    m=Mesh()
    for s in range(4):
        f=1-s*0.16; add_box(m,-base*2.2*f,base*2.2*f,-base*1.6*f,base*1.6*f, s*base*0.18,(s+1)*base*0.18)
    dz=4*base*0.18; _merge(m, gen_frustum(base*1.4,base*1.0,base*1.15,base*0.82,H*0.5), 0,0,dz)
    for nx in (1,-1):
        for yy in np.linspace(-base*0.3,base*0.3,3): add_box(m, nx*base*0.6, nx*base*0.72, yy-base*0.1, yy+base*0.1, dz+H*0.06, dz+H*0.3)
    _steps(m, base*2.2, base*0.0, -base*0.25, base*0.25, 0, dz, n=16); return m
def asm_sudano(H, base, name=""):
    m=Mesh(); _merge(m, gen_frustum(base*2.2,base*1.8,base*1.9,base*1.55,H*0.5))
    for sx in (-1,1):
        for x in np.linspace(-base*0.8,base*0.8,4):
            add_box(m, x-base*0.07,x+base*0.07, sx*base*0.9, sx*base*0.9+base*0.18, 0, H*0.56)
            add_box(m, x-base*0.05,x+base*0.05, sx*base*0.9+base*0.02, sx*base*0.9+base*0.12, H*0.56, H*0.64)
    add_box(m,-base*0.45,base*0.45,-base*1.0,-base*0.7,0,H*0.78)
    _toron(m, base*0.95, base*0.7, np.linspace(H*0.12,H*0.46,3), base*0.14, axis="y"); _toron(m,-base*0.95-base*0.14, base*0.7, np.linspace(H*0.12,H*0.46,3), base*0.14, axis="y")
    return m
def asm_byzantine(H, base, name=""):
    m=Mesh(); body=base*1.3; add_box(m,-body,body,-body,body,0,H*0.4); _cornice(m,-body,body,-body,body,H*0.4,base*0.05,base*0.08)
    for cx,cy in [(body*0.9,0),(-body*0.9,0),(0,body*0.9),(0,-body*0.9)]: _semidome(m, cx, cy, base*1.0, H*0.3)
    add_box(m,-body*0.7,body*0.7,-body*0.7,body*0.7,H*0.4,H*0.5); phi=np.linspace(0,np.pi/2,16); th=np.linspace(0,2*np.pi,40)
    m.add_surface(body*0.78*np.cos(phi)[:,None]*np.cos(th)[None,:], body*0.78*np.cos(phi)[:,None]*np.sin(th)[None,:], H*0.5+body*0.6*np.sin(phi)[:,None]*np.ones(40)[None,:])
    for sx in (-1,1):
        for sy in (-1,1): add_box(m,sx*body-base*0.12,sx*body+base*0.12,sy*body-base*0.12,sy*body+base*0.12,0,H*0.46)
    return m
def asm_romanesque(H, base, name=""):
    m=Mesh(); naveL=base*2.6; naveW=base*1.0; naveH=H*0.5; tw=base*0.5
    add_box(m,-naveL,naveL,-naveW,naveW,0,naveH); _cornice(m,-naveL,naveL,-naveW,naveW,naveH,base*0.04,base*0.07)
    for sy in (-1,1):
        for x in np.linspace(-naveL*0.8,naveL*0.8,9): _place(m,_cyl((x,sy*naveW,0),(x,sy*naveW,naveH*0.7),base*0.05,8))
    th=np.linspace(-np.pi/2,np.pi/2,18); apse_r=naveW*1.1
    m.add_surface(naveL+apse_r*np.cos(th)[None,:]*np.ones(6)[:,None], apse_r*np.sin(th)[None,:]*np.ones(6)[:,None], np.linspace(0,naveH*0.9,6)[:,None]*np.ones(18)[None,:])
    for sy in (-1,1):
        add_box(m,-naveL-tw*0.6,-naveL+tw,sy*naveW-tw,sy*naveW+tw,0,H*0.78)
        b=len(m.V); cx=-naveL+tw*0.2; cy=sy*naveW; st=tw
        m.V+=[(cx-st,cy-st,H*0.78),(cx+st,cy-st,H*0.78),(cx+st,cy+st,H*0.78),(cx-st,cy+st,H*0.78),(cx,cy,H*0.95)]
        for f in [(0,1,4),(1,2,4),(2,3,4),(3,0,4)]: m.F.append((f[0]+b,f[1]+b,f[2]+b))
    return m
def asm_aztec(H, base, name=""):
    m=Mesh(); top=H*0.8; steps=6; nv=4; th=np.linspace(0,2*np.pi,nv+1)+np.pi/4; zs=[]; rs=[]
    for s in range(steps):
        r=(base*1.5)*(1-s/steps); rt=(base*1.5)*(1-(s+0.8)/steps); z0=top*s/steps; z1=top*(s+1)/steps; zs+=[z0,z1,z1]; rs+=[r,r,rt]
    zs=np.array(zs); rs=np.clip(np.array(rs),0.05*base,None)
    m.add_surface(rs[:,None]*np.cos(th)[None,:], rs[:,None]*np.sin(th)[None,:], zs[:,None]*np.ones(nv+1)[None,:])
    for sy in (-1,1): add_box(m, sy*base*0.32-base*0.28, sy*base*0.32+base*0.28, -base*0.3, base*0.3, top, top+H*0.18)
    for sy in (-1,1): _steps(m, base*1.55, base*0.0, sy*base*0.34-base*0.16, sy*base*0.34+base*0.16, 0, top, n=12)
    return m

def gen_bell_stupa(H, dia, nv=28):
    m=Mesh(); t=np.linspace(0,1,22); th=np.linspace(0,2*np.pi,nv); rr=np.empty_like(t)
    for i,ti in enumerate(t):
        if ti<0.35: rr[i]=(dia/2)*(1-0.10*ti)
        elif ti<0.62: u=(ti-0.35)/0.27; rr[i]=(dia/2)*(0.9-0.5*u)
        else: u=(ti-0.62)/0.38; rr[i]=(dia/2)*0.40*(1-u)**1.4
    m.add_surface(rr[:,None]*np.cos(th)[None,:], rr[:,None]*np.sin(th)[None,:], (t*H)[:,None]*np.ones(nv)[None,:])
    return m
def gen_prang(H, base, nv=12):
    m=Mesh(); th=np.linspace(0,2*np.pi,nv); t=np.linspace(0,1,20)
    rr=(base/2)*(1-0.7*t**1.6)*(1+0.05*np.cos(t*np.pi*14)); rr=np.clip(rr,0.03*base,None)
    m.add_surface(rr[:,None]*np.cos(th)[None,:], rr[:,None]*np.sin(th)[None,:], (t*H)[:,None]*np.ones(nv)[None,:])
    return m
def asm_tibetan(H, base, name=""):
    m=Mesh(); _merge(m, gen_frustum(base*2.4,base*1.8,base*2.1,base*1.55,H*0.5)); _merge(m, gen_frustum(base*1.6,base*1.2,base*1.4,base*1.05,H*0.7))
    b=len(m.V); L=base*0.9; m.V+=[(-L,-L,H*0.7),(L,-L,H*0.7),(L,L,H*0.7),(-L,L,H*0.7),(0,0,H*0.86)]
    for f in [(0,1,4),(1,2,4),(2,3,4),(3,0,4)]: m.F.append((f[0]+b,f[1]+b,f[2]+b))
    for z in (H*0.2,H*0.34,H*0.48):
        for y in np.linspace(-base*0.75,base*0.75,5): add_box(m, base*1.02, base*1.07, y-base*0.05, y+base*0.05, z, z+base*0.13)
    return m
def asm_moorish(H, base, name=""):
    m=Mesh(); add_box(m,-base*1.6,base*1.6,-base*1.2,base*1.2,0,base*0.15)
    for sy in (-1,1):
        for x in np.linspace(-base*1.3,base*1.3,7): _place(m,_cyl((x,sy*base*1.0,base*0.15),(x,sy*base*1.0,base*0.15+H*0.4),base*0.05,8))
    add_box(m,-base*1.4,base*1.4,-base*1.05,base*1.05,base*0.15+H*0.4,base*0.15+H*0.5)
    add_box(m, base*1.0, base*1.5, base*0.7, base*1.2, 0, H*0.95); add_box(m, base*1.02, base*1.48, base*0.72, base*1.18, H*0.95, H*1.06)
    phi=np.linspace(0,np.pi/2,12); th=np.linspace(0,2*np.pi,28)
    m.add_surface(base*0.5*np.cos(phi)[:,None]*np.cos(th)[None,:], base*0.5*np.cos(phi)[:,None]*np.sin(th)[None,:], base*0.15+base*0.5*np.sin(phi)[:,None]*np.ones(28)[None,:])
    return m
def asm_achaemenid(H, base, name=""):
    m=Mesh(); add_box(m,-base*2.0,base*2.0,-base*1.6,base*1.6,0,base*0.3); dz=base*0.3
    for x in np.linspace(-base*1.4,base*1.4,5):
        for y in np.linspace(-base*1.1,base*1.1,4):
            _place(m, gen_column(H*0.9, base*0.06), t=(x,y,dz)); add_box(m, x-base*0.12,x+base*0.12, y-base*0.05,y+base*0.05, dz+H*0.88, dz+H*0.97)
    _steps(m, base*2.0, base*1.4, -base*0.4, base*0.4, 0, dz, n=8); return m
def asm_nubian(H, base, name=""):
    m=Mesh()
    for dx,sc in [(0,1.0),(base*1.7,0.72),(-base*1.6,0.6)]:
        bw=base*0.85*sc; _merge(m, gen_frustum(bw,bw,bw*0.05,bw*0.05,H*sc), dx)
        add_box(m, dx+bw*0.5, dx+bw*0.5+base*0.4*sc, -base*0.25*sc, base*0.25*sc, 0, H*sc*0.22)
    return m
def asm_burmese(H, base, name=""):
    m=Mesh()
    for i in range(3):
        s=base*1.6*(1-i*0.18); add_box(m,-s,s,-s,s, i*base*0.2,(i+1)*base*0.2)
    dz=3*base*0.2; add_box(m,-base*0.9,base*0.9,-base*0.9,base*0.9,dz,dz+base*0.15)
    _merge(m, gen_bell_stupa(H*0.62, base*1.1), dz=dz+base*0.15)
    for sx in (-1,1):
        for sy in (-1,1): _merge(m, gen_bell_stupa(H*0.22, base*0.34), sx*base*1.1, sy*base*1.1, dz)
    return m
def asm_thai(H, base, name=""):
    m=Mesh()
    for i in range(3): s=base*1.3*(1-i*0.16); add_box(m,-s,s,-s,s,i*base*0.18,(i+1)*base*0.18)
    dz=3*base*0.18; _merge(m, gen_prang(H*0.7, base*1.0), dz=dz)
    for sx in (-1,1):
        for sy in (-1,1): _merge(m, gen_prang(H*0.28, base*0.3), sx*base*0.7, sy*base*0.7, dz)
    return m

def gen_cone(H, base, nv=14):
    m=Mesh(); th=np.linspace(0,2*np.pi,nv); t=np.linspace(0,1,2); rr=(base/2)*(1-t)
    m.add_surface(rr[:,None]*np.cos(th)[None,:], rr[:,None]*np.sin(th)[None,:], (t*H)[:,None]*np.ones(nv)[None,:]); return m
def gen_stela(H, w):
    m=Mesh(); add_box(m,-w/2,w/2,-w*0.3,w*0.3,0,H*0.9)
    b=len(m.V); m.V+=[(-w/2,-w*0.3,H*0.9),(w/2,-w*0.3,H*0.9),(w/2,w*0.3,H*0.9),(-w/2,w*0.3,H*0.9),(0,0,H)]
    for f in [(0,1,4),(1,2,4),(2,3,4),(3,0,4)]: m.F.append((f[0]+b,f[1]+b,f[2]+b))
    for z in np.linspace(H*0.15,H*0.8,5): add_box(m,-w*0.42,w*0.42,w*0.3,w*0.34,z,z+H*0.03)
    return m
def _hall(m, x0,x1,y0,y1,z0,colh,nx,ny,r):
    for x in np.linspace(x0,x1,nx):
        for y in np.linspace(y0,y1,ny): _place(m,_cyl((x,y,z0),(x,y,z0+colh),r,8))
def asm_sinhalese(H, base, name=""):
    m=Mesh()
    for i in range(3): s=base*1.5*(1-i*0.12); add_box(m,-s,s,-s,s,i*base*0.12,(i+1)*base*0.12)
    dz=3*base*0.12; R=base*1.25; phi=np.linspace(0,np.pi/2,18); th=np.linspace(0,2*np.pi,30)
    m.add_surface(R*np.cos(phi)[:,None]*np.cos(th)[None,:], R*np.cos(phi)[:,None]*np.sin(th)[None,:], dz+R*1.15*np.sin(phi)[:,None]*np.ones(30)[None,:])
    top=dz+R*1.15; add_box(m,-base*0.2,base*0.2,-base*0.2,base*0.2,top,top+base*0.22); _merge(m, gen_cone(H*0.4, base*0.5), dz=top+base*0.22); return m
def asm_korean(H, base, name=""):
    m=Mesh(); add_box(m,-base*1.8,base*1.8,-base*1.2,base*1.2,0,base*0.18); dz=base*0.18
    _hall(m,-base*1.4,base*1.4,-base*0.85,base*0.85,dz,H*0.42,5,2,base*0.07); add_box(m,-base*1.4,base*1.4,-base*0.9,base*0.9,dz+H*0.42,dz+H*0.48)
    b=len(m.V); L=base*2.1; W=base*1.4; ph=H*0.3; z0=dz+H*0.48
    m.V+=[(-L,-W,z0),(L,-W,z0),(L,W,z0),(-L,W,z0),(-L*0.32,0,z0+ph),(L*0.32,0,z0+ph)]
    for f in [(0,1,5),(0,5,4),(3,4,5),(3,5,2),(0,4,3),(1,2,5)]: m.F.append((f[0]+b,f[1]+b,f[2]+b))
    return m
def asm_umayyad(H, base, name=""):
    m=Mesh(); L=base*2.2; add_box(m,-L,L,-L,L,0,base*0.12)
    for (x0,x1,y0,y1) in [(-L,L,-L,-L+base*0.15),(-L,L,L-base*0.15,L),(-L,-L+base*0.15,-L,L)]: add_box(m,x0,x1,y0,y1,0,H*0.45)
    _hall(m,-L*0.85,L*0.85,L*0.2,L*0.85,base*0.12,H*0.5,7,4,base*0.06); add_box(m,-L*0.85,L*0.85,L*0.2,L*0.85,base*0.12+H*0.5,base*0.12+H*0.56)
    add_box(m,L*0.7,L*0.95,L*0.7,L*0.95,0,H*0.95); add_box(m,L*0.72,L*0.93,L*0.72,L*0.93,H*0.95,H*1.05); return m
def asm_mamluk(H, base, name=""):
    m=Mesh(); body=base*1.1
    for i in range(4): z=i*base*0.2; add_box(m,-body,body,-body,body,z,z+base*0.2)
    bz=4*base*0.2; _merge(m, gen_dome_d(H*0.5, base*0.95, ribs=12), dz=bz)
    for i in range(4): r=base*(0.16-0.02*i); add_box(m,body*0.8-r,body*0.8+r,body*0.8-r,body*0.8+r,i*H*0.25,(i+1)*H*0.25)
    _merge(m, gen_cone(H*0.18, base*0.3), dx=body*0.8, dy=body*0.8, dz=H*1.0); return m
def asm_aksumite(H, base, name=""):
    m=Mesh(); add_box(m,-base*1.8,base*1.8,-base*1.2,base*1.2,0,base*0.16)
    for dx,sc in [(0,1.0),(base*1.3,0.6),(-base*1.2,0.7),(base*2.0,0.4)]: _merge(m, gen_stela(H*sc, base*0.4*sc), dx, 0, base*0.16)
    return m
def asm_swahili(H, base, name=""):
    m=Mesh(); add_box(m,-base*1.6,base*1.6,-base*1.0,base*1.0,0,H*0.5); _cornice(m,-base*1.6,base*1.6,-base*1.0,base*1.0,H*0.5,base*0.05,base*0.08)
    add_box(m,-base*0.3,base*0.3,-base*1.0,-base*0.78,H*0.4,H*0.7); _place(m,_cyl((base*1.9,0,0),(base*1.9,0,H*0.9),base*0.22,16)); return m
def asm_zimbabwe(H, base, name=""):
    m=Mesh(); R=base*1.7; th=np.linspace(0.35,2*np.pi-0.35,26)
    for a in th:
        x,y=R*np.cos(a), R*0.82*np.sin(a); add_box(m,x-base*0.14,x+base*0.14,y-base*0.14,y+base*0.14,0,H*0.55)
    _merge(m, gen_cone(H*0.6, base*0.7), dx=R*0.35); _merge(m, gen_cone(H*0.32, base*0.4), dx=R*0.35+base*0.7); return m
def asm_puebloan(H, base, name=""):
    m=Mesh()
    for i in range(4):
        z=i*H*0.22; w=base*2.0-i*base*0.4
        for x in np.linspace(-w+base*0.4,w-base*0.4,max(2,5-i)): add_box(m,x-base*0.32,x+base*0.32,-base*0.9+i*base*0.3,-base*0.3+i*base*0.3,z,z+H*0.22)
    for cx in (-base*0.8,base*0.8): _place(m,_cyl((cx,base*0.8,0),(cx,base*0.8,H*0.18),base*0.35,16))
    return m
def asm_colonial(H, base, name=""):
    m=Mesh(); add_box(m,-base*1.4,base*1.4,-base*2.4,base*0.4,0,H*0.45); add_box(m,-base*1.5,base*1.5,base*0.4,base*0.75,0,H*0.6)
    for sx in (-1,1):
        add_box(m,sx*base*1.1-base*0.3,sx*base*1.1+base*0.3,base*0.45,base*0.75,0,H*0.78); _merge(m, gen_dome_d(H*0.2, base*0.4, ribs=8), sx*base*1.1, base*0.6, H*0.78)
    _merge(m, gen_dome_d(H*0.4, base*0.8), dz=H*0.45); return m
def asm_renaissance(H, base, name=""):
    m=Mesh(); add_box(m,-base*1.5,base*1.5,-base*1.5,base*1.5,0,H*0.4); _cornice(m,-base*1.5,base*1.5,-base*1.5,base*1.5,H*0.4,base*0.05,base*0.1)
    _hall(m,-base*1.3,base*1.3,base*1.5,base*1.5,0,H*0.42,6,1,base*0.09)
    b=len(m.V); ph=base*0.7; z=H*0.42
    m.V+=[(-base*1.4,base*1.5,z),(base*1.4,base*1.5,z),(base*1.4,base*1.8,z),(-base*1.4,base*1.8,z),(-base*1.4,base*1.65,z+ph),(base*1.4,base*1.65,z+ph)]
    for f in [(0,1,5),(0,5,4),(3,4,5),(3,5,2),(0,4,3),(1,2,5)]: m.F.append((f[0]+b,f[1]+b,f[2]+b))
    add_box(m,-base*0.9,base*0.9,-base*0.9,base*0.9,H*0.4,H*0.55); phi=np.linspace(0,np.pi/2,16); th=np.linspace(0,2*np.pi,32)
    m.add_surface(base*0.85*np.cos(phi)[:,None]*np.cos(th)[None,:], base*0.85*np.cos(phi)[:,None]*np.sin(th)[None,:], H*0.55+base*0.8*np.sin(phi)[:,None]*np.ones(32)[None,:])
    add_box(m,-base*0.18,base*0.18,-base*0.18,base*0.18,H*0.55+base*0.8,H*0.55+base*1.0); return m
def asm_baroque(H, base, name=""):
    m=Mesh(); add_box(m,-base*1.6,base*1.6,-base*1.4,base*0.6,0,H*0.5); add_box(m,-base*1.7,base*1.7,base*0.55,base*0.9,0,H*0.62)
    for sx in (-1,1):
        add_box(m,sx*base*1.25-base*0.32,sx*base*1.25+base*0.32,base*0.55,base*0.9,0,H*0.85); _merge(m, gen_onion_dome(H*0.22, base*0.5), sx*base*1.25, base*0.72, H*0.85)
    _merge(m, gen_dome_d(H*0.5, base*0.9), dz=H*0.5); return m
def asm_castle(H, base, name=""):
    m=Mesh(); L=base*2.2; wt=base*0.22; wh=H*0.45
    for (x0,x1,y0,y1) in [(-L,L,-L,-L+wt),(-L,L,L-wt,L),(-L,-L+wt,-L,L),(L-wt,L,-L,L)]:
        add_box(m,x0,x1,y0,y1,0,wh); _crenellate(m,x0,x1,y0,y1,wh,base*0.22,base*0.16,base*0.2)
    for sx in (-1,1):
        for sy in (-1,1):
            _place(m,_cyl((sx*L,sy*L,0),(sx*L,sy*L,wh*1.4),base*0.36,16)); _merge(m, gen_cone(H*0.3,base*0.85), sx*L, sy*L, wh*1.4)
    add_box(m,-base*0.7,base*0.7,-base*0.7,base*0.7,0,H*0.9); _crenellate(m,-base*0.7,base*0.7,-base*0.7,-base*0.55,H*0.9,base*0.2,base*0.14,base*0.18); return m
def asm_modernist(H, base, name=""):
    m=Mesh()
    for x in np.linspace(-base*1.6,base*1.6,5):
        for y in (-base*0.7,base*0.7): _place(m,_cyl((x,y,0),(x,y,H*0.18),base*0.08,10))
    for i in range(4): z=H*0.18+i*H*0.2; add_box(m,-base*1.9,base*1.9,-base*0.95,base*0.95,z,z+H*0.14)
    add_box(m,-base*1.9,base*1.9,-base*0.95,base*0.95,H*0.18,H*0.18+4*H*0.2); return m
def asm_vernacular(H, base, name=""):
    m=Mesh()
    for dx,dy,sc in [(0,0,1.0),(base*1.4,base*0.6,0.7),(-base*1.2,base*0.4,0.75),(base*0.3,-base*1.3,0.6)]:
        _place(m,_cyl((dx,dy,0),(dx,dy,H*0.4*sc),base*0.6*sc,14)); _merge(m, gen_cone(H*0.45*sc, base*1.5*sc), dx, dy, H*0.4*sc)
    return m
def asm_industrial(H, base, name=""):
    m=Mesh(); add_box(m,-base*2.0,base*1.0,-base*1.2,base*1.2,0,H*0.4)
    for x in np.linspace(-base*1.9,base*0.9,6):
        b=len(m.V); m.V+=[(x,-base*1.2,H*0.4),(x+base*0.5,-base*1.2,H*0.4),(x+base*0.5,base*1.2,H*0.4),(x,base*1.2,H*0.4),(x,-base*1.2,H*0.58),(x,base*1.2,H*0.58)]
        for f in [(0,4,5),(0,5,3),(0,1,4),(3,5,2)]: m.F.append((f[0]+b,f[1]+b,f[2]+b))
    _place(m,_cyl((base*1.6,0,0),(base*1.6,0,H*1.0),base*0.3,16)); add_box(m,base*1.45,base*1.75,-base*0.15,base*0.15,H*1.0,H*1.06); return m

ASSEMBLERS = {
    "Kalinga (Odisha temple)": asm_kalinga, "Nagara (North Indian temple)": asm_nagara,
    "Dravidian (South Indian temple)": asm_dravidian, "Indo-Islamic / Mughal": asm_mughal,
    "Indian Buddhist / rock-cut": asm_stupa, "Indian fort architecture": asm_fort,
    "Khmer (Angkor)": asm_khmer, "Gothic": asm_gothic, "Greco-Roman classical": asm_classical,
    "Maya": asm_maya, "Ottoman": asm_mughal, "Chinese imperial": asm_chinese, "Javanese (Indonesia)": asm_javanese,
    "Ancient Egyptian": asm_egyptian, "Mesopotamian / Assyrian": asm_ziggurat, "Persian (Safavid / Timurid)": asm_persian,
    "Russian Orthodox": asm_russian, "Japanese": asm_japanese, "Andean (Inca)": asm_inca,
    "Sudano-Sahelian (West African)": asm_sudano, "Byzantine": asm_byzantine, "Romanesque": asm_romanesque, "Aztec / Teotihuacan": asm_aztec,
    "Tibetan / Himalayan": asm_tibetan, "Moorish / Andalusian": asm_moorish, "Achaemenid / Persian ancient": asm_achaemenid,
    "Nubian": asm_nubian, "Burmese (Bagan)": asm_burmese, "Thai": asm_thai,
    "Sinhalese (Sri Lanka)": asm_sinhalese, "Korean": asm_korean, "Umayyad / Abbasid Islamic": asm_umayyad, "Mamluk": asm_mamluk,
    "Aksumite (Ethiopian)": asm_aksumite, "Swahili coast": asm_swahili, "Great Zimbabwe": asm_zimbabwe, "Puebloan (Ancestral)": asm_puebloan,
    "Colonial Iberian (Americas)": asm_colonial, "Renaissance": asm_renaissance, "Baroque": asm_baroque,
    "Romano-British / Medieval Castle": asm_castle, "Modernist / 20th-century": asm_modernist,
    "Vernacular / earthen": asm_vernacular, "Industrial heritage": asm_industrial,
}
WORLD_CANON = {
    "Kalinga (Odisha temple)":        {"form":"rekha deul + jagamohana", "h_ratio":2.0, "default_h":76.0, "base_f":0.42, "treatise":"Shilpa Prakasha", "rules":["deul ~2x jagamohana","pancharatha plan","raha niches + angasikhara"]},
    "Nagara (North Indian temple)":   {"form":"shekhari (clustered spires)", "h_ratio":2.5, "default_h":60.0, "base_f":0.42, "treatise":"Samarangana Sutradhara", "rules":["central latina ~2.5x sanctum width","urushringa half-spires"]},
    "Dravidian (South Indian temple)":{"form":"vimana + gopuram", "h_ratio":1.6, "default_h":55.0, "base_f":0.50, "treatise":"Mayamata; Manasara", "rules":["stacked diminishing talas","gopuram + shala roof"]},
    "Indo-Islamic / Mughal":          {"form":"double dome + char-bagh + 4 minarets", "h_ratio":1.75, "default_h":45.0, "base_f":0.55, "treatise":"Timurid-Mughal canon", "rules":["bilateral symmetry","gadrooned dome","pishtaq + jali + chhatris"]},
    "Ottoman":                        {"form":"central dome + pencil minarets", "h_ratio":1.7, "default_h":48.0, "base_f":0.55, "treatise":"Ottoman mosque canon (Sinan)", "rules":["cascading domes","pencil minarets"]},
    "Indian Buddhist / rock-cut":     {"form":"stupa: anda + harmika + chhatra", "h_ratio":1.4, "default_h":30.0, "base_f":0.60, "treatise":"Buddhist stupa canon (Sanchi)", "rules":["hemispherical anda","triple chhatra + toranas"]},
    "Indian fort architecture":       {"form":"ramparts + corner bastions + gateway", "h_ratio":1.0, "default_h":28.0, "base_f":0.55, "treatise":"Indian military architecture", "rules":["battlemented walls","corner bastions"]},
    "Khmer (Angkor)":                 {"form":"quincunx of prasats on terraces", "h_ratio":1.8, "default_h":58.0, "base_f":0.45, "treatise":"Khmer temple-mountain canon", "rules":["five towers in quincunx","stepped terraces + naga causeway"]},
    "Gothic":                         {"form":"nave + west towers + spires", "h_ratio":1.6, "default_h":55.0, "base_f":0.45, "treatise":"Gothic cathedral canon", "rules":["cruciform plan","west towers + spires + pinnacles"]},
    "Greco-Roman classical":          {"form":"peripteral colonnade + pediment", "h_ratio":1.3, "default_h":30.0, "base_f":0.55, "treatise":"Vitruvius; classical orders", "rules":["crepidoma steps","fluted columns + pediment"]},
    "Maya":                           {"form":"stepped pyramid + temple shrine", "h_ratio":1.5, "default_h":40.0, "base_f":0.55, "treatise":"Mesoamerican pyramid canon", "rules":["stepped pyramid + stairway","summit shrine + roof comb"]},
    "Chinese imperial":               {"form":"terrace + tiered hip roofs", "h_ratio":1.2, "default_h":28.0, "base_f":0.55, "treatise":"Yingzao Fashi", "rules":["raised terrace + columns","upturned tiered eaves"]},
    "Javanese (Indonesia)":           {"form":"terraced stupa-mountain (mandala)", "h_ratio":1.4, "default_h":35.0, "base_f":0.55, "treatise":"Borobudur mandala canon", "rules":["square galleries of relief","circular terraces of bell stupas","central crown stupa"]},
    "Ancient Egyptian":               {"form":"pylon temple + hypostyle hall + obelisks", "h_ratio":1.4, "default_h":40.0, "base_f":0.50, "treatise":"Egyptian temple canon (Karnak)", "rules":["battered pylon gateway","hypostyle hall of columns","paired obelisks"]},
    "Mesopotamian / Assyrian":        {"form":"stepped ziggurat + grand stair", "h_ratio":1.3, "default_h":30.0, "base_f":0.55, "treatise":"Mesopotamian ziggurat canon", "rules":["receding rectangular terraces","monumental triple stairway","summit shrine"]},
    "Persian (Safavid / Timurid)":    {"form":"iwan facade + tiled dome + framing minarets", "h_ratio":1.7, "default_h":40.0, "base_f":0.55, "treatise":"Persian Islamic canon (Isfahan)", "rules":["central iwan pishtaq","bulbous tiled dome on a drum","twin framing minarets"]},
    "Russian Orthodox":               {"form":"five onion domes on a cube", "h_ratio":1.6, "default_h":40.0, "base_f":0.50, "treatise":"Russian Orthodox canon", "rules":["cubic body","central plus four corner onion domes","tall drums"]},
    "Japanese":                       {"form":"multi-tier timber pagoda", "h_ratio":2.2, "default_h":35.0, "base_f":0.45, "treatise":"Japanese pagoda canon", "rules":["stacked diminishing storeys","deep overhanging eaves","central sorin mast"]},
    "Andean (Inca)":                  {"form":"terraces + battered trapezoidal halls", "h_ratio":1.2, "default_h":25.0, "base_f":0.55, "treatise":"Inca masonry canon", "rules":["agricultural terraces","battered trapezoidal walls","trapezoidal niches and doors"]},
    "Sudano-Sahelian (West African)": {"form":"mud mosque + toron beams + buttresses", "h_ratio":1.2, "default_h":20.0, "base_f":0.55, "treatise":"Sudano-Sahelian canon (Djenné)", "rules":["battered mud-brick walls","engaged buttress pinnacles","protruding toron beams"]},
    "Byzantine":                      {"form":"central dome on pendentives + semidomes", "h_ratio":1.3, "default_h":40.0, "base_f":0.55, "treatise":"Byzantine canon (Hagia Sophia)", "rules":["central dome","semidomes on a Greek-cross plan","corner buttress piers"]},
    "Romanesque":                     {"form":"nave + round arches + apse + west towers", "h_ratio":1.5, "default_h":35.0, "base_f":0.45, "treatise":"Romanesque canon", "rules":["round-arch arcades","semicircular apse","twin west towers"]},
    "Aztec / Teotihuacan":            {"form":"twin-temple stepped pyramid", "h_ratio":1.4, "default_h":35.0, "base_f":0.55, "treatise":"Mesoamerican twin-pyramid canon", "rules":["broad stepped platform","twin summit shrines","double frontal stairway"]},
    "Tibetan / Himalayan":            {"form":"dzong: battered walls + central utse", "h_ratio":1.2, "default_h":30.0, "base_f":0.55, "treatise":"Tibetan dzong canon", "rules":["battered trapezoidal walls","central utse with gilded roof","recessed window rows"]},
    "Moorish / Andalusian":           {"form":"courtyard arcade + square minaret", "h_ratio":1.4, "default_h":30.0, "base_f":0.55, "treatise":"Moorish canon (Alhambra)", "rules":["horseshoe-arch arcades","square Giralda minaret","ribbed hall dome"]},
    "Achaemenid / Persian ancient":   {"form":"apadana: columned hall on a terrace", "h_ratio":1.5, "default_h":35.0, "base_f":0.55, "treatise":"Achaemenid canon (Persepolis)", "rules":["raised stone terrace","tall slender fluted columns","double-animal capitals"]},
    "Nubian":                         {"form":"cluster of steep narrow pyramids", "h_ratio":1.6, "default_h":30.0, "base_f":0.5, "treatise":"Kushite pyramid canon (Meroë)", "rules":["steep narrow pyramids","offering chapel at base","clustered field"]},
    "Burmese (Bagan)":                {"form":"bell stupa (zedi) on terraces", "h_ratio":1.6, "default_h":40.0, "base_f":0.5, "treatise":"Burmese stupa canon (Bagan)", "rules":["square receding terraces","gilded bell dome","tall hti spire"]},
    "Thai":                           {"form":"prang tower on a redented base", "h_ratio":1.7, "default_h":38.0, "base_f":0.5, "treatise":"Thai temple canon (Ayutthaya)", "rules":["redented stepped base","corn-cob prang","corner subsidiary prangs"]},
    "Sinhalese (Sri Lanka)":          {"form":"dagoba on terraces", "h_ratio":1.5, "default_h":40.0, "base_f":0.55, "treatise":"Sinhalese stupa canon", "rules":["bubble-shaped anda","square harmika","tee and finial"]},
    "Korean":                         {"form":"timber hall with broad hip roof", "h_ratio":1.2, "default_h":24.0, "base_f":0.5, "treatise":"Korean timber canon", "rules":["stone platform","post-and-beam frame","broad curved hip roof"]},
    "Umayyad / Abbasid Islamic":      {"form":"hypostyle mosque + square minaret", "h_ratio":1.2, "default_h":24.0, "base_f":0.55, "treatise":"early Islamic hypostyle canon", "rules":["arcaded courtyard","hypostyle prayer hall","square minaret"]},
    "Mamluk":                         {"form":"stone dome + tiered minaret", "h_ratio":1.5, "default_h":35.0, "base_f":0.5, "treatise":"Mamluk canon (Cairo)", "rules":["carved stone ribbed dome","ablaq striping","tiered minaret"]},
    "Aksumite (Ethiopian)":           {"form":"carved stelae field", "h_ratio":1.8, "default_h":30.0, "base_f":0.5, "treatise":"Aksumite stela canon", "rules":["monolithic carved stelae","false-storey bands","stepped base"]},
    "Swahili coast":                  {"form":"coral mosque + pillar tomb", "h_ratio":1.0, "default_h":18.0, "base_f":0.55, "treatise":"Swahili coral-stone canon", "rules":["flat-roofed coral hall","mihrab niche","pillar tomb"]},
    "Great Zimbabwe":                 {"form":"dry-stone enclosure + conical tower", "h_ratio":1.2, "default_h":20.0, "base_f":0.6, "treatise":"Great Zimbabwe canon", "rules":["mortarless elliptical wall","solid conical tower"]},
    "Puebloan (Ancestral)":           {"form":"terraced cliff dwelling + kivas", "h_ratio":1.2, "default_h":18.0, "base_f":0.55, "treatise":"Ancestral Puebloan canon", "rules":["stacked receding rooms","round sunken kivas"]},
    "Colonial Iberian (Americas)":    {"form":"baroque church + twin bell towers", "h_ratio":1.4, "default_h":30.0, "base_f":0.5, "treatise":"Colonial Iberian canon", "rules":["ornate facade","twin bell towers","crossing dome"]},
    "Renaissance":                    {"form":"dome church + classical portico", "h_ratio":1.5, "default_h":40.0, "base_f":0.5, "treatise":"Renaissance canon (Bramante)", "rules":["Greek-cross mass","pedimented portico","dome on a drum"]},
    "Baroque":                        {"form":"dome + twin towers + curved facade", "h_ratio":1.5, "default_h":40.0, "base_f":0.5, "treatise":"Baroque canon (Bernini)", "rules":["curved facade","twin towers","grand dome"]},
    "Romano-British / Medieval Castle":{"form":"keep + round corner towers + moat", "h_ratio":1.3, "default_h":28.0, "base_f":0.55, "treatise":"medieval castle canon", "rules":["battlemented walls","round corner towers","central keep"]},
    "Modernist / 20th-century":       {"form":"glass slab on pilotis", "h_ratio":1.5, "default_h":40.0, "base_f":0.5, "treatise":"Modernist canon (Le Corbusier)", "rules":["raised on pilotis","ribbon windows","flat roof"]},
    "Vernacular / earthen":           {"form":"cluster of thatched huts", "h_ratio":1.0, "default_h":10.0, "base_f":0.6, "treatise":"vernacular canon", "rules":["round earthen walls","conical thatch roofs","clustered compound"]},
    "Industrial heritage":            {"form":"factory shed + chimney", "h_ratio":1.4, "default_h":20.0, "base_f":0.55, "treatise":"industrial-era canon", "rules":["sawtooth-roof shed","tall chimney","functional massing"]},
}
# Zero-skip: every classifier label -> nearest authored assembler.
FALLBACK_ASSEMBLER = {
    "Sinhalese (Sri Lanka)":"Indian Buddhist / rock-cut","Burmese (Bagan)":"Indian Buddhist / rock-cut","Thai":"Indian Buddhist / rock-cut",
    "Japanese":"Chinese imperial","Korean":"Chinese imperial","Tibetan / Himalayan":"Indian fort architecture",
    "Persian (Safavid / Timurid)":"Indo-Islamic / Mughal","Umayyad / Abbasid Islamic":"Indo-Islamic / Mughal","Moorish / Andalusian":"Indo-Islamic / Mughal",
    "Mamluk":"Indo-Islamic / Mughal","Achaemenid / Persian ancient":"Greco-Roman classical","Mesopotamian / Assyrian":"Maya","Ancient Egyptian":"Maya",
    "Byzantine":"Indo-Islamic / Mughal","Romanesque":"Gothic","Renaissance":"Greco-Roman classical","Baroque":"Greco-Roman classical",
    "Russian Orthodox":"Indo-Islamic / Mughal","Romano-British / Medieval Castle":"Indian fort architecture","Aksumite (Ethiopian)":"Greco-Roman classical",
    "Sudano-Sahelian (West African)":"Indian fort architecture","Swahili coast":"Indo-Islamic / Mughal","Great Zimbabwe":"Indian fort architecture",
    "Nubian":"Maya","Aztec / Teotihuacan":"Maya","Andean (Inca)":"Indian fort architecture","Puebloan (Ancestral)":"Indian fort architecture",
    "Colonial Iberian (Americas)":"Greco-Roman classical","Modernist / 20th-century":"Greco-Roman classical","Vernacular / earthen":"Indian fort architecture",
    "Industrial heritage":"Indian fort architecture",
}
def resolve_assembler(tradition):
    if tradition in ASSEMBLERS: return tradition
    return FALLBACK_ASSEMBLER.get(tradition, "Greco-Roman classical")

# --- documented-dimension calibration (cited published figures; real accuracy) ---
FOOTPRINT_FACTOR={}
for _k,_fn in ASSEMBLERS.items():
    _V=np.array(_fn(60.0,24.0,"x").V); FOOTPRINT_FACTOR[_k]=float((_V[:,0].max()-_V[:,0].min())/24.0)
MONUMENT_DIMENSIONS={
 "konâ":{"h":70.0,"base":None,"src":"Konark deul ~70 m (collapsed; ASI)"},
 "konar":{"h":70.0,"base":None,"src":"Konark deul ~70 m (collapsed; ASI)"},
 "taj mahal":{"h":73.0,"base":95.0,"src":"ASI: 73 m incl. finial, ~95 m plinth"},
 "borobudur":{"h":35.0,"base":123.0,"src":"35 m high on 123 m square base"},
 "angkor":{"h":65.0,"base":None,"src":"central tower ~65 m"},
 "notre-dame":{"h":81.0,"base":None,"src":"Reims west towers ~81 m"},
 "chichen":{"h":30.0,"base":55.3,"src":"El Castillo 30 m, base 55.3 m"},
 "parthenon":{"h":13.7,"base":69.5,"src":"stylobate 69.5x30.9 m, ridge ~13.7 m"},
 "acropolis":{"h":13.7,"base":69.5,"src":"Parthenon stylobate 69.5 m, ridge ~13.7 m"},
 "brihadeeswarar":{"h":66.0,"base":None,"src":"Thanjavur vimana ~66 m"},
 "sanchi":{"h":16.5,"base":36.6,"src":"Great Stupa 16.5 m, dia 36.6 m"},
 "khajuraho":{"h":31.0,"base":None,"src":"Kandariya Mahadeva ~31 m"},
 "qutb":{"h":72.5,"base":None,"src":"Qutb Minar 72.5 m"},
}
# auto-merge an attached monument_dimensions.csv (name,height_m,base_width_m,source_url)
for _p in glob.glob("/kaggle/input/**/monument_dimensions.csv", recursive=True):
    try:
        _df=pd.read_csv(_p)
        for _,_r in _df.iterrows():
            MONUMENT_DIMENSIONS[str(_r["name"]).lower()]={"h":float(_r["height_m"]),
                "base":float(_r["base_width_m"]) if not pd.isna(_r.get("base_width_m")) else None,
                "src":str(_r.get("source_url",_p))}
        print("merged documented dims from", _p, "->", len(_df), "rows")
    except Exception as _e: print("dims csv skip:", _e)
def documented_dims(structure):
    s=str(structure).lower()
    for k,v in MONUMENT_DIMENSIONS.items():
        if k in s: return v
    return None
def _HB(structure, key, sv):
    c=WORLD_CANON[key]; d=documented_dims(structure)
    if d:
        H=d["h"]; base=(d["base"]/FOOTPRINT_FACTOR[key]) if d["base"] else H*c["base_f"]
        return H, base, f"documented {d['h']} m ({d['src']})"
    if sv: H=sv*c["h_ratio"]; return H, H*c["base_f"], f"{sv} m surviving x {c['h_ratio']}"
    H=c["default_h"]; return H, H*c["base_f"], "canon default (calibrate to survey)"

def build_structure(name, location="", survived_m=None, show=True):
    """Retrieve -> classify -> resolve assembler (never skips) -> assemble articulated building -> render + export."""
    try: rm=tangible_search(f"{name} {location}".strip(), top_k=1)
    except Exception: rm=[]
    if len(rm):
        rec=heritage_master[heritage_master["name"]==rm.iloc[0]["name"]].iloc[0]; structure=rec["name"]; tradition=best_tradition(rec)
    else: structure=name; tradition=classify_tradition(name)[0][0]
    key=resolve_assembler(tradition); canon=WORLD_CANON[key]
    H,base,basis=_HB(structure,key,survived_m)
    mesh=ASSEMBLERS[key](H,base,structure)
    safe="".join(ch if ch.isalnum() else "_" for ch in structure)[:40]; open(f"/kaggle/working/{safe}_world.obj","w").write(mesh.obj())
    print("="*72); print("WORLD ASSEMBLER:", structure); print("="*72)
    print(f"Tradition : {tradition}" + (f"  (built as {key})" if key!=tradition else ""))
    print(f"Form      : {canon['form']}  ({len(mesh.V)} verts, {len(mesh.F)} faces)")
    print(f"Height    : {H:.1f} m  <- {basis}")
    for rl in canon["rules"]: print(f"  canon rule: {rl}")
    if show: render_shaded(mesh, f"{structure}\n{canon['form']} - {tradition}", f"/kaggle/working/{safe}_world.png", elev=15, azim=-58)
    return mesh, canon

# --- PROOF: one model, any structure, articulated, ZERO skips ---
WORLD_DEMO = [("Konark Sun Temple","Odisha India",38),("Angkor Wat","Cambodia",None),("Notre-Dame Paris","France",None),
    ("Chichen Itza","Mexico",None),("Taj Mahal","India",None),("Parthenon Athens","Greece",None),
    ("Borobudur","Indonesia",None),("Brihadeeswarar Temple","India",None)]
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
fig=plt.figure(figsize=(18,9))
for idx,(nm,loc,sv) in enumerate(WORLD_DEMO):
    try: rm=tangible_search(f"{nm} {loc}".strip(), top_k=1)
    except Exception: rm=[]
    if len(rm):
        rec=heritage_master[heritage_master["name"]==rm.iloc[0]["name"]].iloc[0]; structure=rec["name"]; tradition=best_tradition(rec)
    else: structure=nm; tradition=classify_tradition(nm)[0][0]
    key=resolve_assembler(tradition)
    H,base,_=_HB(structure,key,sv); mesh=ASSEMBLERS[key](H, base, structure)
    safe="".join(ch if ch.isalnum() else "_" for ch in structure)[:40]; open(f"/kaggle/working/{safe}_world.obj","w").write(mesh.obj())
    V=np.array(mesh.V); F=np.array(mesh.F); tris=V[F]
    n=np.cross(tris[:,1]-tris[:,0],tris[:,2]-tris[:,0]); ln=np.linalg.norm(n,axis=1,keepdims=True); ln[ln==0]=1; n=n/ln
    light=np.array([0.4,0.5,0.78]); light=light/np.linalg.norm(light); inten=0.30+0.70*np.clip(n@light,0,1); cols=np.clip(np.array([0.74,0.55,0.37])[None,:]*inten[:,None],0,1)
    ax=fig.add_subplot(2,4,idx+1,projection="3d"); ax.add_collection3d(Poly3DCollection(tris,facecolors=cols,linewidths=0))
    mx=V.max(0); mn=V.min(0); ctr=(mx+mn)/2; rng=(mx-mn).max()/2
    ax.set_xlim(ctr[0]-rng,ctr[0]+rng); ax.set_ylim(ctr[1]-rng,ctr[1]+rng); ax.set_zlim(mn[2],mn[2]+2*rng)
    ax.set_box_aspect((1,1,1.4)); ax.set_axis_off(); ax.view_init(elev=15,azim=-58)
    ax.set_title(f"{structure[:24]}\n{key.split('(')[0].strip()}", fontsize=8)
    print(f"{structure[:32]:34s} -> {key:30s} {len(mesh.V):6d}v")
fig.suptitle("ONE MODEL, ANY STRUCTURE — articulated, every part, zero skips", fontsize=12)
plt.tight_layout(); plt.savefig("/kaggle/working/world_assembler_sheet.png", dpi=110, bbox_inches="tight"); plt.show()
print("\nExported per-structure .obj + world_assembler_sheet.png to /kaggle/working")


## v17 — CONSTRUCTION PACKAGE: CAD export + measured drawings + dimension schedule

The bridge from a culturally-correct 3D model toward something an **architect / construction
firm can actually open and develop**. For every structure this emits a full package:

- **CAD files** — `.obj` (universal), **`.dxf`** (opens directly in AutoCAD / Revit / Rhino),
  **`.glb`** (glTF 2.0 for web `<model-viewer>` and BIM viewers).
- **Measured drawing set** — front elevation, side elevation, top plan and a 3D view, with
  **dimension lines in metres**, a metric scale bar, a title block and the schedule.
- **Dimension schedule** — `.json` bill of dimensions (overall height, footprint W×D, the
  scale basis) plus the cited canon rules.

Everything is written to `/kaggle/working/packages/<structure>/` and zipped to
`/kaggle/working/heritageformer_construction_packages.zip` for one-click download.

> **Honest scope.** This is an *articulated design hypothesis* on cited canon — not a
> construction document. A buildable plan additionally requires measured site survey, a
> structural system (loads, foundations, materials), MEP and local code compliance, signed
> off by licensed professionals. This package is the *input* an architect develops into those
> drawings — the hard cultural/architectural first step, done and exportable.


In [ ]:
# ============================================================
# 40) CONSTRUCTION PACKAGE — CAD export (.obj/.dxf/.glb) + measured drawings
#     + dimension schedule, zipped for download. Reuses ASSEMBLERS / WORLD_CANON
#     / classify_tradition / tangible_search from the v16 world-assembler cell.
# ============================================================
import os, json, struct, shutil
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def export_dxf(mesh, path):
    """ASCII DXF, one 3DFACE per triangle — opens in AutoCAD / Revit / Rhino."""
    V=mesh.V; out=["0","SECTION","2","ENTITIES"]
    for (a,b,c) in mesh.F:
        (x1,y1,z1),(x2,y2,z2),(x3,y3,z3)=V[a],V[b],V[c]
        out+=["0","3DFACE","8","HeritageFormer",
              "10",f"{x1:.4f}","20",f"{y1:.4f}","30",f"{z1:.4f}",
              "11",f"{x2:.4f}","21",f"{y2:.4f}","31",f"{z2:.4f}",
              "12",f"{x3:.4f}","22",f"{y3:.4f}","32",f"{z3:.4f}",
              "13",f"{x3:.4f}","23",f"{y3:.4f}","33",f"{z3:.4f}"]
    out+=["0","ENDSEC","0","EOF"]; open(path,"w").write("\n".join(out))

def export_glb(mesh, path):
    """Binary glTF 2.0 (.glb): positions + per-vertex normals + indices + sandstone material."""
    V=np.array(mesh.V,dtype=np.float32); F=np.array(mesh.F,dtype=np.uint32)
    tris=V[F]; fn=np.cross(tris[:,1]-tris[:,0],tris[:,2]-tris[:,0])
    nrm=np.zeros_like(V); np.add.at(nrm,F[:,0],fn); np.add.at(nrm,F[:,1],fn); np.add.at(nrm,F[:,2],fn)
    ln=np.linalg.norm(nrm,axis=1,keepdims=True); ln[ln==0]=1; nrm=(nrm/ln).astype(np.float32)
    idx=F.flatten().astype(np.uint32); pad=lambda b: b+b"\x00"*((-len(b))%4)
    pos_b,nrm_b,idx_b=pad(V.tobytes()),pad(nrm.tobytes()),pad(idx.tobytes()); blob=pos_b+nrm_b+idx_b
    gltf={"asset":{"version":"2.0","generator":"HeritageFormer"},"scene":0,"scenes":[{"nodes":[0]}],
        "nodes":[{"mesh":0}],"meshes":[{"primitives":[{"attributes":{"POSITION":0,"NORMAL":1},"indices":2,"material":0}]}],
        "materials":[{"pbrMetallicRoughness":{"baseColorFactor":[0.74,0.55,0.37,1.0],"metallicFactor":0.0,"roughnessFactor":0.9},"name":"sandstone"}],
        "buffers":[{"byteLength":len(blob)}],
        "bufferViews":[{"buffer":0,"byteOffset":0,"byteLength":len(pos_b),"target":34962},
                       {"buffer":0,"byteOffset":len(pos_b),"byteLength":len(nrm_b),"target":34962},
                       {"buffer":0,"byteOffset":len(pos_b)+len(nrm_b),"byteLength":len(idx_b),"target":34963}],
        "accessors":[{"bufferView":0,"componentType":5126,"count":len(V),"type":"VEC3","min":V.min(0).tolist(),"max":V.max(0).tolist()},
                     {"bufferView":1,"componentType":5126,"count":len(V),"type":"VEC3"},
                     {"bufferView":2,"componentType":5125,"count":len(idx),"type":"SCALAR"}]}
    js=json.dumps(gltf).encode(); js=js+b" "*((-len(js))%4)
    glb=struct.pack("<III",0x46546C67,2,12+8+len(js)+8+len(blob))
    glb+=struct.pack("<II",len(js),0x4E4F534A)+js; glb+=struct.pack("<II",len(blob),0x004E4942)+blob
    open(path,"wb").write(glb)

def dimension_schedule(mesh, name, tradition, H, canon):
    V=np.array(mesh.V); dims=V.max(0)-V.min(0)
    return {"structure":name,"tradition":tradition,"governing_treatise":canon["treatise"],
            "overall_height_m":round(float(dims[2]),2),"footprint_width_m":round(float(dims[0]),2),
            "footprint_depth_m":round(float(dims[1]),2),"principal_height_basis_m":round(float(H),2),
            "canon_rules":canon["rules"],"mesh_vertices":len(mesh.V),"mesh_faces":len(mesh.F),
            "status":"DESIGN HYPOTHESIS — articulated massing on cited canon. NOT a construction "
                     "document. Requires measured survey, structural engineering, MEP and local "
                     "code compliance by licensed professionals before any build."}

def _dim_h(ax,x0,x1,y,txt):
    ax.annotate("",(x0,y),(x1,y),arrowprops=dict(arrowstyle="<->",color="#1f4e8c",lw=1.0))
    ax.text((x0+x1)/2,y-0.1,txt,ha="center",va="top",fontsize=7,color="#1f4e8c")
def _dim_v(ax,x,y0,y1,txt):
    ax.annotate("",(x,y0),(x,y1),arrowprops=dict(arrowstyle="<->",color="#1f4e8c",lw=1.0))
    ax.text(x-0.6,(y0+y1)/2,txt,ha="right",va="center",fontsize=7,color="#1f4e8c",rotation=90)
def _elevation(ax,V,F,plane,title):
    if plane=="front": P=V[:,[0,2]]; order=np.argsort(V[F][:,:,1].mean(1))
    elif plane=="side": P=V[:,[1,2]]; order=np.argsort(-V[F][:,:,0].mean(1))
    else: P=V[:,[0,1]]; order=np.argsort(V[F][:,:,2].mean(1))
    ax.add_collection(PolyCollection(P[F][order],facecolors="#d9c7a8",edgecolors="#7a6a4f",linewidths=0.15))
    mn=P.min(0); mx=P.max(0); pad=(mx-mn).max()*0.18
    ax.set_xlim(mn[0]-pad,mx[0]+pad); ax.set_ylim(mn[1]-pad*0.7,mx[1]+pad); ax.set_aspect("equal")
    ax.set_title(title,fontsize=8); ax.axis("off"); return mn,mx

def measured_drawing(mesh, name, tradition, sch, path):
    V=np.array(mesh.V); F=np.array(mesh.F); fig=plt.figure(figsize=(15,10))
    ax1=fig.add_subplot(2,3,1); mn,mx=_elevation(ax1,V,F,"front","FRONT ELEVATION")
    _dim_v(ax1,mn[0]-1.0,mn[1],mx[1],f"H {sch['overall_height_m']} m"); _dim_h(ax1,mn[0],mx[0],mn[1]-1.4,f"W {sch['footprint_width_m']} m")
    ax1.plot([mn[0],mx[0]],[mn[1],mn[1]],color="k",lw=0.8)
    ax2=fig.add_subplot(2,3,2); mn2,mx2=_elevation(ax2,V,F,"side","SIDE ELEVATION")
    _dim_h(ax2,mn2[0],mx2[0],mn2[1]-1.4,f"D {sch['footprint_depth_m']} m"); ax2.plot([mn2[0],mx2[0]],[mn2[1],mn2[1]],color="k",lw=0.8)
    ax3=fig.add_subplot(2,3,3); _elevation(ax3,V,F,"plan","TOP PLAN")
    tris=V[F]; n=np.cross(tris[:,1]-tris[:,0],tris[:,2]-tris[:,0]); l=np.linalg.norm(n,axis=1,keepdims=True); l[l==0]=1; n=n/l
    light=np.array([0.4,0.5,0.78]); light=light/np.linalg.norm(light); inten=0.3+0.7*np.clip(n@light,0,1)
    cols=np.clip(np.array([0.74,0.55,0.37])[None,:]*inten[:,None],0,1)
    ax4=fig.add_subplot(2,3,4,projection="3d"); ax4.add_collection3d(Poly3DCollection(tris,facecolors=cols,linewidths=0))
    c=(V.max(0)+V.min(0))/2; r=(V.max(0)-V.min(0)).max()/2
    ax4.set_xlim(c[0]-r,c[0]+r); ax4.set_ylim(c[1]-r,c[1]+r); ax4.set_zlim(V.min(0)[2],V.min(0)[2]+2*r)
    ax4.set_box_aspect((1,1,1.4)); ax4.set_axis_off(); ax4.view_init(elev=15,azim=-58); ax4.set_title("3D VIEW",fontsize=8)
    ax5=fig.add_subplot(2,3,5); ax5.axis("off")
    rows=[f"STRUCTURE : {name}",f"TRADITION : {tradition}",f"CANON     : {sch['governing_treatise']}","",
          f"Overall height : {sch['overall_height_m']} m",f"Footprint WxD  : {sch['footprint_width_m']} x {sch['footprint_depth_m']} m",
          f"Mesh           : {sch['mesh_vertices']} v / {sch['mesh_faces']} f",f"Scale-basis H  : {sch['principal_height_basis_m']} m","","CANON RULES:"]
    rows+=[f"  - {r}" for r in sch["canon_rules"]]
    ax5.text(0,1,"\n".join(rows),va="top",ha="left",fontsize=8,family="monospace")
    ax6=fig.add_subplot(2,3,6); ax6.axis("off")
    for i in range(10): ax6.add_patch(plt.Rectangle((i,0.8),1,0.15,facecolor="k" if i%2 else "w",edgecolor="k"))
    ax6.text(0,0.62,"0",fontsize=7); ax6.text(10,0.62,"10 m",fontsize=7); ax6.set_xlim(-1,12); ax6.set_ylim(0,1.5)
    ax6.text(0,0.34,"DESIGN HYPOTHESIS — articulated massing on cited canon.",fontsize=8,color="#8c1f1f")
    ax6.text(0,0.19,"NOT a construction document. Needs survey + structural",fontsize=8,color="#8c1f1f")
    ax6.text(0,0.04,"engineering + MEP + code sign-off by licensed pros.",fontsize=8,color="#8c1f1f")
    fig.suptitle(f"HERITAGEFORMER — MEASURED DRAWING SET   |   {name}   |   {tradition}",fontsize=12,y=0.99)
    plt.tight_layout(rect=[0,0,1,0.97]); plt.savefig(path,dpi=120,bbox_inches="tight"); plt.close()

def construction_package(name, location="", survived_m=None, outdir="/kaggle/working/packages"):
    try: rm=tangible_search(f"{name} {location}".strip(), top_k=1)
    except Exception: rm=[]
    if len(rm):
        rec=heritage_master[heritage_master["name"]==rm.iloc[0]["name"]].iloc[0]; structure=rec["name"]; tradition=best_tradition(rec)
    else: structure=name; tradition=classify_tradition(name)[0][0]
    key=resolve_assembler(tradition); canon=WORLD_CANON[key]
    H,base,_=_HB(structure,key,survived_m)
    mesh=ASSEMBLERS[key](H,base,structure)
    safe="".join(ch if ch.isalnum() else "_" for ch in structure)[:40]; d=f"{outdir}/{safe}"; os.makedirs(d,exist_ok=True)
    open(f"{d}/{safe}.obj","w").write(mesh.obj()); export_dxf(mesh,f"{d}/{safe}.dxf"); export_glb(mesh,f"{d}/{safe}.glb")
    sch=dimension_schedule(mesh,structure,tradition,H,canon); json.dump(sch,open(f"{d}/{safe}_schedule.json","w"),indent=2)
    measured_drawing(mesh,structure,tradition,sch,f"{d}/{safe}_drawing.png")
    print(f"  {structure[:30]:32s} {tradition.split('(')[0].strip():22s} H {sch['overall_height_m']:6.1f}m  obj+dxf+glb+drawing+schedule")
    return d

# --- build a full package for every demo structure, then zip for download ---
shutil.rmtree("/kaggle/working/packages", ignore_errors=True)
print("Building construction packages (CAD + drawings + schedules):")
for nm,loc,sv in WORLD_DEMO: construction_package(nm,loc,sv)
zip_path=shutil.make_archive("/kaggle/working/heritageformer_construction_packages","zip","/kaggle/working/packages")
print("\nZIP ready:", zip_path)

# preview two measured-drawing sheets inline
for nm in ("Sun Temple, Kon", "Taj Mahal"):
    import glob
    hits=glob.glob(f"/kaggle/working/packages/*/*_drawing.png")
    pick=[h for h in hits if nm.split(",")[0].replace(" ","_")[:8] in h]
    if pick:
        fig=plt.figure(figsize=(14,9)); plt.imshow(plt.imread(pick[0])); plt.axis("off"); plt.show()


## v20 — HeritageFormer Restoration Corpus (HFRC) builder

Generates a **provenance-tracked, reproducible** research corpus from data already in this
notebook — not invented. Every factual row carries `source`, `source_url`,
`verification_status` and `confidence`, so nothing uncertain is presented as verified.

| File | How it's built | Provenance |
|---|---|---|
| `metadata/monuments.csv` | from `heritage_master` (UNESCO) | **real** records |
| `reasoning/analogy_graph.csv` | same-tradition BGE-M3 nearest neighbours | **machine_generated** (confidence = similarity) |
| `architecture/components.csv` | `component_ontology` + world components | hand-authored, `expert_review_needed` |
| `architecture/style_rules.csv` | `CULTURAL_GRAMMAR` proportion rules | cited treatise, `needs_expert_validation` |
| `metadata/materials.csv` | `CULTURAL_GRAMMAR` materials | cited treatise |
| `metadata/restoration_events.csv` | `KNOWN_MISSING` | ASI/UNESCO, `needs_verification` |
| `reasoning/evaluation_queries.csv` | `EVAL_HARD` benchmark | held-out queries |
| `geometry/monument_dimensions.csv` | seeded published figures + template | `documented_unverified` |
| `geometry/monument_parameters.csv` | seeded famous facts (Konark) + template | `documented` / blank |

Output: `/kaggle/working/heritageformer-restoration-corpus/` + a zip you can publish as a
Kaggle Dataset. The `drawings/` and `scans/` folders are created empty — drop in your
orthographic elevations and OpenHeritage3D/CyArk meshes to extend the corpus.

> This is the honest version: the grounded tables are generated from cited sources; the
> dimension/parameter tables are *seeded* with what is genuinely public and **flagged for
> verification** otherwise. It is designed to be cited, not to look complete.


In [ ]:
# ============================================================
# 41) HeritageFormer Restoration Corpus (HFRC) builder — provenance-tracked
# ============================================================
import os, json
import pandas as pd
ROOT="/kaggle/working/heritageformer-restoration-corpus"
for d in ["metadata","geometry","architecture","reasoning","drawings","scans/obj","scans/glb"]:
    os.makedirs(f"{ROOT}/{d}", exist_ok=True)
def _save(df, rel):
    df.to_csv(f"{ROOT}/{rel}", index=False); print(f"  {rel:45s} {len(df):6d} rows")
built={}

# 1) monuments.csv — REAL, from the UNESCO/Heritage3D master table -------------
try:
    mon = heritage_master[heritage_master["kind"]=="tangible"].reset_index(drop=True).copy()
    mon.insert(0,"id",[f"HF{i+1:05d}" for i in range(len(mon))])
    mon["tradition"]=[best_tradition(r) for _,r in mon.iterrows()]
    cols=["id","name","country","region","latitude","longitude","tradition","type","period","source","kind"]
    out=mon[[c for c in cols if c in mon.columns]].copy()
    out["verification_status"]="source_record"   # straight from the public dataset
    _save(out,"metadata/monuments.csv"); built["monuments"]=len(out)
    NAME2ID=dict(zip(mon["name"],mon["id"]))
except Exception as e: print("  monuments.csv skipped:",e); NAME2ID={}

# 2) analogy_graph.csv — same-tradition nearest neighbours (GROUNDED) ----------
try:
    D,I = tangible_index.search(tangible_emb, 7)
    rows=[]
    for i in range(len(tangible_master)):
        src=tangible_master.iloc[i]; st=best_tradition(src)
        for j,score in zip(I[i][1:], D[i][1:]):
            if j<0 or j==i: continue
            dst=tangible_master.iloc[j]; dt=best_tradition(dst)
            if dt==st and float(score)>=0.45:
                rows.append({"target_site":src["name"],"reference_site":dst["name"],
                             "reason":f"same architectural tradition ({st})",
                             "confidence":round(float(score),3),
                             "source":"HeritageFormer BGE-M3 retrieval",
                             "verification_status":"machine_generated"})
    _save(pd.DataFrame(rows),"reasoning/analogy_graph.csv"); built["analogy_graph"]=len(rows)
except Exception as e: print("  analogy_graph.csv skipped:",e)

# 3) components.csv (hand-authored ontology + world component map) -------------
try:
    rows=[]
    for arch,parts in component_ontology.items():
        for p in parts: rows.append({"component":p,"archetype":arch,"scope":"archetype",
            "source":"hand-authored component ontology","verification_status":"expert_review_needed"})
    for trad,parts in TRADITION_COMPONENTS.items():
        for p in parts: rows.append({"component":p,"archetype":trad,"scope":"tradition",
            "source":"hand-authored world components","verification_status":"expert_review_needed"})
    _save(pd.DataFrame(rows),"architecture/components.csv"); built["components"]=len(rows)
except Exception as e: print("  components.csv skipped:",e)

# 4) style_rules.csv + materials.csv (CULTURAL_GRAMMAR; cite treatise) ---------
try:
    sr=[]; mat=[]
    for trad,g in CULTURAL_GRAMMAR.items():
        tr=g.get("treatise","")
        for r in g.get("proportion_rules",[]):
            sr.append({"style":trad,"rule":r,"treatise":tr,"source":tr or "hand-authored",
                       "verification_status":"needs_expert_validation"})
        for mt in g.get("materials",[]):
            mat.append({"style":trad,"material":mt,"source":tr or "hand-authored",
                        "verification_status":"needs_expert_validation"})
    _save(pd.DataFrame(sr),"architecture/style_rules.csv"); built["style_rules"]=len(sr)
    _save(pd.DataFrame(mat),"metadata/materials.csv"); built["materials"]=len(mat)
except Exception as e: print("  style_rules/materials skipped:",e)

# 5) restoration_events.csv (documented losses; curated) ----------------------
try:
    rows=[]
    for site,losses in KNOWN_MISSING.items():
        for comp,note in losses.items():
            rows.append({"site":site,"component":comp,"event":note,
                         "source":"ASI / UNESCO (curated)","verification_status":"needs_verification"})
    _save(pd.DataFrame(rows),"metadata/restoration_events.csv"); built["restoration_events"]=len(rows)
except Exception as e: print("  restoration_events.csv skipped:",e)

# 6) evaluation_queries.csv (the held-out benchmark) --------------------------
try:
    rows=[{"query":q,"expected_site":gold[0],"accepted_aliases":";".join(gold)} for q,gold in EVAL_HARD]
    _save(pd.DataFrame(rows),"reasoning/evaluation_queries.csv"); built["evaluation_queries"]=len(rows)
except Exception as e: print("  evaluation_queries.csv skipped:",e)

# 7) monument_dimensions.csv (seeded published figures; FLAGGED) --------------
try:
    rows=[]
    for k,v in MONUMENT_DIMENSIONS.items():
        rows.append({"name_key":k,"height_m":v["h"],"base_width_m":v["base"],"base_depth_m":v["base"],
                     "source":v["src"],"source_url":"","verification_status":"documented_unverified","confidence":0.8})
    _save(pd.DataFrame(rows),"geometry/monument_dimensions.csv"); built["monument_dimensions"]=len(rows)
except Exception as e: print("  monument_dimensions.csv skipped:",e)

# 8) monument_parameters.csv (seed only well-attested facts; rest blank) ------
SEED_PARAMS=[  # textbook / ASI facts — still flagged for primary-source check
    ("Sun Temple, Konârak","wheel_count",24,"count","ASI",0.99),
    ("Sun Temple, Konârak","horse_count",7,"count","ASI",0.99),
    ("Sun Temple, Konârak","deul_height_m",70,"m","ASI (collapsed)",0.8),
    ("Taj Mahal","minaret_count",4,"count","ASI",0.99),
    ("Buddhist Monuments at Sanchi","torana_count",4,"count","ASI",0.95),
]
try:
    rows=[{"site":s,"parameter":p,"value":val,"unit":u,"source":src,"confidence":c,
           "verification_status":"documented"} for (s,p,val,u,src,c) in SEED_PARAMS]
    _save(pd.DataFrame(rows),"geometry/monument_parameters.csv"); built["monument_parameters"]=len(rows)
except Exception as e: print("  monument_parameters.csv skipped:",e)

# 9) README + schema + provenance registry -----------------------------------
readme=f"""# HeritageFormer Restoration Corpus (HFRC) v0.1

A provenance-tracked corpus for **AI-assisted, culturally-governed heritage reconstruction**.
Generated reproducibly from the HeritageFormer pipeline — not hand-curated by guessing.

## Provenance policy
Every factual row carries `source`, `verification_status`, and (where applicable) `confidence`.
- `source_record`         : taken verbatim from a public dataset (UNESCO/Heritage3D).
- `machine_generated`     : derived by the retrieval engine (BGE-M3); confidence = similarity.
- `documented_unverified` : a published figure cited but not yet checked against the primary source.
- `documented`            : well-attested fact (still verify against ASI/UNESCO primary docs).
- `needs_verification` / `needs_expert_validation` / `expert_review_needed`: NOT yet validated.

**Nothing in this corpus should be treated as verified unless its row says so.**

## Contents
{json.dumps(built, indent=2)}

## To extend
- Drop orthographic elevation drawings into `drawings/<monument>/`.
- Drop OpenHeritage3D / CyArk meshes into `scans/obj|glb/` (these are the validation ground truth).
- Add rows to `geometry/monument_dimensions.csv` / `monument_parameters.csv` with a real `source_url`
  and set `verification_status=verified` only after checking the primary source.
"""
open(f"{ROOT}/README.md","w",encoding="utf-8").write(readme)
json.dump({"version":"0.1","tables":built,
           "provenance_fields":["source","source_url","verification_status","confidence","last_checked"]},
          open(f"{ROOT}/schema.json","w"), indent=2)

import shutil
zip_path=shutil.make_archive("/kaggle/working/HFRC_v0.1","zip",ROOT)
print("\nHFRC built. Tables:",built)
print("Download:",zip_path)
print("Publish: Kaggle -> Create Dataset -> upload HFRC_v0.1.zip (or the folder) -> attach back to this notebook.")


## v24 — RUIN RECONSTRUCTION: broken pieces of history, restored

The project's founding problem: a structure survives only in part — Konark's deul collapsed,
a hill fort lost a bastion, a Greek temple lost its roof. This stage splits a culturally-correct
model into the **surviving fabric** and the **historically-lost parts**, then shows them together:

- **As found** — the surviving stone (weathered grey) + collapse **rubble** at the loss site.
- **Reconstructed** — the same survivors, with the lost element rebuilt and **highlighted**
  (warm tone) so it is never confused with original fabric.

Damage profiles are per-tradition (temple → lost tower, fort/castle → collapsed corner bastion,
classical → lost roof, pyramid → eroded summit). Each reconstruction exports `*_survived.obj`
and `*_restored.obj` so the reconstructed pieces are a separate, reviewable layer — an honest
restoration proposal, not a repaint of the original.


In [ ]:
# ============================================================
# 41) RUIN RECONSTRUCTION — survived fabric vs reconstructed lost parts
# ============================================================
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

def split_ruin(mesh, zone):
    V=np.array(mesh.V); F=np.array(mesh.F)
    if len(F)==0: return Mesh(), Mesh()
    cent=V[F].mean(1); lost=np.array([bool(zone(c[0],c[1],c[2])) for c in cent])
    def submesh(mask):
        m=Mesh(); remap={}
        for fi in np.where(mask)[0]:
            tri=[]
            for vi in F[fi]:
                vi=int(vi)
                if vi not in remap: remap[vi]=len(m.V); m.V.append(mesh.V[vi])
                tri.append(remap[vi])
            m.F.append(tuple(tri))
        return m
    return submesh(~lost), submesh(lost)

def make_rubble(cx, cy, z0, spread, n=46, block=0.6):
    m=Mesh()
    for i in range(n):
        a=i*2.39996; r=spread*(0.15+0.85*((i*13%17)/17.0)); x=cx+r*np.cos(a); y=cy+r*np.sin(a)
        s=block*(0.4+0.6*((i*7%5)/5.0)); h=s*(0.5+0.5*((i*3%4)/4.0)); zb=z0+0.02*(i*11%6)
        add_box(m, x-s, x+s, y-s, y+s, zb, zb+h)
    return m

def damage_for(tradition, name, V):
    mn=V.min(0); mx=V.max(0); H=mx[2]-mn[2]; nm=(name or "").lower(); t=tradition.lower()
    if "konar" in nm or "konâr" in nm: return lambda x,y,z: (z > mn[2]+0.34*H) and (x < (mn[0]+mx[0])*0.5)
    if "fort" in t or "castle" in t or "zimbabwe" in t: return lambda x,y,z: (x > mx[0]*0.42) and (y > mx[1]*0.42)
    if "classical" in t or "greco" in t or "achaemenid" in t or "umayyad" in t: return lambda x,y,z: z > mn[2]+0.62*H
    if any(k in t for k in ("maya","aztec","ziggurat","mesopotam","nubian")): return lambda x,y,z: z > mn[2]+0.70*H
    if any(k in t for k in ("stupa","buddhist","sinhal","burm","javanese")): return lambda x,y,z: z > mn[2]+0.62*H
    return lambda x,y,z: z > mn[2]+0.55*H

def reconstruct_ruin(mesh, tradition, name="", ruin=None):
    V=np.array(mesh.V); zone=ruin or damage_for(tradition, name, V)
    survived, restored = split_ruin(mesh, zone); Rv=np.array(restored.V); rubble=Mesh()
    if len(Rv):
        c=Rv.mean(0); span=(V.max(0)-V.min(0)).max()
        rubble=make_rubble(c[0], c[1], float(V.min(0)[2]), spread=span*0.12, n=46, block=span*0.018)
    return survived, restored, rubble

def _rshade(tris, base, jitter=0.0):
    n=np.cross(tris[:,1]-tris[:,0],tris[:,2]-tris[:,0]); ln=np.linalg.norm(n,axis=1,keepdims=True); ln[ln==0]=1; n=n/ln
    L=np.array([0.4,0.5,0.78]); L=L/np.linalg.norm(L); inten=0.34+0.66*np.clip(n@L,0,1)
    cols=np.clip(np.array(base)[None,:]*inten[:,None],0,1)
    if jitter:
        cnt=tris.mean(1); jj=(np.sin(cnt[:,0]*0.7)+np.sin(cnt[:,2]*0.9))*0.5; cols=np.clip(cols*(1+jitter*jj[:,None]),0,1)
    return cols

def render_reconstruction(survived, restored, rubble, title, path):
    allV=np.array(survived.V+restored.V+rubble.V); ctr=(allV.max(0)+allV.min(0))/2; rng=(allV.max(0)-allV.min(0)).max()/2
    def setup(ax):
        ax.set_xlim(ctr[0]-rng,ctr[0]+rng); ax.set_ylim(ctr[1]-rng,ctr[1]+rng); ax.set_zlim(allV.min(0)[2],allV.min(0)[2]+2*rng)
        ax.set_box_aspect((1,1,1.4)); ax.set_axis_off(); ax.view_init(elev=15,azim=-58)
    def add(ax, m, base, jit=0.0):
        if len(m.F): tris=np.array(m.V)[np.array(m.F)]; ax.add_collection3d(Poly3DCollection(tris,facecolors=_rshade(tris,base,jit),linewidths=0))
    fig=plt.figure(figsize=(15,7))
    ax1=fig.add_subplot(1,2,1,projection="3d"); setup(ax1); add(ax1,survived,(0.60,0.55,0.50),0.12); add(ax1,rubble,(0.42,0.38,0.34))
    ax1.set_title("AS FOUND  (surviving fabric + collapse rubble)",fontsize=9)
    ax2=fig.add_subplot(1,2,2,projection="3d"); setup(ax2); add(ax2,survived,(0.60,0.55,0.50),0.12); add(ax2,restored,(0.82,0.56,0.33))
    ax2.set_title("RECONSTRUCTED  (survived = stone, restored = highlighted)",fontsize=9)
    fig.suptitle(title,fontsize=12); plt.tight_layout(); plt.savefig(path,dpi=120,bbox_inches="tight"); plt.show()

def reconstruct_broken(name, location="", survived_m=None):
    try: rm=tangible_search(f"{name} {location}".strip(), top_k=1)
    except Exception: rm=[]
    if len(rm):
        rec=heritage_master[heritage_master["name"]==rm.iloc[0]["name"]].iloc[0]; structure=rec["name"]; tradition=best_tradition(rec)
    else: structure=name; tradition=classify_tradition(name)[0][0]
    key=resolve_assembler(tradition); canon=WORLD_CANON[key]; H,base,basis=_HB(structure,key,survived_m)
    mesh=ASSEMBLERS[key](H,base,structure); survived,restored,rubble=reconstruct_ruin(mesh,key,structure)
    safe="".join(ch if ch.isalnum() else "_" for ch in structure)[:40]
    open(f"/kaggle/working/{safe}_survived.obj","w").write(survived.obj())
    open(f"/kaggle/working/{safe}_restored.obj","w").write(restored.obj())
    pct=100*len(restored.F)/max(1,len(mesh.F))
    print("="*72); print("RUIN RECONSTRUCTION:", structure); print("="*72)
    print(f"Tradition : {tradition}  ({canon['form']})")
    print(f"Surviving fabric : {len(survived.F)} faces   Reconstructed (lost) : {len(restored.F)} faces ({pct:.0f}%)")
    print(f"Height basis      : {H:.1f} m  <- {basis}")
    print(f"Exported {safe}_survived.obj + {safe}_restored.obj to /kaggle/working")
    render_reconstruction(survived,restored,rubble,f"{structure} — as-found vs reconstructed\n{tradition}",
                          f"/kaggle/working/{safe}_reconstruction.png")
    return survived, restored

# --- ruined structures restored (Konark deul, fort bastion, lost roof, eroded pyramid) ---
RUIN_DEMO=[("Konark Sun Temple","Odisha India",38),
           ("Parthenon Athens","Greece",None),("Chichen Itza","Mexico",None)]  # Raigad: dedicated v25 cell
for nm,loc,sv in RUIN_DEMO: reconstruct_broken(nm,loc,sv)
print("\nEach reconstruction exports a separate *_survived.obj and *_restored.obj layer.")


## v25 - RAIGAD FORT: full parts track + documented-plan model + scan-grounded rebuild

`RAIGAD_PARTS` is a **registry of every documented part of the fort** (gates, bastions, the Bale Killa
citadel, temples, memorials, the 84-tank/11-lake water system incl. Bara Tanki, Kushavarta, Kolimb,
Gangasagar & Hatti, the Darukhana, Hatti Khana, ~300 stone houses, Bhavani Tok...). `raigad_parts_report()`
prints it with each part's **category / zone / condition / whether it is modelled / source** - **44 of 45
parts are modelled** (only Jijabai's samadhi is off-site at Pachad). Sources: [Wikipedia](https://en.wikipedia.org/wiki/Raigad_Fort),
[raigad.gov.in](https://raigad.gov.in/en/tourist-place/forts-in-raigad/),
[raigadropeway.com](https://raigadropeway.com/raigad.html),
[historicnation.in](https://www.historicnation.in/raigad-fort-information/),
[Reminiscing History](https://www.reminiscinghistory.com/product-page/bazarpeth-raigad-fort-maharashtra),
[SIES-OIOP](https://www.siesoiop.in/ancient-temples-of-raigad/), ASI.

**As-found** shows what stands today (gates, walls, bastions, temples, Samadhi, tanks, and the bare **base
pillars** of the wooden palace & durbar). **Reconstructed** raises the **lost timber** on top, highlighted.
Geometry is **documented-plan schematic massing** (`needs_survey`), not a survey.

**Scan-grounded rebuild (end-to-end):** `reconstruct_from_survey()` scales + aligns the canon-reconstructed
lost timber **onto a survey mesh of the real surviving fabric**. Upload a photogrammetry/laser-scan `.obj`
(e.g. a [Sketchfab Raigad scan](https://sketchfab.com/3d-models/raigad-bazarpeth-f4549014ca6c4e22968e8eabfcfc955e)
or CyArk) to `/kaggle/input/raigad-scan/survey.obj` and the cell rebuilds on the **real remains**; with no
scan it falls back to a demo. A single photo can't give metric 3D - overlapping multi-view coverage is required.

In [ ]:
# ============================================================
# 42) RAIGAD FORT - full parts track + documented-plan model + scan-grounded rebuild
#     RAIGAD_PARTS = registry of every documented part (-> raigad_parts_report()).
#     sources in RAIGAD_SOURCES; real survey meshes in RAIGAD_SURVEY_MESHES.
# ============================================================
RAIGAD_EVIDENCE = [
 ("Plateau / area",     "Hill-top plateau ~820 m ASL, ~1,200 acres (largest fort in Asia); points Hirakani(W), Takmak(N), Bhavani(E)", "Wikipedia; raigad.gov.in", "documented"),
 ("Materials / scale",  "Basalt, granite, lime mortar; Hiroji Indulkar built ~300 stone houses, offices, a garrison, gardens & the market (1656-74)", "Wikipedia; maharashtriantraditions", "documented"),
 ("Coronation",         "Rajyabhishek of Chhatrapati Shivaji Maharaj, 6 June 1674, in the Rajsabha", "historical record", "documented"),
 ("Architect",          "Hiroji Indulkar - name engraved on the first step / east wall of the Jagdishwar temple (Shaka 1596)", "Jagdishwar inscription; siesoiop.in", "documented"),
 ("Fortification",      "Curtain wall (tatbandi) + bastions (incl. Khubladha Buruj) ringing the cliff; ~1,737 steps to the top", "Wikipedia; site survey", "documented"),
 ("Maha Darwaja",       "Great Gate (W ascent), two bastions ~20-21 m (65-70 ft) high; top rises 180 m above it", "Wikipedia; maharashtriantraditions", "documented"),
 ("Other gates",        "Chit Darwaja & Jit Darwaja on the ascent; Palkhi Darwaja (N, royal); Mena Darwaja (S, women)", "Wikipedia; maharashtriantraditions", "documented"),
 ("Nagarkhana Darwaja", "Main doorway to the king's court; acoustically designed (doorway -> throne)", "Wikipedia", "documented"),
 ("Rajsabha (Durbar)",  "Court hall; throne (Meghdambari/Singhasan) oriented to face EAST; only column bases remain today", "Wikipedia; ASI", "documented"),
 ("Raj Bhavan (palace)","Royal palace built of WOOD; only the stone base pillars remain today", "Wikipedia", "documented"),
 ("Ranivasa",           "Queen's quarters - six windowless chambers, each with a private restroom", "Wikipedia", "documented"),
 ("Ashtapradhan Wada",  "Quarters of the eight ministers, within the Bale Killa", "raigad.gov.in", "documented"),
 ("Granaries",          "Row of three deep, dark chambers right of the Palkhi Darwaja", "Wikipedia", "documented"),
 ("Bazar Peth",         "Market ~850 ft long: 22 stone shops EACH side of a ~40 ft central path; wood columns + terracotta roofs on high plinths (roofs lost)", "Reminiscing History; raigad.gov.in", "documented"),
 ("Holicha Mal",        "Open ground; the Bazar Peth lies to its north", "raigad.gov.in", "documented"),
 ("Jagdishwar Temple",  "Hindu-Islamic hybrid: cubic sanctum with a MUGHAL-STYLE DOME; monolithic seated Nandi facing the shrine; survives intact", "Wikipedia; siesoiop.in", "documented"),
 ("Deepmala",           "Stone lamp-tower (deepstambha) in the temple court", "site survey", "documented_uncertain"),
 ("Samadhi",            "Memorial of Shivaji (d. 3 Apr 1680): low OCTAGONAL platform, later raised with an octagonal meghdambari canopy; opposite the temple's east entrance", "Wikipedia; RTF case study", "documented"),
 ("Waghya memorial",    "Memorial of Shivaji's dog Waghya, beside the Samadhi", "Wikipedia", "documented"),
 ("Hirakani Buruj",     "Bastion built over a steep western cliff (the Hirkani legend)", "Wikipedia", "documented_legend"),
 ("Takmak Tok",         "Northern execution cliff, now fenced off", "Wikipedia", "documented"),
 ("Watchtowers",        "Three watchtowers originally; only two survive (the third destroyed by bombardment)", "Wikipedia", "documented"),
 ("Gangasagar Lake",    "Artificial reservoir, said to use Ganga water at the coronation", "Wikipedia", "documented"),
 ("Hatti Talav",        "Elephant Lake, a second reservoir on the plateau", "Wikipedia", "documented"),
 ("Destruction",        "Bombarded by the British in 1818 (Third Anglo-Maratha War); timber palace burnt, stone survives", "Wikipedia", "documented"),
 ("UNESCO",             "Component of the serial 'Maratha Military Landscapes of India' nomination", "UNESCO nomination", "verify"),
 ("Dimensions",         "Exact heights/footprints are SCHEMATIC relative massing from the documented plan", "-", "needs_survey"),
]
RAIGAD_SOURCES = [
 "https://en.wikipedia.org/wiki/Raigad_Fort",
 "https://raigad.gov.in/en/tourist-place/forts-in-raigad/",
 "https://www.reminiscinghistory.com/product-page/bazarpeth-raigad-fort-maharashtra",
 "https://www.siesoiop.in/ancient-temples-of-raigad/",
 "Archaeological Survey of India (ASI) - Raigad, Monument of National Importance",
]
# Real photogrammetry/3D meshes of the surviving fabric (feed into reconstruct_from_survey via load_mesh_obj):
RAIGAD_SURVEY_MESHES = [
 "https://sketchfab.com/3d-models/raigad-free-model-992c2beb1952418f90ec70285c829b57",
 "https://sketchfab.com/3d-models/nagarkhana-raigad-a0c9f863de3140f8a58f2845fd726687",
 "https://sketchfab.com/3d-models/raigad-bazarpeth-f4549014ca6c4e22968e8eabfcfc955e",
 "https://sketchfab.com/3d-models/chatrapati-shivaji-maharaj-samadhi-raigad-1c9a78e0d2054dca970cecdbe2b2200d",
 "CyArk / OpenHeritage3D (openheritage3d.org) - for institutional laser-scan ground truth",
]

RAIGAD_PARTS = [
 ("Fortification wall (tatbandi)","Defense","perimeter","survives",True,"Wikipedia; ASI"),
 ("Maha Darwaja (Gomukhi)","Gate","W","survives",True,"Wikipedia"),
 ("Chit Darwaja","Gate","W ascent","survives",True,"maharashtriantraditions"),
 ("Nane Darwaja","Gate","W ascent","survives",True,"raigadropeway"),
 ("Jit Darwaja","Gate","W ascent","survives",True,"maharashtriantraditions"),
 ("Mena Darwaja","Gate","S","survives",True,"Wikipedia"),
 ("Palkhi Darwaja","Gate","N","survives",True,"Wikipedia"),
 ("Wagh Darwaja (escape)","Gate","NE","survives",True,"raigad.gov.in"),
 ("Nagarkhana Darwaja","Gate","centre","survives",True,"Wikipedia"),
 ("Hirakani Buruj","Bastion","W cliff","survives",True,"Wikipedia"),
 ("Khubladha Buruj","Bastion","N","survives",True,"historicnation"),
 ("Bhavani Buruj","Bastion","E","survives",True,"raigadropeway"),
 ("Watchtowers (2 of 3)","Bastion","perimeter","survives (1 lost)",True,"Wikipedia"),
 ("Takmak Tok","Cliff point","N","survives (fenced)",True,"Wikipedia"),
 ("Bhavani Tok","Cliff point","E","survives",True,"raigadropeway"),
 ("Bale Killa (citadel)","Royal","W-centre","survives (walls)",True,"raigad.gov.in"),
 ("Raj Bhavan (palace)","Royal","W-centre","base pillars only",True,"Wikipedia"),
 ("Royal Bath + 2 tanks","Royal","W-centre","survives",True,"historicnation"),
 ("Khalbat-khana (cellar)","Royal","W-centre","survives (sunken)",True,"historicnation"),
 ("Rajsabha (Durbar)","Royal","W-centre","base pillars only",True,"Wikipedia; ASI"),
 ("Meghdambari / Singhasan","Royal","W-centre","lost -> reconstructed",True,"Wikipedia"),
 ("Ranivasa (6 chambers)","Royal","W-centre","survives (windowless)",True,"Wikipedia"),
 ("Ashtapradhan Wada","Royal","W-centre","ruined -> recon",True,"raigad.gov.in"),
 ("Other wadas","Royal","various","ruined",True,"raigadropeway"),
 ("Granaries (3 chambers)","Utility","W-centre","survives",True,"Wikipedia"),
 ("Bazar Peth (22/side)","Civic","centre","survives (plinth/walls)",True,"Reminiscing History"),
 ("Holicha Mal (open ground)","Civic","centre","survives",True,"raigad.gov.in"),
 ("~300 stone houses","Civic","various","ruined (representative)",True,"maharashtriantraditions"),
 ("Jagdishwar Mandir","Temple","E","survives",True,"Wikipedia; SIES-OIOP"),
 ("Wadeshwar Mandir","Temple","centre","survives",True,"raigadropeway"),
 ("Shirkai Devi Temple","Temple","centre","survives",True,"raigadropeway"),
 ("Bhavani Temple","Temple","E","survives",True,"raigadropeway"),
 ("Deepmala (lamp tower)","Temple","E","survives",True,"site survey"),
 ("Shivaji Samadhi (octagonal)","Memorial","E","survives",True,"Wikipedia; RTF"),
 ("Waghya memorial","Memorial","E","survives (modern)",True,"Wikipedia"),
 ("Hiroji Indulkar Samadhi","Memorial","E","survives",True,"Wikipedia"),
 ("Jijabai Samadhi","Memorial","off-site (Pachad)","survives",False,"Wikipedia"),
 ("Gangasagar Lake","Water","centre","survives",True,"Wikipedia"),
 ("Hatti Talav (Elephant Lake)","Water","centre","survives",True,"Wikipedia"),
 ("Kushavarta Talav","Water","centre","survives",True,"raigadropeway"),
 ("Kolimb Talav","Water","centre","survives",True,"raigadropeway"),
 ("Bara Tanki (12 tanks)","Water","N","ruined",True,"historicnation"),
 ("Darukhana (ammunition)","Military","centre","survives",True,"raigad.gov.in"),
 ("Hatti Khana (stable)","Utility","centre","ruined",True,"raigadropeway"),
 ("Stambha (twin pillars)","Misc","N","survives",True,"site survey"),
]

def raigad_parts_report():
    print("="*104); print("RAIGAD FORT - PARTS TRACK  (every documented part -> condition -> modelled?)"); print("="*104)
    print("%-30s %-12s %-9s %-22s %-8s %s" % ("PART","CATEGORY","ZONE","CONDITION","MODELLED","SOURCE")); print("-"*104)
    for nm,cat,zone,cond,mod,src in RAIGAD_PARTS:
        print("%-30s %-12s %-9s %-22s %-8s %s" % (nm,cat,zone,cond,"yes" if mod else "OFF-SITE",src))
    nmod=sum(1 for p in RAIGAD_PARTS if p[4]); print("-"*104)
    print("%d parts tracked; %d modelled on-fort (%.0f%%); %d off-site/not-modelled."
          % (len(RAIGAD_PARTS), nmod, 100*nmod/len(RAIGAD_PARTS), len(RAIGAD_PARTS)-nmod))


def _rg_crenX(m, x0, x1, y0, y1, z, mw, gap, h):           # merlons stepping in +X
    x=x0
    while x < x1-1e-6: add_box(m, x, min(x+mw,x1), y0, y1, z, z+h); x+=mw+gap

def _rg_crenY(m, x0, x1, y0, y1, z, mw, gap, h):           # merlons stepping in +Y
    y=y0
    while y < y1-1e-6: add_box(m, x0, x1, y, min(y+mw,y1), z, z+h); y+=mw+gap

def _rg_shikhara(bw, H, n=8):
    """Curvilinear temple tower from stacked frustums + amalaka + kalasha (no gen_rekha dep)."""
    m=Mesh(); z=0.0
    for i in range(n):
        t0=i/n; t1=(i+1)/n
        w0=bw*(1-0.55*t0**2.1); w1=bw*(1-0.55*t1**2.1); h=H/n
        _merge(m, gen_frustum(w0,w0,w1,w1,h), 0,0,z); z+=h
    _merge(m, gen_frustum(bw*0.5,bw*0.5,bw*0.34,bw*0.34,bw*0.16), 0,0,z); z+=bw*0.16   # amalaka
    _merge(m, gen_frustum(bw*0.13,bw*0.13,0.0,0.0,bw*0.26), 0,0,z)                     # kalasha
    return m

def _rg_dome(rad, H, steps=6):
    """Hemispherical stone dome from stacked frustums + finial (no gen_dome_d dep)."""
    m=Mesh(); z=0.0
    for i in range(steps):
        t0=i/steps; t1=(i+1)/steps
        w0=2*rad*np.cos(t0*np.pi/2); w1=2*rad*np.cos(t1*np.pi/2); h=H/steps
        _merge(m, gen_frustum(w0,w0,w1,w1,h), 0,0,z); z+=h
    _merge(m, gen_frustum(rad*0.22,rad*0.22,0.0,0.0,rad*0.5), 0,0,z)                   # finial
    return m

def _rg_bastion(m, cx, cy, r, h, cap=True):
    """Cylindrical buruj: drum + cornice ring + merlon crown."""
    _place(m, _cyl((cx,cy,0),(cx,cy,h),r,16))
    _place(m, _cyl((cx,cy,h),(cx,cy,h+r*0.12),r*1.12,16))                              # cornice ring
    for k in range(14):                                                               # merlon crown
        a=2*np.pi*k/14; bx=cx+r*1.02*np.cos(a); by=cy+r*1.02*np.sin(a)
        add_box(m, bx-r*0.12, bx+r*0.12, by-r*0.12, by+r*0.12, h+r*0.12, h+r*0.55)

def _rg_chhatri(m, cx, cy, z, r, h, npill=4):
    """Domed kiosk: slab + corner pillars + dome cap."""
    add_box(m, cx-r*1.25, cx+r*1.25, cy-r*1.25, cy+r*1.25, z, z+h*0.08)
    for k in range(npill):
        a=np.pi/4+2*np.pi*k/npill; px=cx+r*np.cos(a); py=cy+r*np.sin(a)
        _place(m, _cyl((px,py,z+h*0.08),(px,py,z+h*0.62),r*0.16,8))
    add_box(m, cx-r*1.15, cx+r*1.15, cy-r*1.15, cy+r*1.15, z+h*0.62, z+h*0.72)         # entablature
    _merge(m, _rg_dome(r*1.05, h*0.5, 5), cx, cy, z+h*0.72)

def _rg_deepstambha(m, cx, cy, z, h):
    """Maratha lamp tower: tiered fluted pillar with lamp rings + flared top."""
    _place(m, _cyl((cx,cy,z),(cx,cy,z+h*0.78),h*0.05,10))
    for i in range(4):
        zr=z+h*0.25+i*h*0.16; _place(m, _cyl((cx,cy,zr),(cx,cy,zr+h*0.018),h*0.11,12))
    add_box(m, cx-h*0.09, cx+h*0.09, cy-h*0.09, cy+h*0.09, z+h*0.78, z+h*0.86)
    _merge(m, _rg_dome(h*0.10, h*0.16, 4), cx, cy, z+h*0.86)

def _rg_rampart(m, x0, x1, y0, y1, z0, h, th, gate_y=None, gw=2.4):
    """Rectangular fortified curtain wall ring + merlon battlement; optional west gate gap."""
    add_box(m, x0, x1, y0, y0+th, z0, z0+h); add_box(m, x0, x1, y1-th, y1, z0, z0+h)
    add_box(m, x1-th, x1, y0, y1, z0, z0+h)
    if gate_y is not None:
        add_box(m, x0, x0+th, y0, gate_y-gw, z0, z0+h); add_box(m, x0, x0+th, gate_y+gw, y1, z0, z0+h)
    else:
        add_box(m, x0, x0+th, y0, y1, z0, z0+h)
    mw, gap, mh = th*0.9, th*0.7, h*0.28
    _rg_crenX(m, x0, x1, y0, y0+th*0.6, z0+h, mw, gap, mh); _rg_crenX(m, x0, x1, y1-th*0.6, y1, z0+h, mw, gap, mh)
    _rg_crenY(m, x1-th*0.6, x1, y0, y1, z0+h, mw, gap, mh); _rg_crenY(m, x0, x0+th*0.6, y0, y1, z0+h, mw, gap, mh)

def _rg_gate(sv, cx, cy, w, depth, h):
    """Gomukhi grand gate: two huge flanking bastions + recessed corbel-arched passage + battlement."""
    _rg_bastion(sv, cx, cy-w*1.05, w*0.62, h*1.08); _rg_bastion(sv, cx, cy+w*1.05, w*0.62, h*1.08)
    pw=w*0.62
    add_box(sv, cx-depth*0.5, cx+depth*0.5, cy-w*1.0, cy-pw, 0, h)
    add_box(sv, cx-depth*0.5, cx+depth*0.5, cy+pw, cy+w*1.0, 0, h)
    for i in range(4):
        inset=pw*(1-(i+1)/5.0); zz=h*0.62+i*0.07*h
        add_box(sv, cx-depth*0.42, cx+depth*0.42, cy-inset, cy+inset, zz, zz+0.07*h)
    add_box(sv, cx-depth*0.5, cx+depth*0.5, cy-w*1.0, cy+w*1.0, h*0.92, h*1.0)
    _rg_crenY(sv, cx-depth*0.5, cx+depth*0.5, cy-w*1.0, cy+w*1.0, h*1.0, w*0.45, w*0.35, h*0.28)

def _rg_pitch_roof(rs, x0, x1, y0, y1, z0, ph):
    """Lost timber pitched/hip roof (restored layer) over a rectangular footprint."""
    b=len(rs.V); cx=(x0+x1)/2
    rs.V += [(x0,y0,z0),(x1,y0,z0),(x1,y1,z0),(x0,y1,z0),
             (cx-(x1-x0)*0.12,(y0+y1)/2,z0+ph),(cx+(x1-x0)*0.12,(y0+y1)/2,z0+ph)]
    for f in [(0,1,5),(0,5,4),(3,4,5),(3,5,2),(0,4,3),(1,2,5)]: rs.F.append((f[0]+b,f[1]+b,f[2]+b))

def _rg_stub(m, x, y, z, r, h):
    """Surviving stone column-base (the '...only the base pillars remain')."""
    add_box(m, x-r*1.5, x+r*1.5, y-r*1.5, y+r*1.5, z, z+h*0.4)
    _place(m, _cyl((x,y,z+h*0.4),(x,y,z+h),r,8))

def _rg_timber_hall(rs, x0, x1, y0, y1, z0, wallh, roofh, posts=None):
    """Reconstructed lost-timber building: light walls + pitched roof (+ optional gallery posts)."""
    add_box(rs, x0, x1, y0, y0+(y1-y0)*0.06, z0, z0+wallh)
    add_box(rs, x0, x1, y1-(y1-y0)*0.06, y1, z0, z0+wallh)
    add_box(rs, x0, x0+(x1-x0)*0.04, y0, y1, z0, z0+wallh)
    add_box(rs, x1-(x1-x0)*0.04, x1, y0, y1, z0, z0+wallh)
    _rg_pitch_roof(rs, x0-(x1-x0)*0.05, x1+(x1-x0)*0.05, y0-(y1-y0)*0.05, y1+(y1-y0)*0.05, z0+wallh, roofh)

def _rg_octa(m, cx, cy, z, r, h):           # octagonal prism (Samadhi platform / drum)
    _place(m, _cyl((cx,cy,z),(cx,cy,z+h), r, 8))

def _rg_octa_chhatri(m, cx, cy, z, r, h):   # octagonal domed meghdambari canopy on 8 pillars
    add_box(m, cx-r*1.2, cx+r*1.2, cy-r*1.2, cy+r*1.2, z, z+h*0.08)
    for k in range(8):
        a=2*np.pi*k/8; px=cx+r*np.cos(a); py=cy+r*np.sin(a)
        _place(m, _cyl((px,py,z+h*0.08),(px,py,z+h*0.6),r*0.12,8))
    _place(m, _cyl((cx,cy,z+h*0.6),(cx,cy,z+h*0.7),r*1.15,8))     # octagonal eave
    _merge(m, _rg_dome(r*1.0, h*0.55, 6), cx, cy, z+h*0.7)

def _rg_house(sv, rs, x, y, z, w, d, h):           # stone house: walls survive, roof lost
    add_box(sv, x, x+w, y, y+d, z, z+h)
    _rg_pitch_roof(rs, x-w*0.06, x+w*1.06, y-d*0.06, y+d*1.06, z+h, h*0.55)

def _rg_tank(m, x0, x1, y0, y1, z0, depth, steps=4):  # stepped sunken stone tank
    sx=(x1-x0)*0.1; sy=(y1-y0)*0.1
    for i in range(steps):
        add_box(m, x0+i*sx, x1-i*sx, y0+i*sy, y1-i*sy, z0-depth*(i+1)/steps, z0-depth*i/steps)
    add_box(m, x0+steps*sx, x1-steps*sx, y0+steps*sy, y1-steps*sy, z0-depth-0.08, z0-depth+0.12)

def _rg_temple(sv, rs, cx, cy, z, base, h, dome=True):  # small shrine: cube + dome/shikhara + porch
    add_box(sv, cx-base, cx+base, cy-base, cy+base, z, z+h*0.55)
    if dome: _merge(sv, _rg_dome(base*0.9, h*0.6, 5), cx, cy, z+h*0.55)
    else:    _merge(sv, _rg_shikhara(base*1.0, h*0.9), cx, cy, z+h*0.55)
    add_box(sv, cx-base*1.5, cx-base, cy-base*0.7, cy+base*0.7, z, z+h*0.18)         # porch floor (W)
    _rg_pitch_roof(rs, cx-base*1.6, cx-base*0.9, cy-base*0.8, cy+base*0.8, z+h*0.6, base*0.7)

def _rg_postern(sv, cx, cy, w, h):                 # small escape/secondary gate
    add_box(sv, cx-w*0.6, cx-w*0.2, cy-w*0.5, cy+w*0.5, 0, h)
    add_box(sv, cx+w*0.2, cx+w*0.6, cy-w*0.5, cy+w*0.5, 0, h)
    add_box(sv, cx-w*0.6, cx+w*0.6, cy-w*0.5, cy+w*0.5, h*0.8, h)

def build_raigad(scale=4.0):
    sv=Mesh(); rs=Mesh(); s=scale
    _merge(sv, gen_frustum(78*s, 34*s, 72*s, 28*s, 0.6*s)); bz=0.6*s
    PX0,PX1,PY0,PY1=-35*s,35*s,-13*s,13*s
    _rg_rampart(sv, PX0, PX1, PY0, PY1, bz, 2.6*s, 0.9*s, gate_y=0.0, gw=3.0*s)       # tatbandi
    for bx,by in [(PX0,PY0),(PX0,PY1),(PX1,PY0),(PX1,PY1),(-10*s,PY0),(8*s,PY1),(22*s,PY0),(-24*s,PY1)]:
        _rg_bastion(sv, bx, by, 1.4*s, 3.1*s)
    _rg_bastion(sv, PX0+0.6*s, -9.0*s, 2.1*s, 4.0*s)                                  # Hirakani Buruj
    _rg_bastion(sv, -2*s, PY1, 1.5*s, 3.6*s)                                          # Khubladha Buruj (N)
    _rg_bastion(sv, PX1, 4*s, 1.6*s, 3.6*s)                                           # Bhavani Buruj (E)
    _rg_chhatri(sv, PX0+0.6*s, -9.0*s, bz+4.0*s, 0.85*s, 1.8*s)                       # watchtower 1
    _rg_chhatri(sv, PX1, -4*s, bz+3.1*s, 0.85*s, 1.8*s)                               # watchtower 2

    # ---- W gated ascent: Maha Darwaja then a gomukhi corridor of Chit/Nane/Jit gates ----
    _rg_gate(sv, PX0+1.0*s, 0.0, 2.6*s, 3.4*s, 7.0*s)                                 # Maha Darwaja
    for gx in (PX0+3.5*s, PX0+6.0*s, PX0+8.5*s):                                      # Chit, Nane, Jit gates
        add_box(sv, gx-0.3*s, gx+0.3*s, -3.0*s, -1.3*s, bz, bz+3.0*s)                 # corridor wall N
        add_box(sv, gx-0.3*s, gx+0.3*s, 1.3*s, 3.0*s, bz, bz+3.0*s)                   # corridor wall S
        add_box(sv, gx-0.4*s, gx+0.4*s, -1.5*s, 1.5*s, bz+2.4*s, bz+3.0*s)            # gate lintel

    # ---- Bale Killa citadel ----
    add_box(sv, -26*s, -12*s, -8*s, 9*s, bz, bz+0.3*s)                                # citadel terrace
    rbx0,rbx1,rby0,rby1=-25*s,-16*s,-7*s,1*s                                          # Raj Bhavan palace
    add_box(sv, rbx0-0.5*s, rbx1+0.5*s, rby0-0.5*s, rby1+0.5*s, bz, bz+0.55*s); rbz=bz+0.55*s
    for x in np.linspace(rbx0+0.8*s, rbx1-0.8*s, 6):
        for y in np.linspace(rby0+0.8*s, rby1-0.8*s, 5): _rg_stub(sv, x, y, rbz, 0.22*s, 0.9*s)
    _rg_timber_hall(rs, rbx0, rbx1, rby0, rby1, rbz, 3.0*s, 3.2*s)
    _rg_timber_hall(rs, rbx0+1.0*s, rbx1-1.0*s, rby0+1.0*s, rby1-1.0*s, rbz+3.0*s, 1.6*s, 2.0*s)
    _rg_tank(sv, -25.5*s, -23.5*s, -8.6*s, -7.2*s, bz, 1.4*s); _rg_tank(sv, -22.5*s, -20.5*s, -8.6*s, -7.2*s, bz, 1.4*s)  # 2 palace tanks
    add_box(sv, -19.5*s, -17.5*s, -8.6*s, -7.2*s, bz, bz+0.5*s)                       # Royal Bath
    add_box(sv, -16*s, -13*s, -7*s, -4*s, bz-1.6*s, bz-0.1*s)                          # Khalbat-khana (sunken cellar)
    for i in range(6):                                                               # Ranivasa
        x=-25*s+i*1.45*s; add_box(sv, x, x+1.15*s, 3.4*s, 6.6*s, bz, bz+2.8*s)
        _rg_pitch_roof(rs, x-0.1*s, x+1.25*s, 3.3*s, 6.7*s, bz+2.8*s, 1.5*s)
    for i in range(3):                                                               # granaries
        x=-15.5*s+i*1.5*s; add_box(sv, x, x+1.2*s, 8.5*s, 11.5*s, bz, bz+2.6*s)
        _merge(sv, gen_frustum(1.2*s,3.0*s,0.5*s,2.6*s,0.7*s), x+0.6*s, 10.0*s, bz+2.6*s)
    _rg_bastion(sv, -18*s, PY1-0.3*s, 1.0*s, 4.0*s); _rg_bastion(sv, -15*s, PY1-0.3*s, 1.0*s, 4.0*s)
    add_box(sv, -18*s, -15*s, PY1-1.0*s, PY1-0.4*s, bz, bz+3.2*s)                     # Palkhi Darwaja
    _rg_bastion(sv, -22*s, PY0+0.3*s, 0.9*s, 3.6*s); _rg_bastion(sv, -19.5*s, PY0+0.3*s, 0.9*s, 3.6*s)
    add_box(sv, -22*s, -19.5*s, PY0+0.4*s, PY0+1.0*s, bz, bz+3.0*s)                   # Mena Darwaja
    djx0,djx1,djy0,djy1=-14*s,-5*s,-9.5*s,-2.5*s                                      # Rajsabha
    add_box(sv, djx0-0.4*s, djx1+0.4*s, djy0-0.4*s, djy1+0.4*s, bz, bz+0.8*s); djz=bz+0.8*s
    for i in range(4): add_box(sv, djx1+0.4*s+i*0.16*s, djx1+0.6*s+i*0.16*s, djy0-0.4*s, djy1+0.4*s, bz, bz+0.8*s-i*0.18*s)
    for x in np.linspace(djx0+0.9*s, djx1-0.9*s, 5):
        for y in np.linspace(djy0+0.9*s, djy1-0.9*s, 4):
            _rg_stub(sv, x, y, djz, 0.3*s, 0.7*s); _place(rs, gen_column(4.4*s, 0.3*s), t=(x,y,djz+0.7*s))
    add_box(rs, djx0+0.4*s, djx0+2.0*s, (djy0+djy1)/2-1.2*s, (djy0+djy1)/2+1.2*s, djz+0.7*s, djz+1.4*s)
    for px in (djx0+0.6*s, djx0+1.8*s):
        for py in ((djy0+djy1)/2-1.0*s, (djy0+djy1)/2+1.0*s): _place(rs, _cyl((px,py,djz+1.4*s),(px,py,djz+3.8*s),0.1*s,8))
    _merge(rs, _rg_dome(1.0*s, 1.6*s, 5), djx0+1.2*s, (djy0+djy1)/2, djz+3.8*s)
    _rg_pitch_roof(rs, djx0-0.3*s, djx1+0.3*s, djy0-0.3*s, djy1+0.3*s, djz+5.0*s, 3.4*s)
    _rg_bastion(sv, djx1+0.2*s, djy0+1.4*s, 0.9*s, 5.0*s); _rg_bastion(sv, djx1+0.2*s, djy1-1.4*s, 0.9*s, 5.0*s)
    add_box(sv, djx1-0.2*s, djx1+0.6*s, djy0+1.4*s, djy1-1.4*s, bz, bz+4.4*s)         # Nagarkhana
    for i in range(4):                                                               # Ashtapradhan Wada + wadas
        x=-12*s+i*1.7*s; add_box(sv, x, x+1.3*s, 8.0*s, 7.4*s, bz, bz+0.5*s)
        _rg_timber_hall(rs, x, x+1.3*s, 7.4*s, 8.0*s, bz+0.5*s, 1.8*s, 1.3*s)
    for i in range(3):
        x=-26*s+i*1.6*s; add_box(sv, x, x+1.2*s, -11*s, -9.5*s, bz, bz+0.45*s)        # other wadas
        _rg_timber_hall(rs, x, x+1.2*s, -11*s, -9.5*s, bz+0.45*s, 1.6*s, 1.2*s)

    # ---- centre: Holicha Mal, water system, temples, Bazar Peth, depots ----
    add_box(sv, -8*s, -2*s, -4*s, 4*s, bz, bz+0.18*s)                                 # Holicha Mal (open ground)
    _rg_tank(sv, -8*s, -4*s, 5*s, 9*s, bz, 1.8*s)                                     # Kushavarta Talav
    _rg_temple(sv, rs, -6*s, 10.5*s, bz, 0.8*s, 2.6*s, dome=False)                    # Shiva temple by Kushavarta (Wadeshwar)
    _rg_tank(sv, -2*s, 2*s, 6.5*s, 10.5*s, bz, 1.8*s)                                 # Gangasagar Lake
    _rg_tank(sv, 4*s, 8*s, 7*s, 11*s, bz, 1.6*s)                                      # Hatti Talav
    add_box(sv, 9*s, 13*s, 8*s, 11*s, bz, bz+0.4*s)                                   # Hatti Khana (stable)
    for c in range(4): _place(sv, _cyl((9.6*s+c*1.0*s,9.5*s,bz+0.4*s),(9.6*s+c*1.0*s,9.5*s,bz+2.2*s),0.12*s,8))
    _rg_tank(sv, -10*s, -7*s, -10.5*s, -7.5*s, bz, 1.6*s)                             # Kolimb Talav
    for r in range(3):                                                               # Bara Tanki (12 tanks, N)
        for c in range(4): _rg_tank(sv, -4*s+c*1.4*s, -3.2*s+c*1.4*s, 11.2*s+r*0.55*s, 11.6*s+r*0.55*s, bz, 0.9*s, 3)
    _rg_temple(sv, rs, 0*s, -10.5*s, bz, 0.7*s, 2.2*s, dome=True)                     # Shirkai Devi temple (S)
    add_box(sv, 16*s, 19*s, -10.5*s, -8*s, bz, bz+1.8*s)                              # Darukhana (ammunition depot)
    _merge(sv, gen_frustum(3.0*s,2.5*s,1.0*s,1.0*s,1.0*s), 17.5*s, -9.25*s, bz+1.8*s)

    # Bazar Peth: 22 stone shops each side of a ~40 ft street, ~850 ft long
    bx0=-1*s; pitch=1.0*s; nshop=22
    add_box(sv, bx0-0.5*s, bx0+nshop*pitch+0.5*s, -3.6*s, 3.6*s, bz, bz+0.4*s); pl=bz+0.4*s
    for sy in (-1,1):
        yy=sy*2.4*s
        for i in range(nshop):
            x=bx0+i*pitch
            add_box(sv, x, x+0.82*s, yy-0.8*s, yy+0.8*s, pl, pl+2.0*s)
            add_box(sv, x+0.28*s, x+0.54*s, yy-sy*0.82*s, yy-sy*0.78*s, pl, pl+1.4*s)
            _place(rs, _cyl((x+0.05*s,yy-sy*0.85*s,pl),(x+0.05*s,yy-sy*0.85*s,pl+1.9*s),0.07*s,6))
            _rg_pitch_roof(rs, x-0.06*s, x+0.88*s, yy-1.0*s, yy+1.0*s, pl+2.0*s, 1.4*s)

    # ~300 stone houses (representative residential cluster, S of the market)
    for r in range(4):
        for c in range(9):
            hx=2*s+c*1.7*s; hy=-11.5*s+r*1.0*s; _rg_house(sv, rs, hx, hy, bz, 1.1*s, 0.7*s, 1.3*s)

    add_box(sv, 12*s, 15*s, PY1-1.4*s, PY1-0.2*s, bz, bz+0.4*s)                       # Takmak Tok + stambha
    for px in (12.6*s,14.2*s): _place(sv, _cyl((px,PY1-0.8*s,bz+0.4*s),(px,PY1-0.8*s,bz+3.6*s),0.2*s,12))
    _rg_postern(sv, 18*s, PY1-0.5*s, 1.6*s, 3.0*s)                                    # Wagh Darwaja (escape)

    # ---- E: Jagdishwar (Mughal dome), Samadhi (octagonal), Bhavani temple/Tok ----
    tx,ty=27*s,-4.5*s
    add_box(sv, tx-3.4*s, tx+3.4*s, ty-3.6*s, ty+3.6*s, bz, bz+0.7*s); tz=bz+0.7*s
    add_box(sv, tx-1.9*s, tx+1.9*s, ty-1.9*s, ty+1.9*s, tz, tz+2.6*s)
    _merge(sv, gen_frustum(3.0*s,3.0*s,2.4*s,2.4*s,0.5*s), tx, ty, tz+2.6*s)
    _merge(sv, _rg_dome(1.5*s, 2.6*s, 6), tx, ty, tz+3.1*s)                            # Mughal-style dome
    add_box(sv, tx-3.0*s, tx-0.4*s, ty-2.3*s, ty+2.3*s, tz, tz+0.4*s)
    for mx in (tx-2.6*s, tx-0.9*s):
        for my in (ty-1.9*s, ty+1.9*s): _place(sv, gen_column(2.8*s,0.18*s), t=(mx,my,tz+0.4*s))
    _rg_pitch_roof(rs, tx-3.1*s, tx-0.3*s, ty-2.1*s, ty+2.1*s, tz+3.2*s, 1.2*s)
    _rg_chhatri(sv, tx-4.4*s, ty, tz, 0.7*s, 1.8*s); add_box(sv, tx-4.65*s, tx-4.15*s, ty-0.35*s, ty+0.45*s, tz+0.1*s, tz+0.7*s)
    _rg_deepstambha(sv, tx-5.6*s, ty+2.6*s, tz, 4.8*s)
    add_box(sv, tx-5.0*s, tx+3.6*s, ty-3.9*s, ty-3.6*s, bz, bz+1.3*s); add_box(sv, tx-5.0*s, tx+3.6*s, ty+3.6*s, ty+3.9*s, bz, bz+1.3*s)
    add_box(sv, tx+3.4*s, tx+3.9*s, ty-1.0*s, ty+1.0*s, bz, bz+1.5*s)
    sx,sy=31*s,-4.5*s                                                                 # Samadhi (octagonal)
    _rg_octa(sv, sx, sy, bz, 2.0*s, 0.4*s); _rg_octa(sv, sx, sy, bz+0.4*s, 1.6*s, 0.5*s)
    add_box(sv, sx-0.5*s, sx+0.5*s, sy-0.5*s, sy+0.5*s, bz+0.9*s, bz+1.5*s)
    _rg_octa_chhatri(sv, sx, sy, bz+0.9*s, 1.5*s, 3.2*s)
    _rg_chhatri(sv, sx, sy+3.0*s, bz, 0.5*s, 1.3*s)                                   # Waghya
    _rg_chhatri(sv, sx+2.0*s, sy-2.0*s, bz, 0.42*s, 1.1*s)                            # Hiroji Samadhi
    _rg_temple(sv, rs, 33*s, 6*s, bz, 0.7*s, 2.2*s, dome=False)                       # Bhavani Temple (E)
    add_box(sv, PX1-1.0*s, PX1, 3*s, 5*s, bz, bz+0.3*s)                               # Bhavani Tok platform

    rub=make_rubble(-20.5*s, -3.0*s, bz, spread=4.0*s, n=70, block=0.22*s)
    return sv, rs, rub

def load_mesh_obj(path):
    """Minimal OBJ loader -> Mesh. Accepts a photogrammetry / laser-scan export of surviving fabric."""
    m=Mesh()
    for ln in open(path):
        if ln.startswith("v "):
            p=ln.split(); m.V.append((float(p[1]),float(p[2]),float(p[3])))
        elif ln.startswith("f "):
            idx=[int(q.split("/")[0]) for q in ln.split()[1:]]
            idx=[(i-1) if i>0 else (len(m.V)+i) for i in idx]
            for k in range(1,len(idx)-1): m.F.append((idx[0],idx[k],idx[k+1]))
    return m

def _rg_bbox(m):
    V=np.array(m.V); return V.min(0), V.max(0)

def reconstruct_from_survey(survey, builder=build_raigad):
    """PHOTO/SCAN-GROUNDED reconstruction. `survey` is a Mesh of the SURVIVING fabric, e.g. produced by
    photogrammetry (COLMAP / Meshroom over many OVERLAPPING photos) or a laser scan. The canon-grounded
    LOST parts are scaled and aligned ONTO that real survey, so the reconstruction is built on top of the
    measured remains rather than an idealised plan. Coarse bbox/centroid + uniform-scale registration is
    used here; refine with ICP/feature alignment for production. Returns (survived=survey, restored=overlay).
    NOTE: a single photo cannot yield metric 3D - overlapping multi-view coverage (or a scan) is required."""
    sv_ref, rs_ref, _ = builder()
    a0,a1=_rg_bbox(sv_ref); b0,b1=_rg_bbox(survey)
    sc=float((b1-b0).max()/max((a1-a0).max(),1e-9)); ca=(a0+a1)/2; cb=(b0+b1)/2
    out=Mesh()
    for (x,y,z) in rs_ref.V:
        out.V.append((float((x-ca[0])*sc+cb[0]), float((y-ca[1])*sc+cb[1]), float((z-ca[2])*sc+cb[2])))
    out.F=list(rs_ref.F)
    return survey, out

# --- (1) THE TRACK: every documented part -> condition -> modelled? ---
raigad_parts_report()

# --- (2) evidence dossier ---
print("\n"+"="*104); print("RAIGAD FORT - EVIDENCE DOSSIER  (component -> source -> verification status)"); print("="*104)
print("%-22s %-22s %s" % ("COMPONENT","STATUS","SOURCE / NOTE")); print("-"*104)
for comp, note, src, status in RAIGAD_EVIDENCE:
    print("%-22s %-22s %s" % (comp, status, src)); print("%-22s %-22s -> %s" % ("", "", note))
ndoc=sum(1 for r in RAIGAD_EVIDENCE if r[3].startswith("documented")); print("-"*104)
print("%d/%d elements documented (cited)." % (ndoc, len(RAIGAD_EVIDENCE))); print("Sources:", " | ".join(RAIGAD_SOURCES))

# --- (3) render: AS FOUND (today) vs RECONSTRUCTED (prime) ---
def render_raigad(sv, rs, rub, path):
    s=4.0; X0,X1,Y0,Y1=-37*s,37*s,-14*s,14*s; cx=(X0+X1)/2; cy=(Y0+Y1)/2; zb=0.6*s; ztop=zb+12*s; W=(X1-X0)/2
    def setup(ax,az=-62,el=28):
        ax.set_xlim(cx-W,cx+W); ax.set_ylim(cy-W,cy+W); ax.set_zlim(zb-3*s, ztop)
        ax.set_box_aspect(((X1-X0),(Y1-Y0),(ztop-zb)*2.4)); ax.set_axis_off(); ax.view_init(elev=el,azim=az)
    def add(ax,m,base,jit=0.0):
        if len(m.F):
            tris=np.array(m.V)[np.array(m.F)]; ax.add_collection3d(Poly3DCollection(tris,facecolors=_rshade(tris,base,jit),linewidths=0))
    fig=plt.figure(figsize=(28,7))
    a1=fig.add_subplot(1,2,1,projection="3d"); setup(a1); add(a1,sv,(0.60,0.55,0.50),0.12); add(a1,rub,(0.42,0.38,0.34))
    a1.set_title("AS FOUND (today's remains)  -  surviving stone + base pillars",fontsize=9)
    a2=fig.add_subplot(1,2,2,projection="3d"); setup(a2); add(a2,sv,(0.60,0.55,0.50),0.12); add(a2,rs,(0.82,0.56,0.33))
    a2.set_title("RECONSTRUCTED (prime)  -  stone (grey) + lost timber (highlighted)",fontsize=9)
    fig.suptitle("Raigad Fort - documented-plan reconstruction (44 parts)\ncoronation 1674 | wooden palace burnt 1818, only base pillars remain",fontsize=12)
    plt.tight_layout(); plt.savefig(path,dpi=110,bbox_inches="tight"); plt.show()

sv, rs, rub = build_raigad(scale=4.0)
open("/kaggle/working/Raigad_survived.obj","w").write(sv.obj()); open("/kaggle/working/Raigad_restored.obj","w").write(rs.obj())
print("\nSurviving stone : %d faces   Reconstructed timber : %d faces   Rubble : %d faces" % (len(sv.F), len(rs.F), len(rub.F)))
print("Exported Raigad_survived.obj + Raigad_restored.obj to /kaggle/working")
render_raigad(sv, rs, rub, "/kaggle/working/Raigad_reconstruction.png")

# --- (4) PHOTO/SCAN-GROUNDED REBUILD (end-to-end): build the lost timber ON the real remains ---
# Pipeline: overlapping site photos --(COLMAP/Meshroom SfM, offline)--> survey.obj  (OR download a real mesh)
print("\nReal downloadable survey meshes to ground on (Sketchfab / CyArk):")
for u in RAIGAD_SURVEY_MESHES: print("   ", u)
def ground_on_survey(survey, path):
    survived, overlay = reconstruct_from_survey(survey)        # scale+align canon lost-parts onto the survey
    allV=np.array(survived.V+overlay.V); ctr=(allV.max(0)+allV.min(0))/2; rng=(allV.max(0)-allV.min(0)).max()/2
    fig=plt.figure(figsize=(15,9)); ax=fig.add_subplot(111,projection="3d")
    ax.set_xlim(ctr[0]-rng,ctr[0]+rng); ax.set_ylim(ctr[1]-rng,ctr[1]+rng); ax.set_zlim(allV.min(0)[2],allV.min(0)[2]+1.8*rng)
    ax.set_box_aspect((1.7,1,0.9)); ax.set_axis_off(); ax.view_init(elev=24,azim=-60)
    for m,c,j in [(survived,(0.58,0.56,0.52),0.10),(overlay,(0.82,0.55,0.33),0.0)]:
        if len(m.F):
            tris=np.array(m.V)[np.array(m.F)]; ax.add_collection3d(Poly3DCollection(tris,facecolors=_rshade(tris,c,j),linewidths=0))
    ax.set_title("SCAN-GROUNDED: surviving fabric from survey (grey) + reconstructed lost timber on top (terracotta)",fontsize=9)
    plt.savefig(path,dpi=110,bbox_inches="tight"); plt.show()
import os
SCAN="/kaggle/input/raigad-scan/survey.obj"     # <- upload a Sketchfab/CyArk .obj here to ground on the REAL remains
survey = load_mesh_obj(SCAN) if os.path.exists(SCAN) else sv   # fallback: procedural survived stands in for the scan
print("\nGrounding on:", ("REAL scan "+SCAN) if os.path.exists(SCAN) else "procedural survived fabric (demo; no scan uploaded)")
ground_on_survey(survey, "/kaggle/working/Raigad_grounded.png")
print("A single photo cannot give metric 3D - overlapping multi-view photos (SfM) or a laser scan are required.")
